[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/alg/u6_aplicaciones.ipynb)

# Álg. U6 · Aplicaciones a ingeniería
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 6 del bloque de álgebra lineal. Cada sección resuelve con vectores, matrices y sistemas un problema que vas a encontrar en las materias de tu carrera, en el mismo orden que el libro:

1. Equilibrio de fuerzas en 3D, momento y trabajo
2. Análisis de mallas con leyes de Kirchhoff
3. Mezclas y asignación de recursos
4. Armadura por el método de nodos
5. Ajuste de datos por mínimos cuadrados
6. Transformaciones de coordenadas

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección tiene una celda de teoría, código que la recorre paso a paso, una calculadora, sus casos de prueba y una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `ALG-U6` (bloque, unidad). Las celdas de código de cada sección vienen de `alg/u6_aplicaciones/NN_*.py`; los fragmentos que el libro imprime, de `alg/u6_aplicaciones/fragmentos/`.

**Convenciones.**

- **Las listas se escriben un elemento por renglón** (o con los elementos separados por `;`), y los números de cada elemento se separan con comas o espacios. Cada calculadora dice qué lleva cada renglón: un anclaje `x, y, z` (sección 1); una rama `R, malla, malla` y una fuente `V, malla` (sección 2); un ingrediente con sus porcentajes (sección 3); un nodo `nombre, x, y`, una barra `nombre, nombre`, un apoyo `nombre, x` o `nombre, y` y una carga `nombre, Fx, Fy` (sección 4).
- Un vector suelto (una carga, un punto, una traslación) se escribe como lista de números: `0, 0, -90`. También se acepta `(0, 0, -90)`.
- **Los datos de la sección 5** se escriben como listas de números separados por espacios o comas: `0 2 3.6` o `0, 2, 3.6` (hasta 500).
- Punto decimal. Puedes escribir fracciones como `2/3` (también `2 / 3`); los decimales se convierten a fracciones exactas (`0.3` se lee como $\tfrac{3}{10}$), así que las tensiones, las corrientes, las cantidades de una mezcla y las fuerzas en las barras se calculan sin error de redondeo. Las raíces que salen de las longitudes (por ejemplo $\sqrt2$ en un vector unitario) se dejan exactas y se muestran también en decimales. Se aceptan los signos menos tipográficos (− y –) y un espacio después del signo (`- 1`).
- **Unidades.** Cada calculadora dice en qué unidades van sus datos (o tiene un campo para escribirlas) y las escribe en sus resultados. Un ángulo pide **siempre** su unidad, `"grados"` o `"radianes"`: no hay unidad por omisión.
- Mallas, nodos de un circuito, cables, ingredientes y datos se cuentan desde 1 (los pasos de tiempo de la sección 5, desde 0: $x[0],x[1],\dots$). En lo que muestran las calculadoras, el signo menos es siempre `-` y las raíces se escriben con `√`. Los decimales que se muestran se *redondean* (no se truncan) a las cifras significativas que elijas; si una fracción exacta resulta demasiado larga, la calculadora muestra solo sus decimales y lo avisa.
- Si usas las funciones desde código, les puedes pasar texto (como en las calculadoras) o listas de Python: `equilibrio_particula([(2, 2, 1), (-2, 1, 2), (1, -2, 2)], (0, 0, -90))`.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re, io, textwrap, contextlib, time
from decimal import Decimal, ROUND_HALF_UP, localcontext
from fractions import Fraction
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registra la proyección "3d")
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import sympy as sp
import ipywidgets as widgets
from IPython.display import display, Math

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Utilidades compartidas por todo el notebook (las mismas de las unidades 4 y 5): leer números y matrices
# exactos, redondear y mostrar
matplotlib.rcParams["axes.unicode_minus"] = False     # signo menos ASCII también en los ejes de las gráficas
_NUMERO = re.compile(r"[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?")
_AYUDA_NUMERO = "usa enteros, fracciones como 2/3 o decimales con punto"


_LIMITE = Fraction(10) ** 15                  # entradas entre 1e-15 y 1e15 en valor absoluto (o 0)


def _en_rango(q, texto):
    if q != 0 and not Fraction(1) / _LIMITE <= abs(q) <= _LIMITE:
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    return sp.Rational(q.numerator, q.denominator)


def numero(valor, nombre="el número"):
    """Un número como racional exacto de sympy: '0.1' -> 1/10, '2/3' -> 2/3, 5 -> 5, 1.2 -> 6/5."""
    if valor is None:
        raise ValueError(f"{nombre} está vacío: escribe un número.")
    if isinstance(valor, bool):
        raise ValueError(f"«{valor}» no es un número: {_AYUDA_NUMERO}.")
    if isinstance(valor, (int, np.integer)):
        return _en_rango(Fraction(int(valor)), valor)
    if isinstance(valor, (float, np.floating)) or (isinstance(valor, sp.Basic) and valor.is_Float):
        v = float(valor)
        if not math.isfinite(v):
            raise ValueError(f"{nombre} no es finito (escribiste {v}).")
        return _en_rango(Fraction(repr(v)), repr(v))      # repr da el decimal más corto: 0.1 -> '0.1' -> 1/10 exacto
    if isinstance(valor, sp.Basic):
        if valor.is_Rational:
            return valor                             # ya es exacto (por ejemplo, un resultado intermedio): no se acota
        raise ValueError(f"«{valor}» no es un número racional: {_AYUDA_NUMERO}.")
    texto = re.sub(r"\s+", "", str(valor)).replace("−", "-").replace("–", "-")      # acepta los signos menos tipográficos
    partes = texto.split("/")
    formas = [_NUMERO.fullmatch(p) for p in partes]
    if not texto or len(partes) > 2 or not all(formas):
        raise ValueError(f"«{str(valor).strip()}» no es un número: {_AYUDA_NUMERO}." if texto else
                         f"{nombre} está vacío: escribe un número.")
    if any(f.group(2) and abs(int(f.group(2)[1:])) > 30 for f in formas):   # 1e999999 no se calcula: se rechaza antes
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    num, *den = (Fraction(p) for p in partes)
    if den and den[0] == 0:
        raise ValueError(f"«{texto}» divide entre cero.")
    return _en_rango(num / den[0] if den else num, texto)


def _entradas(k):
    return f"{k} entrada" if k == 1 else f"{k} entradas"


def _componentes(k):
    return f"{k} componente" if k == 1 else f"{k} componentes"


def _desde_renglones(renglones, nombre):
    """Lista de renglones (cada uno, lista de entradas) -> Matrix de racionales, con mensajes claros."""
    if not renglones or not renglones[0]:
        raise ValueError(f"escribe {nombre} por renglones, por ejemplo: 1, 2; 3, 4")
    n = len(renglones[0])
    salida = []
    for i, renglon in enumerate(renglones, 1):
        if len(renglon) != n:
            raise ValueError(f"en {nombre}, el renglón {i} tiene {_entradas(len(renglon))} y el 1 tiene {n}: "
                             "todos los renglones deben tener el mismo número de entradas")
        fila = []
        for j, v in enumerate(renglon, 1):
            try:
                fila.append(numero(v))
            except ValueError as err:
                raise ValueError(f"en {nombre}, renglón {i}, columna {j}: {err}") from None
        salida.append(fila)
    return sp.Matrix(salida)


def _normaliza_entrada(texto):
    """Signos menos tipográficos (− y –) como '-', llaves como paréntesis y sin espacios alrededor de '/' ni entre un
    signo y su número: '2 / 3' -> '2/3', '- 1' -> '-1', '{(1, 1), (0, 1)}' -> '((1, 1), (0, 1))'."""
    t = texto.replace("−", "-").replace("–", "-").replace("{", "(").replace("}", ")")
    t = re.sub(r"\s*/\s*", "/", t)
    return re.sub(r"(^|[\s,;(\[])([+-])\s+(?=[\d.])", r"\1\2", t)


def _trozos(texto):
    """Texto -> lista de renglones, cada uno lista de entradas (texto). Acepta ';' o saltos de línea entre renglones,
    comas o espacios entre entradas, y la forma de Python [[1, 2], [3, 4]], (1, 2), (3, 4) o {(1, 2), (3, 4)}.
    Devuelve también el número del primer renglón con una entrada vacía (o None)."""
    t = _normaliza_entrada(texto)
    t = re.sub(r"[\]\)]\s*,?\s*[\[\(]", ";", t)            # [[1, 2], [3, 4]] o (1, 2), (3, 4): cada grupo es un renglón
    t = re.sub(r"[\[\]\(\)]", " ", t)
    renglones = [r.strip() for r in re.split(r"[;\n]", t)]
    renglones = [r for r in renglones if r]
    entradas, vacia = [], None
    for i, r in enumerate(renglones, 1):
        partes = re.split(r"\s*,\s*|\s+", r)
        if "" in partes and vacia is None:
            vacia = i
        entradas.append(partes)
    return entradas, vacia


def matriz(texto, nombre="la matriz"):
    """Lee una matriz escrita por renglones ('1, 2; 3, 4' o un renglón por línea) como Matrix de racionales exactos."""
    if not isinstance(texto, str):
        return como_matriz(texto, nombre)
    entradas, vacia = _trozos(texto)
    if vacia is not None:
        raise ValueError(f"en {nombre}, el renglón {vacia} tiene una entrada vacía: "
                         "revisa que no haya dos comas seguidas ni una coma al principio o al final")
    try:
        return _desde_renglones(entradas, nombre)
    except ValueError as err:
        if "todos los renglones deben tener" in str(err):
            raise ValueError(f"{err}{pista_coma(texto)}") from None
        raise


def pista_coma(texto):
    """La pista de usar punto decimal, solo si la lectura es de veras ambigua: hay comas entre dígitos sin espacio
    (1,5) y además números separados solo por espacios (1,5 2,5), que es como se escriben decimales con coma."""
    if isinstance(texto, str) and re.search(r"\d,\d", texto) and re.search(r"\d[ \t]+[-+.]?\d", texto):
        return "; si escribiste decimales, usa punto decimal: 1.5, no 1,5"
    return ""


def como_matriz(A, nombre="la matriz"):
    """Acepta texto, Matrix de sympy, lista de listas o arreglo de numpy y devuelve una Matrix de racionales exactos."""
    if isinstance(A, str):
        return matriz(A, nombre)
    if isinstance(A, sp.MatrixBase):
        renglones = A.tolist()
    elif isinstance(A, np.ndarray):
        if A.ndim > 2:
            raise ValueError(f"{nombre} debe ser una tabla de números (renglones y columnas)")
        renglones = np.atleast_2d(A).tolist()
    elif isinstance(A, (list, tuple)):
        anidada = [isinstance(f, (list, tuple, np.ndarray, sp.MatrixBase)) for f in A]
        if A and all(anidada):
            renglones = [list(f) for f in A]
        elif not any(anidada):
            renglones = [list(A)]                  # una lista sola es un renglón
        else:
            raise ValueError(f"en {nombre}, mezclaste números sueltos con renglones")
    else:
        renglones = [[A]]
    return _desde_renglones(renglones, nombre)


def vector(texto, nombre="el vector"):
    """Lee un vector ('1, 2, 3', '1; 2; 3', (1, 2, 3) o [1, 2, 3]) como columna de racionales exactos.
    Los mensajes de error dicen qué componente está mal."""
    ejemplo = f"escribe {nombre} como lista de números, por ejemplo: 5, 4, 1"
    if texto is None:
        raise ValueError(ejemplo)
    if isinstance(texto, str):
        filas, vacia = _trozos(texto)
        if vacia is not None:
            raise ValueError(f"en {nombre} hay una componente vacía: revisa que no haya dos comas seguidas "
                             "ni una coma al principio o al final")
        if not filas:
            raise ValueError(ejemplo)
        if len(filas) == 1:
            entradas = filas[0]
        elif all(len(f) == 1 for f in filas):
            entradas = [f[0] for f in filas]
        else:
            raise ValueError(f"{nombre} debe ser una lista de números, por ejemplo: 5, 4, 1 (escribiste varios renglones)")
    elif isinstance(texto, (list, tuple, np.ndarray, sp.MatrixBase)):
        plano = ((isinstance(texto, np.ndarray) and texto.ndim == 1) or
                 (isinstance(texto, (list, tuple)) and
                  not any(isinstance(e, (list, tuple, np.ndarray, sp.MatrixBase)) for e in texto)))
        if not plano:                                   # Matrix, arreglo 2D o lista de listas
            return _vector_de_matriz(texto, nombre, ejemplo)
        entradas = list(texto)
        if not entradas:
            raise ValueError(ejemplo)
    else:
        entradas = [texto]                              # un número solo
    valores = []
    for j, e in enumerate(entradas, 1):
        try:
            valores.append(numero(e))
        except ValueError as err:
            raise ValueError(f"en {nombre}, componente {j}: {err}") from None
    return sp.Matrix(valores)


def _vector_de_matriz(texto, nombre, ejemplo):
    """Un vector dado como Matrix, arreglo o lista anidada: debe tener un solo renglón o una sola columna."""
    try:
        v = como_matriz(texto, nombre)
    except ValueError as err:
        if str(err).startswith("escribe "):
            raise ValueError(ejemplo) from None
        raise
    if v.rows == 1 or v.cols == 1:
        return v.reshape(v.rows * v.cols, 1)
    raise ValueError(f"{nombre} debe ser una lista de números, por ejemplo: 5, 4, 1 (escribiste una matriz {dims(v)})")


def dims(M):
    """Tamaño 'm×n' de una matriz."""
    return f"{M.rows}×{M.cols}"


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales.
    Un racional exacto se redondea desde su valor exacto (p/q con 50 cifras), no desde su aproximación en punto
    flotante: 1249999999999999999/10^19 con 2 cifras da 0.12, no 0.13. Una expresión exacta con raíces, senos o
    cosenos se evalúa con 50 cifras antes de redondear."""
    with localcontext() as ctx:
        ctx.prec = 50
        if isinstance(valor, sp.Basic) and not valor.is_Rational and valor.is_number:
            valor = sp.N(valor, 50)
            if valor.is_real is False:
                return "no definido"
            d = Decimal(str(valor)) if valor.is_Float else Decimal(repr(float(valor)))
        elif isinstance(valor, (sp.Rational, Fraction, int, np.integer)) and not isinstance(valor, bool):
            p, q = (int(valor.p), int(valor.q)) if isinstance(valor, sp.Rational) else (
                (valor.numerator, valor.denominator) if isinstance(valor, Fraction) else (int(valor), 1))
            d = Decimal(p) / Decimal(q)
        else:
            v = float(valor)
            if math.isnan(v):
                return "no definido"
            if math.isinf(v):
                return "+∞" if v > 0 else "-∞"
            d = Decimal(repr(v))
        if d == 0:
            return "0"
        q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
        if q.adjusted() != d.adjusted():      # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
            q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
        e = q.adjusted()
        if e > 15 or e < -4:                  # muy grandes o muy chicos: notación científica, 1.234×10^-7
            return f"{format(q.scaleb(-e), 'f')}×10^{e}"
        if q.adjusted() >= n:                 # enteros grandes: exactos tal cual; redondeados, con ×10^
            return f"{int(q):d}" if q == d else f"{format(q.scaleb(-e), 'f')}×10^{e}"
        return format(q, "f")


def decimales_fijos(valor, d=2):
    """Redondea (5 sube) a d decimales fijos, desde el valor exacto: sqrt(3) con 2 -> '1.73'; 1 -> '1.00'."""
    with localcontext() as ctx:
        ctx.prec = 50
        x = Decimal(str(sp.N(sp.sympify(valor), 40)))
        return format(x.quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP), "f")


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una función rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def _mensaje(fn):
    """Texto del ValueError que lanza fn() ('' si no lanza): para probar que el mensaje explica el error."""
    try:
        fn()
    except ValueError as err:
        return str(err)
    return ""


# --- formato de números y matrices ---------------------------------------------------------------
_SUB = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
_SUP = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")


def _sub(*indices):
    """Subíndices: _sub(1, 2) -> '₁₂'; con índices de dos cifras los separa con coma."""
    if all(k < 10 for k in indices):
        return "".join(str(k) for k in indices).translate(_SUB)
    return "₍" + ",".join(str(k) for k in indices).translate(_SUB) + "₎"


def _R(n):
    """Nombre del espacio en texto plano: _R(3) -> 'R³'."""
    return "R" + str(n).translate(_SUP)


def _txt(e):
    """Una cantidad exacta como texto legible: sqrt(3)/2 -> √3/2, 2*sin(30) -> 2·sen(30), pi/6 -> π/6."""
    t = str(e)
    t = re.sub(r"sqrt\((\d+)\)", r"√\1", t).replace("sqrt(", "√(")
    return t.replace("**", "^").replace("*", "·").replace("sin(", "sen(").replace("pi", "π")


def _latex(x):
    """LaTeX de sympy con la convención del libro: sen en lugar de sin."""
    return sp.latex(x).replace(r"\sin", r"\operatorname{sen}")


def _exacta_legible(M, largo=28, total=400):
    """True si la forma exacta de M se lee bien: cada entrada con pocos caracteres y no demasiadas en total.
    Si no, las calculadoras muestran solo los decimales y lo avisan."""
    textos = [_txt(e) for e in M]
    return all(len(t) <= largo for t in textos) and sum(len(t) for t in textos) <= total


_LARGA = "la forma exacta es muy larga: se muestra redondeada"


def _igual(nombre, x, n=4):
    """'nombre = x' con x exacto (y sus decimales si no es entero); si la forma exacta es muy larga,
    'nombre ≈ decimales (aviso)'."""
    x = sp.sympify(x)
    if len(_txt(x)) > 28:
        return f"{nombre} ≈ {cifras(x, n)} ({_LARGA})"
    return f"{nombre} = {_txt(x)}" + ("" if x.is_integer else f" ≈ {cifras(x, n)}")


def _en_linea(M):
    """Matriz en una línea con la misma notación de entrada: [2, 3; 5, -2]."""
    return "[" + "; ".join(", ".join(_txt(e) for e in M.row(i)) for i in range(M.rows)) + "]"


def _tupla(v, n=None):
    """Vector como (a, b, c); con n, además sus decimales redondeados a n cifras. Si la forma exacta es muy larga,
    solo los decimales (a 4 cifras si no se da n), con el aviso."""
    if not _exacta_legible(v):
        return "≈ (" + ", ".join(cifras(e, n or 4) for e in v) + f") ({_LARGA})"
    exacto = "(" + ", ".join(_txt(e) for e in v) + ")"
    if n is None or all(e.is_integer for e in v):
        return exacto
    return exacto + " ≈ (" + ", ".join(cifras(e, n) for e in v) + ")"


def _aprox(v, n):
    """Decimales redondeados de un vector: '(0.5000, 1.333)'; con una sola entrada, sin paréntesis."""
    textos = [cifras(e, n) for e in v]
    return textos[0] if len(textos) == 1 else "(" + ", ".join(textos) + ")"


def _plano(M, fmt=None, corte=None):
    """Texto plano de una matriz con columnas alineadas; corte = columna tras la que se dibuja una barra."""
    fmt = fmt or _txt
    celdas = [[fmt(M[i, j]) for j in range(M.cols)] for i in range(M.rows)]
    anchos = [max(len(celdas[i][j]) for i in range(M.rows)) for j in range(M.cols)]
    lineas = []
    for fila in celdas:
        partes = [c.rjust(a) for c, a in zip(fila, anchos)]
        if corte:
            partes = partes[:corte] + ["|"] + partes[corte:]
        lineas.append("[ " + "  ".join(partes) + " ]")
    return "\n".join(lineas)


_TEX = [("⁻¹", "^{-1}"), ("ᵀ", "^{T}"), ("²", "^{2}"), ("·", r"\cdot "), ("×", r"\times "), ("≈", r"\approx "),
        ("←", r"\leftarrow "), ("−", "-"), ("|", r"\,|\,")]
_SUB_TEX = re.compile("[₀₁₂₃₄₅₆₇₈₉]+")
_PROSA = re.compile(r"[A-Za-zÁÉÍÓÚáéíóúñü]{3,}")


def _a_tex(nombre):
    """Nombre de una cantidad en LaTeX. Si lleva palabras ('forma escalonada reducida'), va completo en \\text{...},
    con sus espacios. Si no, se traducen los símbolos (P_C⁻¹·P_B -> P_C^{-1}\\cdot P_B) y los subíndices (A2 -> A_{2},
    v1 -> \\mathbf v_{1}). Las calculadoras pasan el LaTeX a mano cuando el nombre mezcla palabras y símbolos."""
    if _PROSA.search(re.sub(r"\bdet\b", "", nombre)):
        return r"\text{" + nombre.replace("{", r"\{").replace("}", r"\}") + "}"
    t = nombre
    for a, b in _TEX:
        t = t.replace(a, b)
    t = re.sub(r"\bdet\s+", r"\\det ", t)
    t = _SUB_TEX.sub(lambda m: "_{" + m.group(0).translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")) + "}", t)
    return re.sub(r"(?<![A-Za-z0-9\\])([A-Za-z])(\d+)(?![\d/.])",
                  lambda m: (r"\mathbf " if m.group(1) in "vbeu" else "") + m.group(1) + "_{" + m.group(2) + "}", t)


def _tex_cifra(t):
    """Un número ya redondeado, en LaTeX: '1.234×10^-7' -> '1.234\\times 10^{-7}'."""
    return re.sub(r"×10\^(-?\d+)", r"\\times 10^{\1}", t)


def _tex_decimales(M, n):
    filas = r" \\ ".join(" & ".join(_tex_cifra(cifras(M[i, j], n)) for j in range(M.cols)) for i in range(M.rows))
    return r"\left[\begin{matrix}" + filas + r"\end{matrix}\right]"


class _Formula(Math):
    """Math de IPython con una versión en texto plano (la que se ve donde no se dibuja LaTeX)."""
    def __init__(self, tex, texto):
        super().__init__(tex)
        self._texto = texto

    def __repr__(self):
        return self._texto


class _Bloque:
    """Salida en Markdown (texto en recuadro y fórmulas en LaTeX) con su versión en texto plano."""
    def __init__(self, partes):
        self._partes = partes

    @staticmethod
    def _envuelve(texto, ancho=100):
        """Corta los renglones largos (en un recuadro de código no se ajustan solos); conserva la sangría."""
        lineas = []
        for linea in texto.rstrip("\n").split("\n"):
            sangria = " " * (len(linea) - len(linea.lstrip(" ")))
            lineas += textwrap.wrap(linea, ancho, subsequent_indent=sangria + "  ") if len(linea) > ancho else [linea]
        return "\n".join(lineas)

    def _repr_markdown_(self):
        return "\n\n".join(f"$${tex}$$" if tipo == "tex" else "```text\n" + self._envuelve(texto) + "\n```"
                            for tipo, tex, texto in self._partes)

    def __repr__(self):
        return "\n".join(texto.rstrip("\n") for _, _, texto in self._partes)


_BLOQUES = []      # bloques abiertos: lo que se muestra dentro de uno se junta y se entrega de una vez


@contextlib.contextmanager
def bloque():
    """Junta lo que se imprime y las fórmulas que se muestran dentro del bloque y lo entrega como UNA sola salida.
    Las calculadoras lo usan para que su resultado aparezca completo de una vez, en el orden en que se produjo."""
    partes, buf = [], io.StringIO()

    def vacia():
        if buf.getvalue():
            partes.append(("texto", "", buf.getvalue().replace("= ≈", "≈")))     # un valor que salió redondeado
            buf.seek(0); buf.truncate()

    _BLOQUES.append((partes, vacia))
    try:
        with contextlib.redirect_stdout(buf):
            yield
    finally:
        vacia()
        _BLOQUES.pop()
        if partes:
            _emite(_Bloque(partes)) if _BLOQUES else display(_Bloque(partes))


def _emite(formula):
    """Muestra una fórmula, o la agrega al bloque abierto si lo hay."""
    if not _BLOQUES:
        display(formula)
        return
    partes, vacia = _BLOQUES[-1]
    vacia()
    if isinstance(formula, _Bloque):
        partes.extend(formula._partes)
    else:
        partes.append(("tex", formula.data, formula._texto))


def muestra(nombre, M, n=None, tex=None):
    """Muestra 'nombre = M' con LaTeX (con su tamaño si es matriz). Con n agrega los decimales redondeados a n cifras.
    tex es el nombre en LaTeX (si no se da, se arma con _a_tex). Si la forma exacta es muy larga, solo los decimales."""
    tex_nombre = tex if tex is not None else _a_tex(nombre)
    if isinstance(M, sp.MatrixBase):
        if _exacta_legible(M):
            tex = f"{tex_nombre} = {_latex(M)}"
            texto = f"{nombre}  ({dims(M)}) =\n{_plano(M)}"
            if n is not None and not all(e.is_integer for e in M):
                tex += r" \approx " + _tex_decimales(M, n)
                texto += f"\n≈\n{_plano(M, lambda e: cifras(e, n))}"
        else:
            tex = f"{tex_nombre} \\approx {_tex_decimales(M, n or 4)}" + r"\quad\text{(" + _LARGA + ")}"
            texto = f"{nombre}  ({dims(M)}) ≈\n{_plano(M, lambda e: cifras(e, n or 4))}\n({_LARGA})"
        tex += rf"\qquad ({M.rows}\times {M.cols})"
    else:
        M = sp.sympify(M)
        tex, texto = f"{tex_nombre} = {_latex(M)}", f"{nombre} = {_txt(M)}"
        if n is not None and M.is_number and not M.is_integer:
            tex += rf" \approx {_tex_cifra(cifras(M, n))}"
            texto += f" ≈ {cifras(M, n)}"
    _emite(_Formula(tex, texto))

In [ ]:
# Utilidades de esta unidad: tablas con un elemento por renglón, unidades, eliminación exacta, gráficas, pruebas
# y lectura de listas de datos
def _es_cero(e):
    """True si e vale 0. Un racional se compara directo; un número exacto (con raíces, senos o cosenos) se evalúa con
    50 cifras, sin simplificarlo (simplificar radicales anidados puede tardar minutos); una expresión con símbolos se
    simplifica."""
    e = sp.sympify(e)
    if e.is_Rational:
        return e == 0
    if e.is_number:                                   # en decimales con 50 cifras: rápido y suficiente
        return bool(abs(sp.N(e, 50)) < sp.Float("1e-40", 50))
    return sp.simplify(e) == 0


def _simplifica_uno(e):
    """Una entrada simplificada. Un número que vale 0, 1 o -1 (comprobado con 50 cifras) se escribe así; uno con senos o
    cosenos, o muy largo (más de 40 operaciones), se deja como está: simplify lo reconstruiría y sympy podría
    desarrollar el coseno en radicales anidados, lo que puede tardar minutos."""
    if e.is_Rational:
        return e
    if e.is_number:
        for v in (0, 1, -1):
            if _es_cero(e - v):
                return sp.Integer(v)
        if e.has(sp.sin, sp.cos) or sp.count_ops(e) > 40:
            return e
    return sp.simplify(e)


def _simplifica(M):
    """Simplifica las entradas que no son racionales (las racionales se quedan como están), con _simplifica_uno."""
    return M.applyfunc(_simplifica_uno)


def _factor(a):
    """Coeficiente delante de un renglón: 1 -> '', 2 -> '2·', 1/2 -> '(1/2)·', -1 -> '(-1)·'."""
    if a == 1:
        return ""
    if a.is_integer and a > 0:
        return f"{a}·"
    return f"({a})·"


def _op_suma(i, m, k):
    """Texto de R_i ← R_i - m·R_k con el signo que toca: m = 2 -> 'R2 ← R2 - 2·R1'; m = -3 -> 'R2 ← R2 + 3·R1'."""
    if m.is_number and m.is_real and m < 0:
        return f"R{i} ← R{i} + {_factor(-m)}R{k}"
    return f"R{i} ← R{i} - {_factor(m)}R{k}"


def _elimina(M, n_a, con_b=True):
    """Eliminación de Gauss y después de Jordan, exacta, sobre una copia de M (la misma de las unidades 4 y 5).

    Los pivotes se buscan en las primeras n_a columnas; se intercambian renglones solo si el pivote es 0 y abajo hay
    una entrada distinta de cero (como a mano). Con con_b, al final se busca un pivote en la columna n_a (la de b):
    si aparece, hay un renglón 0 = c. Devuelve un diccionario: E (escalonada), R (reducida), pivotes (columnas con
    pivote, desde 0), libres, r (rango), r_aum (rango con la columna b), imposible ((renglón, c) o None) y pasos."""
    E = M.copy()
    racional = all(e.is_Rational for e in E)
    limpia = (lambda X: X) if racional else _simplifica
    m = E.rows
    pasos = [("fase", "Gauss: ceros debajo de cada pivote (forma escalonada)", None)]
    pivotes, libres, f, imposible = [], [], 0, None
    for c in range(n_a + (1 if con_b and M.cols > n_a else 0)):
        es_b = c == n_a
        k = next((i for i in range(f, m) if not _es_cero(E[i, c])), None)
        if k is None:
            if not es_b:
                libres.append(c)
            continue
        if k != f:
            E.row_swap(f, k)
            pasos.append(("op", f"R{f + 1} ↔ R{k + 1}", E.copy()))
        for i in range(f + 1, m):
            if not _es_cero(E[i, c]):
                mult = limpia(sp.Matrix([E[i, c] / E[f, c]]))[0]
                E[i, :] = limpia(E[i, :] - mult * E[f, :])
                E[i, c] = 0
                pasos.append(("op", _op_suma(i + 1, mult, f + 1), E.copy()))
        if es_b:
            imposible = (f + 1, E[f, c])
        else:
            pivotes.append(c)
        f += 1
    R = E.copy()
    pasos.append(("fase", "Jordan: cada pivote en 1 y ceros también arriba (forma escalonada reducida)", None))
    todos = list(enumerate(pivotes)) + ([(len(pivotes), n_a)] if imposible is not None else [])
    for f, c in reversed(todos):
        if not _es_cero(R[f, c] - 1):
            k = limpia(sp.Matrix([1 / R[f, c]]))[0]
            R[f, :] = limpia(k * R[f, :])
            R[f, c] = 1
            pasos.append(("op", f"R{f + 1} ← {_factor(k)}R{f + 1}", R.copy()))
        for i in range(f):
            if not _es_cero(R[i, c]):
                mult = R[i, c]
                R[i, :] = limpia(R[i, :] - mult * R[f, :])
                R[i, c] = 0
                pasos.append(("op", _op_suma(i + 1, mult, f + 1), R.copy()))
    return {"E": E, "R": R, "pivotes": pivotes, "libres": libres, "r": len(pivotes),
            "r_aum": len(pivotes) + (imposible is not None), "imposible": imposible, "pasos": pasos}


def _direcciones(R, pivotes, n):
    """Base de Nul(A) leída de la forma reducida R de A (sus primeras n columnas): un vector por variable libre,
    con esa libre en 1 y las demás libres en 0."""
    direcciones = []
    for c_libre in [c for c in range(n) if c not in pivotes]:
        d = sp.zeros(n, 1)
        d[c_libre] = 1
        for f, c in enumerate(pivotes):
            d[c] = -R[f, c_libre]
        direcciones.append(d)
    return direcciones


def base_nulo(M):
    """Base exacta de Nul(M) = {x : M x = 0} (lista vacía si solo contiene al 0)."""
    el = _elimina(M, M.cols, con_b=False)
    return _direcciones(el["R"], el["pivotes"], M.cols)


def _lista(cosas):
    """Números en una frase: [1, 2, 3] -> '1, 2 y 3'."""
    cosas = [str(c) for c in cosas]
    return cosas[0] if len(cosas) == 1 else ", ".join(cosas[:-1]) + " y " + cosas[-1]


def _coma(cosas):
    """Nombres separados con comas: ['v1', 'v2'] -> 'v1, v2'."""
    return ", ".join(str(c) for c in cosas)


def _flotante(M):
    """Matrix de sympy -> arreglo de numpy en punto flotante."""
    return np.array(sp.Matrix(M).evalf().tolist(), dtype=float)


def normaliza_texto(texto):
    """Minúsculas, sin acentos ni signos de puntuación y con un solo espacio entre palabras: '¿Dirección?' -> 'direccion'."""
    import unicodedata
    t = unicodedata.normalize("NFD", str(texto).lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    return " ".join(re.sub(r"[^a-z0-9ñ ]+", " ", t).split())


# Gráficas: un color, un estilo de línea y un marcador por elemento, en orden fijo, para que se lean también en
# blanco y negro
_COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
_ESTILOS = ["-", "--", ":", "-.", (0, (5, 1, 1, 1.5, 1, 1.5)), (0, (1, 4))]          # continuo, discontinuo, punteado...
_MARCAS = ["o", "s", "^", "D", "v", "P"]


def _flecha(ax, origen, punta, color, estilo, ancho, texto=None, desplazamiento=(6, 6)):
    """Flecha en el plano de origen a punta; con texto, su rótulo junto a la punta (sobre un fondo blanco)."""
    if np.hypot(*(np.asarray(punta, float) - np.asarray(origen, float))) > 1e-12:
        ax.annotate("", xy=punta, xytext=origen,
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=ancho, linestyle=estilo, shrinkA=0, shrinkB=0))
    if texto:
        ax.annotate(texto, xy=punta, xytext=desplazamiento, textcoords="offset points", fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))


def _captura(fn, *args, **kwargs):
    """Ejecuta fn sin mostrar nada (ni texto ni gráficas) y devuelve (texto, error): lo que imprimió o mostró y la
    excepción que lanzó (None si no lanzó). Sirve para las pruebas de humo de las calculadoras."""
    partes, buf = [], io.StringIO()

    def vacia():
        if buf.getvalue():
            partes.append(("texto", "", buf.getvalue()))
            buf.seek(0); buf.truncate()

    mostrar, error = plt.show, None
    _BLOQUES.append((partes, vacia))
    plt.show = lambda *a, **k: plt.close("all")
    try:
        with contextlib.redirect_stdout(buf):
            fn(*args, **kwargs)
    except Exception as err:
        error = err
    finally:
        vacia()
        _BLOQUES.pop()
        plt.show = mostrar
        plt.close("all")
    return "\n".join(texto for _, _, texto in partes), error


def _humo(calculadora, controles, **cambios):
    """Prueba de humo: la calculadora corre con los valores de sus controles (y los cambios dados), no lanza ninguna
    excepción, escribe algo y no dice «Revisa la entrada»."""
    valores = {k: w.value for k, w in controles.items()}
    valores.update(cambios)
    texto, error = _captura(calculadora, **valores)
    return error is None and texto.strip() != "" and "Revisa la entrada" not in texto


def evalua(pruebas):
    """Ejecuta una vez cada caso (nombre, función que devuelve True o False). Devuelve [(nombre, ok)]; un caso
    que lanza una excepción cuenta como falla y su nombre dice por qué, sin interrumpir a los demás."""
    resultados = []
    for nombre, prueba in pruebas:
        try:
            ok = bool(prueba())
        except Exception as err:
            ok, nombre = False, f"{nombre}  (error: {type(err).__name__}: {err})"
        resultados.append((nombre, ok))
    return resultados


def corre_pruebas(pruebas, seccion):
    """Escribe OK o FALLA en cada caso y el resumen de la sección."""
    resultados = evalua(pruebas)
    for nombre, ok in resultados:
        print("OK  " if ok else "FALLA", nombre)
    print(f"Sección {seccion}: {sum(ok for _, ok in resultados)}/{len(pruebas)} casos pasan")


def _cerca(a, b, tol=1e-9):
    """True si |a - b| <= tol (para comparar resultados en decimales en las pruebas)."""
    return abs(float(a) - float(b)) <= tol


_COND_AVISO = 1000               # número de condición a partir del cual se avisa que la matriz es casi singular


def _condicion(M, por_renglones=False):
    """Número de condición (en decimales) de una matriz exacta: grande si es casi singular; inf si es singular. Con
    por_renglones=True, cada renglón se divide antes entre su norma, para que renglones de escalas muy distintas (un
    renglón de unos y otro de porcentajes chicos) no den un número grande sin que la matriz sea casi singular."""
    F = _flotante(M)
    if por_renglones:
        normas = np.linalg.norm(F, axis=1)
        normas[normas == 0] = 1.0
        F = F / normas[:, None]
    with np.errstate(all="ignore"):
        c = float(np.linalg.cond(F))
    return c if math.isfinite(c) else math.inf


class Diagnostico(ValueError):
    """Un hallazgo físico legítimo (cables en un mismo plano, Z singular, ingredientes iguales...), no un error de
    formato: las calculadoras lo muestran como «Diagnóstico: ...», y los errores de formato como «Revisa la entrada: ...»."""


def _reporta(err):
    """Lo que escribe una calculadora cuando no puede calcular: «Diagnóstico: ...» si es un hallazgo físico y
    «Revisa la entrada: ...» si es un error en lo que se escribió."""
    print("Diagnóstico:" if isinstance(err, Diagnostico) else "Revisa la entrada:", err)


def _punto(u, v):
    """Producto interior exacto de dos vectores columna: u·v = uᵀv."""
    return (u.T * v)[0]


def _es_nulo(v):
    """True si v es el vector cero."""
    return all(_es_cero(e) for e in v)


def _valor(x, n=4):
    """Un valor exacto como texto, con sus decimales si no es entero: '2', '1/3 ≈ 0.3333', '6·√2 ≈ 8.485'. Si la forma
    exacta es muy larga, solo los decimales: '≈ 1.234 (aviso)'. Un valor decimal (de numpy) se redondea: '≈ 0.8'."""
    if isinstance(x, (float, np.floating)):
        return f"≈ {cifras(x, n)}"
    x = sp.sympify(x)
    if len(_txt(x)) > 28:
        return f"≈ {cifras(x, n)} ({_LARGA})"
    return _txt(x) + ("" if x.is_integer else f" ≈ {cifras(x, n)}")


def _eq(nombre, texto):
    """'nombre = texto', o 'nombre ≈ ...' si el texto ya es aproximado (empieza con '≈')."""
    return f"{nombre} {texto}" if texto.startswith("≈") else f"{nombre} = {texto}"


def _decimal_exacto(e, n=15):
    """El texto decimal EXACTO de un racional con decimal finito de a lo más n cifras significativas ('0.3', '2.5',
    '-80', '0.0199'), o None si no lo tiene (1/3, o más de n cifras)."""
    if isinstance(e, (int, np.integer)) and not isinstance(e, bool):
        e = sp.Integer(int(e))
    if not isinstance(e, sp.Rational):
        return None
    if e == 0:
        return "0"
    q = int(e.q)
    for f in (2, 5):
        while q % f == 0:
            q //= f
    if q != 1:
        return None
    with localcontext() as ctx:
        ctx.prec = 60
        d = (Decimal(int(e.p)) / Decimal(int(e.q))).normalize()
    if len(d.as_tuple().digits) > n or not -6 <= d.adjusted() <= 15:
        return None
    return format(d, "f")


def _dec_o_cifras(x, n=4):
    """Para rótulos de gráficas: el decimal exacto si es corto ('30', '0.8', '-2.5'); si no, redondeado a n cifras."""
    if isinstance(x, sp.Basic) and x.is_Rational:
        corto = _decimal_exacto(x, n)
        if corto is not None:
            return corto
    return cifras(x, n)


def _tupla_dec(v, n=4):
    """Vector para rótulos de gráficas: '(2, 0.5, 0)', '(0.7071, 0.7071, 0)'."""
    return "(" + ", ".join(_dec_o_cifras(sp.sympify(e), n) for e in v) + ")"


def _ntxt(e, n=15, cif=4):
    """Un valor exacto como texto, en decimal si su decimal es finito y corto ('0.3', '2.5') y si no, con fracciones
    y raíces ('1/3', '6·√2'). Si la forma exacta es muy larga, sus decimales redondeados a cif cifras con el aviso."""
    corto = _decimal_exacto(e, n)
    if corto is not None:
        return corto
    t = _txt(sp.sympify(e))
    return t if len(t) <= 28 else f"≈ {cifras(e, cif)} ({_LARGA})"


def _magnitud(nombre, x, unidad, n=4):
    """Una cantidad con su unidad: 'T1 = 30 N', 'i1 = 2.5 A' (decimal exacto de a lo más n cifras), 'T1 = 6·√2 N ≈ 8.485 N'
    o 'AB = 20/3 kN ≈ 6.667 kN' (decimales redondeados a n cifras). Si la forma exacta es muy larga o el valor viene de
    numpy: 'R ≈ 20.17 Ω'."""
    u = f" {unidad}" if unidad else ""
    if isinstance(x, (float, np.floating)):
        return f"{nombre} ≈ {cifras(x, n)}{u}"
    x = sp.sympify(x)
    corto = _decimal_exacto(x, n)
    if corto is not None:
        return f"{nombre} = {corto}{u}"
    if len(_txt(x)) > 28:
        return f"{nombre} ≈ {cifras(x, n)}{u} ({_LARGA})"
    return f"{nombre} = {_txt(x)}{u} ≈ {cifras(x, n)}{u}"


def _combinacion(coefs, letra):
    """Lado izquierdo de una ecuación: '6·i1 - 4·i2', '0.46·x1 + 0.18·x2'. Solo los términos con coeficiente distinto
    de cero; el coeficiente se escribe siempre, también el 1."""
    partes = []
    for j, z in enumerate(coefs, 1):
        if z == 0:
            continue
        signo = ("-" if z < 0 else "") if not partes else ("- " if z < 0 else "+ ")
        partes.append(f"{signo}{_ntxt(abs(z))}·{letra}{j}")
    return " ".join(partes) if partes else "0"


def _cantidad(x, unidad, n=4):
    """Un valor con su unidad, sin nombre: '30 N', '6·√2 N ≈ 8.485 N', '≈ 20.17 Ω'."""
    t = _magnitud("", x, unidad, n)
    return t[3:] if t.startswith(" = ") else t[1:]


def _cuenta(k, singular, plural):
    """'1 dato', '5 datos': el número con el sustantivo en singular o en plural."""
    return f"{k} {singular if k == 1 else plural}"


def _aviso_coma(*textos):
    """Aviso (aunque no haya error) si la entrada tiene comas entre dígitos y números separados solo por espacios,
    que es como se escriben los decimales con coma: la calculadora leyó cada coma como separador."""
    if any(pista_coma(t) for t in textos):
        print("Aviso: hay comas entre dígitos (1,5) y números separados solo por espacios. Si escribiste decimales "
              "con coma, usa punto decimal (1.5, no 1,5): la calculadora leyó cada coma como separador.")


def nombre_unidad(texto, que, ejemplo):
    """La unidad que escribe el alumno (N, kN, kg, L, s...), sin espacios a los lados. Vacía o muy larga: ValueError."""
    t = str(texto if texto is not None else "").strip()
    if not t:
        raise ValueError(f"escribe la unidad de {que}, por ejemplo {ejemplo}")
    if len(t) > 12:
        raise ValueError(f"la unidad de {que} es muy larga (escribiste «{t}»): usa un símbolo corto, por ejemplo {ejemplo}")
    return t


def _sin_articulo(cosa):
    """'la rama' -> 'rama', 'el nodo' -> 'nodo'."""
    partes = cosa.split(" ", 1)
    return partes[1] if len(partes) == 2 and partes[0] in ("el", "la") else cosa


def _indefinido(cosa):
    """'la rama' -> 'una rama', 'el nodo' -> 'un nodo' (el artículo viene con el sustantivo: concordancia de género)."""
    return ("una " if cosa.startswith("la ") else "un ") + _sin_articulo(cosa)


def _pista_decimal(texto):
    """La pista del punto decimal si hay una coma entre dos dígitos (3,6), para listas donde la coma separa datos."""
    if isinstance(texto, str) and re.search(r"\d,\d", texto):
        return "; si escribiste decimales, usa punto decimal: 3.6, no 3,6"
    return ""


def _filas_crudas(texto, nombre, cosa, ejemplo):
    """Texto o lista de Python -> lista de renglones, cada uno lista de entradas (texto o número), sin convertir.
    cosa lleva su artículo ('la rama', 'el nodo'), para que los mensajes concuerden en género."""
    if texto is None:
        raise ValueError(f"escribe {nombre} con {_indefinido(cosa)} por renglón, por ejemplo: {ejemplo}")
    if isinstance(texto, str):
        filas, vacia = _trozos(texto)
        if vacia is not None:
            raise ValueError(f"en {nombre}, el renglón {vacia} tiene una entrada vacía: revisa que no haya dos comas "
                             "seguidas ni una coma al principio o al final")
    elif isinstance(texto, sp.MatrixBase):
        filas = texto.tolist()
    elif isinstance(texto, np.ndarray):
        filas = np.atleast_2d(texto).tolist()
    elif isinstance(texto, (list, tuple)):
        filas = []
        for i, f in enumerate(texto, 1):
            if isinstance(f, str):
                sub, vacia = _trozos(f)
                if vacia is not None or len(sub) != 1:
                    raise ValueError(f"en {nombre}, {cosa} {i} debe ir en un solo renglón, por ejemplo: {ejemplo}")
                filas.append(sub[0])
            elif isinstance(f, (list, tuple, np.ndarray, sp.MatrixBase)):
                filas.append(list(f))
            else:
                raise ValueError(f"en {nombre}, escribe cada {_sin_articulo(cosa)} como una lista, por ejemplo: {ejemplo}")
    else:
        raise ValueError(f"escribe {nombre} con {_indefinido(cosa)} por renglón, por ejemplo: {ejemplo}")
    if not filas or not any(filas):
        raise ValueError(f"escribe {nombre} con {_indefinido(cosa)} por renglón, por ejemplo: {ejemplo}")
    return filas


def tabla(texto, nombre, cosa, columnas, maximo, ejemplo):
    """Lee una lista de elementos (anclajes, ramas, fuentes, ingredientes...) escrita UN ELEMENTO POR RENGLÓN o con
    los elementos separados por ';', con `columnas` números cada uno (None: todos con los mismos que el primero).
    También acepta una lista de Python [(2, 1, 0), (4, 1, 2)] (las ramas de la sección 2). Devuelve una lista de
    renglones de racionales exactos. cosa lleva su artículo ('la rama', 'el anclaje'). Los mensajes de error dicen qué
    elemento y qué número están mal."""
    filas = _filas_crudas(texto, nombre, cosa, ejemplo)
    if len(filas) > maximo:
        raise ValueError(f"{nombre}: escribiste {len(filas)} renglones y esta calculadora acepta hasta {maximo} "
                         f"(recuerda: cada renglón es {_indefinido(cosa)})")
    k = columnas if columnas is not None else len(filas[0])
    salida = []
    for i, f in enumerate(filas, 1):
        if len(f) != k:
            raise ValueError(f"en {nombre}, {cosa} {i} tiene {_cuenta(len(f), 'número', 'números')} y debe tener "
                             f"{k}, por ejemplo: {ejemplo}{pista_coma(texto) or _pista_decimal(texto)}")
        renglon = []
        for j, e in enumerate(f, 1):
            try:
                renglon.append(numero(e))
            except ValueError as err:
                raise ValueError(f"en {nombre}, {_sin_articulo(cosa)} {i}, número {j}: {err}") from None
        salida.append(renglon)
    return salida


def _entero(valor, donde, minimo, maximo):
    """Un número entero entre minimo y maximo (un número de malla o de nodo): ValueError claro si no lo es."""
    if not valor.is_integer or not minimo <= valor <= maximo:
        raise ValueError(f"{donde} debe ser un entero de {minimo} a {maximo} (escribiste {_txt(valor)})")
    return int(valor)


def _tupla_u(v, unidad, n=None):
    """Vector con su unidad: '(0.3, 0.4, 0) m' (decimales exactos), '(-80, 60, 0) N·m'; si alguna componente no tiene
    decimal finito y corto, en forma exacta y, con n, también sus decimales redondeados: '(5/3, 1/3) A ≈ (1.667, 0.3333) A'."""
    u = f" {unidad}" if unidad else ""
    if not _exacta_legible(v):
        return "≈ (" + ", ".join(cifras(e, n or 4) for e in v) + f"){u} ({_LARGA})"
    cortos = [_decimal_exacto(e, n or 15) for e in v]
    if all(c is not None for c in cortos):
        return "(" + ", ".join(cortos) + ")" + u
    exacto = "(" + ", ".join(_txt(e) for e in v) + ")" + u
    if n is None:
        return exacto
    return exacto + " ≈ (" + ", ".join(cifras(e, n) for e in v) + ")" + u


_MAX_DATOS = 500                 # datos de la sección 5: hasta 500 por lista
_MAX_EXACTO = 12                 # con hasta 12 ecuaciones (o datos) el cálculo es exacto; con más, en decimales


def _texto_dato(e):
    """Un dato como se escribió: el texto tal cual; un número de Python o de sympy, en su forma corta."""
    if isinstance(e, str):
        return e
    if isinstance(e, (float, np.floating)):
        return repr(float(e))
    if isinstance(e, sp.Basic):
        return _txt(e)
    return str(e)


def datos(texto, nombre="los datos", con_texto=False):
    """Lee una lista de números (separados por espacios, comas, ';' o saltos de línea) como lista de racionales
    exactos: '0 2 3.6' -> [0, 2, 18/5]. Acepta también una lista o un arreglo de números. Hasta 500 datos; los
    mensajes de error dicen qué dato está mal. Con con_texto=True devuelve también los datos como se escribieron."""
    ejemplo = f"escribe {nombre} como lista de números separados por espacios o comas, por ejemplo: 0 1 2 3"
    if texto is None:
        raise ValueError(ejemplo)
    if isinstance(texto, str):
        filas, vacia = _trozos(texto)
        if vacia is not None:
            raise ValueError(f"en {nombre} hay un dato vacío: revisa que no haya dos comas seguidas ni una coma al "
                             "principio o al final")
        entradas = [e for fila in filas for e in fila]
    elif isinstance(texto, np.ndarray):
        entradas = list(texto.ravel())
    elif isinstance(texto, sp.MatrixBase):
        entradas = list(texto)
    elif isinstance(texto, (list, tuple)):
        if any(isinstance(e, (list, tuple, np.ndarray, sp.MatrixBase)) for e in texto):
            raise ValueError(f"{nombre} debe ser una lista de números, sin listas dentro")
        entradas = list(texto)
    else:
        entradas = [texto]                              # un número solo
    if not entradas:
        raise ValueError(ejemplo)
    if len(entradas) > _MAX_DATOS:
        raise ValueError(f"{nombre} son {len(entradas)} números; esta calculadora acepta hasta {_MAX_DATOS}")
    valores = []
    for j, e in enumerate(entradas, 1):
        try:
            valores.append(numero(e))
        except ValueError as err:
            raise ValueError(f"en {nombre}, dato {j}: {err}") from None
    return (valores, [_texto_dato(e) for e in entradas]) if con_texto else valores


def _num(c, n=4):
    """Un número para tablas y ecuaciones: si es un racional exacto con decimal finito de a lo más n cifras
    significativas, ese decimal tal cual (10.02, 1.5, -0.2, 3); si no, redondeado a n cifras con cifras()."""
    if isinstance(c, (int, np.integer)) and not isinstance(c, bool):
        c = sp.Integer(int(c))
    if isinstance(c, sp.Rational):
        q = int(c.q)
        for f in (2, 5):
            while q % f == 0:
                q //= f
        if q == 1:
            with localcontext() as ctx:
                ctx.prec = 60
                d = (Decimal(int(c.p)) / Decimal(int(c.q))).normalize()
            if d == 0:
                return "0"
            if len(d.as_tuple().digits) <= n and -4 <= d.adjusted() <= 15:
                return format(d, "f")
    return cifras(c, n)


def muestra_decimales(nombre, M, n=4, tex=None):
    """Muestra una matriz o un vector de punto flotante (de numpy), redondeado a n cifras significativas.
    Un vector (arreglo de una dimensión) se muestra como columna."""
    M = np.asarray(M, dtype=float)
    M = M.reshape(-1, 1) if M.ndim == 1 else M
    S = sp.Matrix(M.tolist())
    tex_nombre = tex if tex is not None else _a_tex(nombre)
    _emite(_Formula(f"{tex_nombre} \\approx {_tex_decimales(S, n)}\\qquad ({S.rows}\\times {S.cols})",
                    f"{nombre}  ({dims(S)}) ≈\n{_plano(S, lambda e: cifras(e, n))}"))

## 1. Equilibrio de fuerzas en 3D, momento y trabajo

**Fuerza a lo largo de un cable.** Un cable que va del nudo $P$ al anclaje $Q$ jala al nudo hacia $Q$. Si su tensión es $T$, la fuerza que hace sobre el nudo es

$$\mathbf F=T\,\mathbf u,\qquad \mathbf u=\frac{Q-P}{\|Q-P\|}\quad(\text{el vector unitario de }P\text{ hacia }Q).$$

**Equilibrio.** Una partícula (un nudo donde se juntan cables) está en equilibrio si $\sum\mathbf F=\mathbf 0$: en 3D son tres ecuaciones, una por componente. Un cuerpo rígido, que además puede girar, necesita $\sum\mathbf F=\mathbf 0$ y $\sum\mathbf M=\mathbf 0$: seis ecuaciones.

**Tres cables y una carga.** Si en el nudo actúa la carga $\mathbf W$ y los vectores unitarios de los cables son las columnas de $U=[\,\mathbf u_1\ \mathbf u_2\ \mathbf u_3\,]$,

$$T_1\mathbf u_1+T_2\mathbf u_2+T_3\mathbf u_3+\mathbf W=\mathbf 0\qquad\Longleftrightarrow\qquad U\mathbf T=-\mathbf W .$$

- Un cable solo jala. Si alguna tensión sale negativa, ese cable tendría que empujar: **con cables, esa geometría no sostiene la carga**, aunque el sistema tenga solución. Si en lugar de cables son barras o patas (un trípode, por ejemplo), que sí pueden empujar, una tensión negativa es una **compresión** y la geometría sí sostiene la carga.
- Si los tres cables están en un mismo plano, $U$ es singular ($\det U=0$): no hay solución única, y una carga fuera de ese plano no se puede equilibrar.

**Momento.** El momento respecto a un punto $O$ de una fuerza $\mathbf F$ aplicada en $P$ es

$$\mathbf M_O=\mathbf r\times\mathbf F,\qquad \mathbf r=P-O\quad(\text{del punto }O\text{ al punto de aplicación}).$$

El orden importa: $\mathbf F\times\mathbf r=-\mathbf M_O$. $\mathbf M_O$ es perpendicular a $\mathbf r$ y a $\mathbf F$ (es el eje alrededor del cual la fuerza tiende a hacer girar), y su norma es la fuerza por el brazo de palanca: $\|\mathbf M_O\|=\|\mathbf F\|\,d$, con $d$ la distancia de $O$ a la línea de acción de $\mathbf F$.

**Trabajo.** Una fuerza constante $\mathbf F$ a lo largo de un desplazamiento recto $\mathbf d$ hace el trabajo $W=\mathbf F\cdot\mathbf d=\|\mathbf F\|\,\|\mathbf d\|\cos\theta$ (aquí $W$ es el trabajo, no la carga): en J si $\mathbf F$ está en N y $\mathbf d$ en m. Solo cuenta la componente de la fuerza a lo largo del desplazamiento; una fuerza perpendicular al desplazamiento no hace trabajo.

In [ ]:
def _vector3(p, nombre):
    """Un vector de R³ (tres componentes x, y, z) como columna de racionales exactos."""
    v = vector(p, nombre)
    if v.rows != 3:
        raise ValueError(f"{nombre} debe tener tres componentes, x, y, z (escribiste {_cuenta(v.rows, 'componente', 'componentes')})")
    return v


def vector_unitario(P, Q):
    """u = (Q - P)/‖Q - P‖, exacto (con raíces si hace falta): la dirección de P hacia Q, de longitud 1. Si P = Q,
    ValueError «P y Q son el mismo punto»."""
    P, Q = vector(P, "P"), vector(Q, "Q")
    if P.rows != Q.rows:
        raise ValueError(f"P tiene {_cuenta(P.rows, 'componente', 'componentes')} y Q tiene {Q.rows}: deben tener el "
                         "mismo número de componentes")
    d = Q - P
    if _es_nulo(d):
        raise ValueError("P y Q son el mismo punto: no definen ninguna dirección (‖Q - P‖ = 0)")
    return d / sp.sqrt(_punto(d, d))


def _anclajes(anclajes):
    """Los anclajes como lista de columnas exactas de R³ (uno por renglón)."""
    filas = tabla(anclajes, "los anclajes", "el anclaje", 3, 12, "2, 2, 1 (las coordenadas x, y, z del anclaje)")
    return [sp.Matrix(f) for f in filas]


def equilibrio_particula(anclajes, carga, nudo=(0, 0, 0), n_cifras=4):
    """Equilibrio de un nudo sostenido por TRES cables: resuelve U T = -W, exacto. anclajes: los tres puntos Q1, Q2,
    Q3 (uno por renglón); carga: el vector W que actúa en el nudo; nudo: el punto P donde se juntan los cables.
    Diccionario con U (vectores unitarios como columnas, exactos), tensiones (lista exacta), todas_positivas (True si
    las tres son > 0), sostiene (True si ninguna es negativa), avisos (por ejemplo, «el cable 3 tendría que empujar»),
    comprobacion (T1·u1 + T2·u2 + T3·u3 + W, que debe ser el vector cero), equilibrio (True si lo es), det_U,
    longitudes, direcciones (Qj - P), anclajes, W, nudo y cond (número de condición de U). Si los tres cables están en
    un mismo plano, Diagnostico (un ValueError) «los cables están en un mismo plano: con esta geometría no hay solución
    única». n_cifras: las cifras de los decimales en los avisos."""
    Q = _anclajes(anclajes)
    if len(Q) != 3:
        raise ValueError(f"escribiste {_cuenta(len(Q), 'anclaje', 'anclajes')}: este planteamiento usa exactamente tres "
                         "cables, porque en 3D hay tres ecuaciones (una por componente) y U debe ser cuadrada")
    W = _vector3(carga, "la carga")
    N = _vector3(nudo, "el nudo")
    D = [q - N for q in Q]
    for j, d in enumerate(D, 1):
        if _es_nulo(d):
            raise ValueError(f"el anclaje {j} coincide con el nudo: ese cable no tiene dirección")
    V = sp.Matrix.hstack(*D)                   # direcciones sin normalizar: entradas racionales
    det_V = V.det()
    if det_V == 0:
        raise Diagnostico("los cables están en un mismo plano: con esta geometría no hay solución única "
                          "(U es singular, det U = 0)")
    L = [sp.sqrt(_punto(d, d)) for d in D]
    U = sp.Matrix.hstack(*[d / l for d, l in zip(D, L)])
    q = V.LUsolve(-W)                          # V q = -W con q_j = T_j/L_j: se resuelve con fracciones, sin raíces
    T = [q[j] * L[j] for j in range(3)]
    suma = _simplifica(W + sum((T[j] * U[:, j] for j in range(3)), sp.zeros(3, 1)))
    avisos = []
    for j, t in enumerate(T, 1):
        if t < 0:
            avisos.append(f"el cable {j} tendría que empujar (T{j} < 0): un cable solo jala, así que con cables esta "
                          "geometría no sostiene la carga; si son barras o patas (pueden empujar), una tensión negativa "
                          "es compresión y la geometría sí sostiene la carga")
        elif t == 0:
            avisos.append(f"el cable {j} queda flojo (T{j} = 0): con esta carga no trabaja")
    cond = _condicion(U)
    if cond > _COND_AVISO:
        avisos.append(f"los cables están casi en un mismo plano (número de condición de U ≈ {cifras(cond, n_cifras)}): "
                      "las tensiones pueden salir muy grandes y cambian mucho si un anclaje se mueve un poco")
    return {"U": U, "tensiones": T, "todas_positivas": all(t > 0 for t in T), "sostiene": all(t >= 0 for t in T),
            "avisos": avisos, "comprobacion": suma, "equilibrio": _es_nulo(suma), "det_U": det_V / (L[0] * L[1] * L[2]),
            "longitudes": L, "direcciones": D, "anclajes": Q, "W": W, "nudo": N, "cond": cond}


def momento(F, P, O=(0, 0, 0)):
    """Momento de la fuerza F aplicada en el punto P respecto al punto O: M = r × F con r = P - O, exacto."""
    F, P, O = _vector3(F, "F"), _vector3(P, "P"), _vector3(O, "O")
    return (P - O).cross(F)


def trabajo(F, d):
    """Trabajo de una fuerza constante F en el desplazamiento recto d: W = F·d, exacto."""
    F, d = vector(F, "F"), vector(d, "d")
    if F.rows != d.rows:
        raise ValueError(f"F tiene {_cuenta(F.rows, 'componente', 'componentes')} y d tiene {d.rows}: deben tener el "
                         "mismo número de componentes")
    return _punto(F, d)


def _par_txt(e):
    """Un número dentro de una cuenta: 0.3, 50, (-200), (1/3)."""
    t = _ntxt(e)
    return f"({t})" if (e < 0 or "/" in t or "√" in t) else t


def _informe_momento(F, P, O, n_cifras, uF="N", uL="m"):
    F, P, O = _vector3(F, "F"), _vector3(P, "P"), _vector3(O, "O")
    r = P - O
    M = r.cross(F)
    (rx, ry, rz), (Fx, Fy, Fz) = list(r), list(F)
    p = _par_txt
    print(f"r = P - O = {_tupla_u(r, uL)},  F = {_tupla_u(F, uF)}")
    print("M = r × F = (ry·Fz - rz·Fy,  rz·Fx - rx·Fz,  rx·Fy - ry·Fx)")
    print(f"  = ({p(ry)}·{p(Fz)} - {p(rz)}·{p(Fy)},  {p(rz)}·{p(Fx)} - {p(rx)}·{p(Fz)},  "
          f"{p(rx)}·{p(Fy)} - {p(ry)}·{p(Fx)})")
    print(f"  = {_tupla_u(M, uF + '·' + uL, n_cifras)}")
    perpendicular = _punto(M, r) == 0 and _punto(M, F) == 0
    print(_magnitud("‖M‖", sp.sqrt(_punto(M, M)), uF + "·" + uL, n_cifras) +
          ("" if _es_nulo(M) else (";  M·r = 0 y M·F = 0: M es perpendicular a r y a F" if perpendicular else
                                   ";  ¡M no es perpendicular a r y a F!")))
    return M


def _informe_equilibrio(res, unidad, n_cifras):
    N, W, U = res["nudo"], res["W"], res["U"]
    print(f"Nudo P = {_tupla_u(N, 'm')};  carga W = {_tupla_u(W, unidad)}")
    for j in range(3):
        d, l = res["direcciones"][j], res["longitudes"][j]
        print(f"Cable {j + 1}: Q{j + 1} - P = {_tupla_u(d, 'm')},  {_magnitud(f'‖Q{j + 1} - P‖', l, 'm', n_cifras)},  "
              f"u{j + 1} = {_tupla(U[:, j], n_cifras)}")
    muestra("U", U, n_cifras, tex=r"U=[\,\mathbf u_1\ \mathbf u_2\ \mathbf u_3\,]")
    print(_magnitud("det U", res["det_U"], "", n_cifras) + ";  es distinto de 0: los cables no están en un mismo plano")
    print(f"Equilibrio del nudo: T1·u1 + T2·u2 + T3·u3 + W = 0, es decir, U T = -W = {_tupla_u(-W, unidad)}")
    if all(_es_cero(e) for e in U.T * U - sp.eye(3)):
        print("Aquí UᵀU = I (los tres cables son perpendiculares entre sí), así que T = -UᵀW sin resolver nada.")
    print("Tensiones:")
    for j, t in enumerate(res["tensiones"], 1):
        print("  " + _magnitud(f"T{j}", t, unidad, n_cifras))
    if res["todas_positivas"]:
        print("Veredicto: los tres cables jalan (las tres tensiones son positivas): esta geometría sostiene la carga.")
    elif res["sostiene"]:
        print("Veredicto: esta geometría sostiene la carga, pero no todos los cables trabajan.")
    else:
        print("Veredicto: con cables, esta geometría NO sostiene la carga (un cable no puede empujar). Si son barras o "
              "patas, que sí pueden empujar, las tensiones negativas son compresiones y la geometría sí la sostiene.")
    for aviso in res["avisos"]:
        print("Aviso:", aviso + ".")
    print("Comprobación: T1·u1 + T2·u2 + T3·u3 + W = " + _tupla_u(res["comprobacion"], unidad) +
          (" (la suma de fuerzas es cero)" if res["equilibrio"] else " (¡no es cero!)"))


def dibuja_cables(anclajes, carga, tensiones, nudo=(0, 0, 0), unidad="N", n_cifras=4):
    """Dibujo en 3D del nudo, los cables (un estilo de línea y un marcador por cable), la dirección unitaria de cada
    cable (flecha corta, no a escala), su tensión y la carga (flecha negra gruesa). tensiones puede ser None."""
    Q = _anclajes(anclajes)
    W, N = _vector3(carga, "la carga"), _vector3(nudo, "el nudo")
    Nf, Wf = _flotante(N).ravel(), _flotante(W).ravel()
    Qf = [_flotante(q).ravel() for q in Q]
    largo = max([float(np.linalg.norm(q - Nf)) for q in Qf] + [1e-12])
    fig = plt.figure(figsize=(7.4, 8.0))
    ax = fig.add_subplot(projection="3d")
    fig.subplots_adjust(bottom=0.2)
    puntos = [Nf] + Qf
    for j, q in enumerate(Qf):
        k = j % len(_COLORES)
        texto_T = ""
        if tensiones is not None:
            t = tensiones[j]
            texto_T = f"T{j + 1} = {_dec_o_cifras(t, n_cifras)} {unidad}" + ("  (empujaría)" if float(t) < 0 else "")
        ax.plot(*np.array([Nf, q]).T, linestyle=_ESTILOS[k], color=_COLORES[k], lw=2.2,
                label=f"cable {j + 1}" + (f": {texto_T}" if texto_T else ""))
        ax.scatter(*q, marker=_MARCAS[k], s=60, color=_COLORES[k], depthshade=False)
        d = q - Nf
        if np.linalg.norm(d) > 1e-12 * largo:
            u = d / np.linalg.norm(d)
            ax.quiver(*Nf, *(0.3 * largo * u), color=_COLORES[k], linewidth=2.8, arrow_length_ratio=0.3)
        ax.text(*q, f" Q{j + 1}", fontsize=9)
        if texto_T:                                   # la tensión, a media altura del cable (lejos de los ejes)
            ax.text(*(Nf + 0.6 * d), f" T{j + 1}", fontsize=9, color="black",
                    bbox=dict(boxstyle="round,pad=0.1", facecolor="white", edgecolor="none", alpha=0.75))
    nW = float(np.linalg.norm(Wf))
    if nW > 0:
        punta = Nf + 0.45 * largo * Wf / nW
        ax.quiver(*Nf, *(punta - Nf), color="black", linewidth=3.2, arrow_length_ratio=0.22)
        ax.plot([], [], "-", color="black", lw=3.2, label=f"carga W: ‖W‖ = {_dec_o_cifras(sp.sqrt(_punto(W, W)), n_cifras)} {unidad}")
        ax.text(*punta, f"  W", fontsize=10)
        puntos.append(punta)
    ax.scatter(*Nf, marker="o", s=70, color="black", depthshade=False)
    ax.text(*Nf, "  P (nudo)", fontsize=9)
    ax.plot([], [], "-", color="0.4", lw=2.8, label="flechas cortas: dirección unitaria de cada cable (no a escala)")
    Pm = np.array(puntos)
    centro = (Pm.max(axis=0) + Pm.min(axis=0)) / 2
    radio = max(float((Pm.max(axis=0) - Pm.min(axis=0)).max()) / 2, 1e-9) * 1.15
    ax.set_xlim(centro[0] - radio, centro[0] + radio)
    ax.set_ylim(centro[1] - radio, centro[1] + radio)
    ax.set_zlim(centro[2] - radio, centro[2] + radio)
    ax.set_box_aspect((1, 1, 1))                          # misma escala en los tres ejes
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_zlabel("z (m)")
    ax.view_init(elev=20, azim=-60)
    ax.set_title("Nudo en equilibrio: cables, direcciones unitarias y carga", fontsize=10)
    fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncol=2, fontsize=8)
    plt.show()

**Recorrido paso a paso.** La lámpara del libro: cuelga de un nudo en el origen, sostenido por tres cables con anclajes en $(2,2,1)$, $(-2,1,2)$ y $(1,-2,2)$ m, y pesa 90 N, así que la carga es $\mathbf W=(0,0,-90)$ N. Cada cable mide 3 m, los vectores unitarios son $\tfrac13(2,2,1)$, $\tfrac13(-2,1,2)$ y $\tfrac13(1,-2,2)$, y resolver $U\mathbf T=-\mathbf W$ da $\mathbf T=(30,\,60,\,60)$ N: los tres cables jalan. Después, el momento de $\mathbf F=(0,0,-200)$ N aplicada en $P=(0.3,\,0.4,\,0)$ m respecto al origen: $\mathbf M=\mathbf r\times\mathbf F=(-80,\,60,\,0)$ N·m, de norma $100=200\cdot0.5$ (la fuerza es vertical y el brazo de palanca es la distancia horizontal de $O$ a $P$, 0.5 m); con el orden invertido sale $\mathbf F\times\mathbf r=(80,\,-60,\,0)$. Al final, el trabajo de $\mathbf F=(40,30,0)$ N: en $\mathbf d=(5,0,0)$ m hace 200 J, y en $\mathbf d=(3,4,0)$ m, del mismo largo, 240 J, porque ahí la fuerza y el desplazamiento forman un ángulo menor.

In [ ]:
_lampara = equilibrio_particula("2, 2, 1; -2, 1, 2; 1, -2, 2", "0, 0, -90")
_informe_equilibrio(_lampara, "N", 4)
dibuja_cables(_lampara["anclajes"], _lampara["W"], _lampara["tensiones"], unidad="N")
print("\n" + "=" * 90 + "\n\nMomento de F = (0, 0, -200) N aplicada en P = (0.3, 0.4, 0) m, respecto al origen:")
_M = _informe_momento("0, 0, -200", "0.3, 0.4, 0", "0, 0, 0", 4)
_brazo = sp.sqrt(sp.Rational(3, 10) ** 2 + sp.Rational(2, 5) ** 2)
print(f"Brazo de palanca: F es vertical, así que es la distancia horizontal de O a P, √(0.3² + 0.4²) = "
      f"{_ntxt(_brazo)} m;  ‖F‖·d = 200·{_ntxt(_brazo)} = {_ntxt(200 * _brazo)} N·m = ‖M‖")
print(f"Con el orden invertido: F × r = {_tupla_u(vector('0, 0, -200').cross(vector('0.3, 0.4, 0')), 'N·m')} = -M")
print("\n" + "=" * 90 + "\n\nTrabajo de F = (40, 30, 0) N:")
for _d in ("5, 0, 0", "3, 4, 0"):
    _Wt = trabajo("40, 30, 0", _d)
    print(f"  en d = {_tupla_u(vector(_d), 'm')}: W = F·d = {_txt(_Wt)} J;  cos θ = W/(‖F‖·‖d‖) = {_txt(_Wt)}/(50·5) = "
          f"{_ntxt(_Wt / 250)}")
print(f"  con d = (0, 0, 2) m, perpendicular a F: W = {_txt(trabajo('40, 30, 0', '0, 0, 2'))} J")

**Calculadora.** Escribe los **tres anclajes**, uno por renglón, como `x, y, z` en metros (el nudo está en el origen), la carga $\mathbf W$ que actúa en el nudo y la unidad de fuerza. La calculadora arma $U$, resuelve $U\mathbf T=-\mathbf W$, dice si la geometría sostiene la carga y dibuja los cables. Si en lugar de cables son barras o patas (pueden empujar), una tensión negativa es una compresión y la geometría sí sostiene la carga. Prueba mover un anclaje en horizontal hasta que, vistos desde arriba, dos cables queden alineados (por ejemplo, el anclaje 2 en `-2, -2, 2`: sale $\mathbf T=(90,\ 60\sqrt3,\ 0)$ N y el cable 3 queda flojo), poner los tres anclajes a la misma altura que el nudo (los cables quedan en un plano) y los anclajes `1, 0, 1`, `-1, 0, 1`, `0, 1, 1` con la carga `0, 2, -10`.

In [ ]:
def calculadora_equilibrio(anclajes_txt, carga_txt, unidad, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        try:
            u = nombre_unidad(unidad, "fuerza", "N o kN")
            res = equilibrio_particula(anclajes_txt, carga_txt, n_cifras=n_cifras)
        except Diagnostico as err:            # cables en un plano: se dice y se dibuja la geometría, sin tensiones
            _reporta(err)
            res = None
        except ValueError as err:
            _reporta(err); return
        _aviso_coma(anclajes_txt, carga_txt)
        if res is not None:
            _informe_equilibrio(res, u, n_cifras)
    if res is None:
        dibuja_cables(anclajes_txt, carga_txt, None, unidad=u, n_cifras=n_cifras)
    else:
        dibuja_cables(res["anclajes"], res["W"], res["tensiones"], unidad=u, n_cifras=n_cifras)


_CONTROLES_1 = dict(
    anclajes_txt=widgets.Textarea(value="2, 2, 1\n-2, 1, 2\n1, -2, 2", description="anclajes (m):",
                                  continuous_update=False, layout=widgets.Layout(width="320px", height="100px"),
                                  style={"description_width": "initial"}),
    carga_txt=widgets.Text(value="0, 0, -90", description="carga W =", continuous_update=False),
    unidad=widgets.Text(value="N", description="unidad de fuerza:", continuous_update=False,
                        style={"description_width": "initial"}),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_equilibrio, **_CONTROLES_1);

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
_r2 = sp.sqrt(2)
PRUEBAS_1 = [
    ("vector_unitario((0, 0, 0), (2, 2, 1)) = (2/3, 2/3, 1/3)",
     lambda: tuple(vector_unitario((0, 0, 0), (2, 2, 1))) == (sp.Rational(2, 3), sp.Rational(2, 3), sp.Rational(1, 3))),
    ("lámpara: anclajes (2, 2, 1), (-2, 1, 2), (1, -2, 2) y carga (0, 0, -90) → T = (30, 60, 60) exactas, todas "
     "positivas y suma de fuerzas cero",
     lambda: (lambda r: r["tensiones"] == [30, 60, 60] and r["todas_positivas"] and r["equilibrio"]
                        and r["comprobacion"] == sp.zeros(3, 1) and not r["avisos"])(
         equilibrio_particula("2, 2, 1; -2, 1, 2; 1, -2, 2", "0, 0, -90"))),
    ("anclajes (1, 0, 1), (-1, 0, 1), (0, 1, 1) con carga (0, 2, -10) → T = (6√2, 6√2, -2√2) y el aviso del cable 3",
     lambda: (lambda r: [sp.simplify(t - e) for t, e in zip(r["tensiones"], (6 * _r2, 6 * _r2, -2 * _r2))] == [0, 0, 0]
                        and not r["todas_positivas"] and not r["sostiene"] and r["equilibrio"]
                        and len(r["avisos"]) == 1 and r["avisos"][0].startswith("el cable 3 tendría que empujar"))(
         equilibrio_particula([(1, 0, 1), (-1, 0, 1), (0, 1, 1)], (0, 2, -10)))),
    ("anclajes coplanares (1, 0, 1), (-1, 0, 1), (2, 0, 1) → ValueError «los cables están en un mismo plano»",
     lambda: _mensaje(lambda: equilibrio_particula("1, 0, 1; -1, 0, 1; 2, 0, 1", "0, 0, -10")).startswith(
         "los cables están en un mismo plano: con esta geometría no hay solución única")),
    ("momento((0, 0, -200), (0.3, 0.4, 0)) = (-80, 60, 0), con M·r = 0 y M·F = 0",
     lambda: (lambda M, r, F: M == vector("-80, 60, 0") and _punto(M, r) == 0 and _punto(M, F) == 0)(
         momento((0, 0, -200), (0.3, 0.4, 0)), vector("0.3, 0.4, 0"), vector("0, 0, -200"))),
    ("momento respecto a O = (1, 0, 0) de F = (0, 10, 0) aplicada en P = (3, 0, 0) → (0, 0, 20) (y no el del origen, "
     "(0, 0, 30))",
     lambda: momento((0, 10, 0), (3, 0, 0), (1, 0, 0)) == vector("0, 0, 20")
             and momento((0, 10, 0), (3, 0, 0)) == vector("0, 0, 30")),
    ("trabajo((40, 30, 0), (5, 0, 0)) = 200 y con d = (3, 4, 0) = 240; con una fuerza perpendicular a d, 0",
     lambda: trabajo((40, 30, 0), (5, 0, 0)) == 200 and trabajo((40, 30, 0), (3, 4, 0)) == 240
             and trabajo((40, 30, 0), (-3, 4, 7)) == 0),
    ("vector_unitario((1, 1, 1), (1, 1, 1)) → ValueError «P y Q son el mismo punto»",
     lambda: _mensaje(lambda: vector_unitario((1, 1, 1), (1, 1, 1))).startswith("P y Q son el mismo punto")),
    ("prueba de humo: la calculadora corre con sus valores iniciales y con un cable que empujaría, sin error y sin "
     "«Revisa la entrada»",
     lambda: _humo(calculadora_equilibrio, _CONTROLES_1)
             and _humo(calculadora_equilibrio, _CONTROLES_1, anclajes_txt="1, 0, 1\n-1, 0, 1\n0, 1, 1",
                       carga_txt="0, 2, -10", unidad="kN", n_cifras=6)),
]
corre_pruebas(PRUEBAS_1, 1)

**Contrasta (sección 1).** Calcula a mano el momento respecto al origen de $\mathbf F=(0,50,0)$ N aplicada en $(0.2,\,0,\,0)$ m: primero $\mathbf r$, después $\mathbf r\times\mathbf F$ componente por componente. Escribe `mi_momento = (Mx, My, Mz)` en N·m.

In [ ]:
mi_momento = None     # tres números (Mx, My, Mz) en N·m, por ejemplo (1, -2, 0.5)

if mi_momento is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mio = vector(mi_momento, "mi_momento")
    except ValueError as err:
        print(f"Revisa tu respuesta: {str(err).rstrip('.')}.")
    else:
        if mio.rows != 3:          # respuesta imposible: la referencia todavía no se muestra
            print(f"El momento es un vector de R³, con tres componentes (escribiste {mio.rows}): revisa tu respuesta y "
                  "vuelve a ejecutar la celda.")
        else:
            F_c, P_c = vector("0, 50, 0"), vector("0.2, 0, 0")          # los datos del enunciado
            ref = momento(F_c, P_c)
            print(f"La calculadora da: M = r × F = {_tupla_u(ref, 'N·m')}.")
            if all(abs(float(a - b)) <= 0.01 for a, b in zip(mio, ref)):
                print("Coinciden (con tolerancia de 0.01 N·m por componente).")
            elif all(abs(float(a + b)) <= 0.01 for a, b in zip(mio, ref)):
                print("NO coinciden: tu vector es el opuesto. Probablemente calculaste F × r; el momento es r × F, con r "
                      "del origen al punto de aplicación (el orden importa: F × r = -M).")
            else:
                print("NO coinciden: escribe r = (0.2, 0, 0) y F = (0, 50, 0) y aplica r × F = (ry·Fz - rz·Fy, "
                      "rz·Fx - rx·Fz, rx·Fy - ry·Fx).")
            print("\nEl procedimiento completo:")
            _informe_momento(F_c, P_c, (0, 0, 0), 4)

## 2. Análisis de mallas con leyes de Kirchhoff

**Leyes de Kirchhoff.** *Ley de voltajes:* al recorrer un lazo cerrado, la suma de las subidas y las caídas de voltaje es cero. *Ley de corrientes:* la corriente que entra a un nodo es igual a la que sale.

**Corrientes de malla.** A cada malla (un lazo que no encierra a otro) se le asigna una corriente $i_k$ en **sentido horario**. Con $b$ ramas y $n$ nodos hay $b-n+1$ mallas independientes. La ley de voltajes en cada malla da un sistema $Z\,\mathbf i=\mathbf V$:

- $Z_{ii}$ es la suma de las resistencias de la malla $i$, y $Z_{ij}=-$(la resistencia que comparten las mallas $i$ y $j$). Por eso $Z$ es simétrica.
- $V_i$ es la suma de las fuentes de la malla $i$: positiva si al recorrer la malla en el sentido de su corriente se pasa de $-$ a $+$ (el voltaje sube, la fuente empuja en ese sentido), negativa si se pasa de $+$ a $-$.
- Por una rama compartida pasa la **diferencia** de las dos corrientes de malla: $i_1-i_2$ en el sentido de la malla 1.
- Una corriente negativa no es un error: circula en sentido contrario al supuesto.

**Balance de potencia.** La potencia que disipan las resistencias, $\sum R\,i^2$, es igual a la que entregan las fuentes, $\sum V\,i$. Ojo: con las corrientes que salen del mismo sistema, el balance **siempre** cierra, porque $\mathbf i^TZ\,\mathbf i=\mathbf i^T\mathbf V$ es la misma ecuación multiplicada por $\mathbf i^T$. Por eso **no detecta un signo mal puesto en una fuente**: con el signo equivocado salen otras corrientes, y el balance también cierra.

**Análisis de nodos.** La alternativa es tomar como incógnitas los voltajes de los nodos respecto a tierra: con las conductancias $G=1/R$, la ley de corrientes en cada nodo da $G\,\mathbf v=\mathbf i_s$, con $G_{ii}$ = suma de las conductancias conectadas al nodo $i$, $G_{ij}=-$(la conductancia entre $i$ y $j$) e $\mathbf i_s$ las corrientes de las fuentes que entran a cada nodo.

En esta sección solo hay resistencias y fuentes independientes: de voltaje en el análisis de mallas y de corriente en el de nodos. **Cómo se escriben:** en mallas, cada rama es `R, malla, otra malla` (0 si la resistencia es solo de una malla) y cada fuente es `V, malla`, con $V$ positivo si empuja en el sentido horario de esa malla; una fuente en una rama compartida se escribe dos veces, `V, 1` y `-V, 2`. En nodos, cada rama es `R, nodo, otro nodo` (0 es tierra) y cada fuente es `I, nodo`, con $I$ positiva si entra al nodo.

In [ ]:
_MAX_MALLAS = 12                 # sistemas de hasta 12×12, exactos
_MAX_RAMAS = 30


def _ramas(ramas, que):
    """Las ramas (R, a, b) con R > 0, a de 1 a 12 y b de 0 a 12, b distinto de a. que: 'malla' o 'nodo'."""
    filas = tabla(ramas, "las ramas", "la rama", 3, _MAX_RAMAS,
                  "2, 1, 0 (R en ohms, " + ("malla y otra malla, o 0 si es solo de esa malla)" if que == "malla" else
                                            "nodo y otro nodo, o 0 si va a tierra)"))
    salida = []
    for j, (R, a, b) in enumerate(filas, 1):
        if R <= 0:
            raise ValueError(f"la rama {j} tiene R = {_ntxt(R)} Ω: una resistencia debe ser positiva (R > 0)")
        if que == "malla":
            a = _entero(a, f"en la rama {j}, el número de la malla", 1, _MAX_MALLAS)
            b = _entero(b, f"en la rama {j}, el número de la segunda malla (0 si no hay otra)", 0, _MAX_MALLAS)
            if a == b:
                raise ValueError(f"la rama {j} va de la malla {a} a la misma malla {a}: escribe dos mallas distintas (o 0 "
                                 "en la segunda si la resistencia es solo de esa malla)")
        else:
            a = _entero(a, f"en la rama {j}, el número del primer nodo (0 es tierra)", 0, _MAX_MALLAS)
            b = _entero(b, f"en la rama {j}, el número del segundo nodo (0 es tierra)", 0, _MAX_MALLAS)
            if a == b:
                raise ValueError(f"la rama {j} va del nodo {a} al mismo nodo {a}: escribe dos nodos distintos")
        salida.append((R, a, b))
    return salida


def _fuentes(fuentes, que, letra):
    """Las fuentes (valor, malla o nodo); una lista vacía o un texto vacío es «sin fuentes»."""
    if fuentes is None or (isinstance(fuentes, str) and not fuentes.strip()) or (
            isinstance(fuentes, (list, tuple)) and len(fuentes) == 0):
        return []
    filas = tabla(fuentes, "las fuentes", "la fuente", 2, _MAX_RAMAS,
                  f"10, 1 ({letra} y el número de {que})")
    return [(v, _entero(k, f"en la fuente {j}, el número de {que}", 1, _MAX_MALLAS)) for j, (v, k) in enumerate(filas, 1)]


def _textos_rama(rs, corrientes, n):
    """El texto de la corriente de cada rama, con su sentido; los decimales, con n cifras."""
    textos = []
    for j, ((R, a, b), c) in enumerate(zip(rs, corrientes), 1):
        cuenta = f"i{a}" if not b else f"i{a} - i{b}"
        lugar = f"solo en la malla {a}" if not b else f"compartida por las mallas {a} y {b}"
        valor = lambda x: _cantidad(x, "A", n)
        if c == 0:
            sentido = f"{cuenta} = 0 A: no circula corriente"
        elif c > 0:
            sentido = f"{cuenta} = {valor(c)}, en el sentido horario de la malla {a}"
        else:
            sentido = (f"{cuenta} = {valor(c)}: circulan {valor(-c)} " +
                       (f"en sentido antihorario de la malla {a}" if not b else f"en el sentido horario de la malla {b}") +
                       " (al revés de lo supuesto)")
        textos.append(f"rama {j} (R = {_ntxt(R, cif=n)} Ω, {lugar}): {sentido}")
    return textos


def mallas(ramas, fuentes):
    """Análisis de mallas, exacto. ramas: lista de (R, malla_a, malla_b), con malla_b = 0 si la resistencia es solo
    de malla_a (las mallas se numeran desde 1); fuentes: lista de (V, malla), con V positivo si empuja en el sentido
    horario de esa malla. Diccionario con Z, V, i (exactas), corrientes (la de cada rama, en el sentido horario de
    su primera malla), corrientes_rama (texto por resistencia, con su sentido), potencia_R (Σ R i²), potencia_fuentes
    (Σ V i), avisos y ramas, fuentes. Si R ≤ 0, ValueError; si Z es singular, Diagnostico (un ValueError) con la
    explicación."""
    rs, fs = _ramas(ramas, "malla"), _fuentes(fuentes, "malla", "V en volts")
    m = max([max(a, b) for _, a, b in rs] + [k for _, k in fs])
    Z, V = sp.zeros(m, m), sp.zeros(m, 1)
    for R, a, b in rs:
        Z[a - 1, a - 1] += R
        if b:
            Z[b - 1, b - 1] += R
            Z[a - 1, b - 1] -= R
            Z[b - 1, a - 1] -= R
    for v, k in fs:
        V[k - 1] += v
    vacias = [k + 1 for k in range(m) if Z[k, k] == 0]
    if vacias:
        raise ValueError(f"la malla {vacias[0]} no tiene ninguna resistencia: su renglón de Z es cero y Z es singular "
                         "(revisa la numeración de las mallas: deben ir de 1 en 1, sin saltarse ninguna)")
    if Z.det() == 0:
        raise Diagnostico("Z es singular: las ecuaciones de malla no determinan las corrientes (revisa que cada grupo de "
                         "mallas tenga al menos una resistencia que no sea compartida)")
    i = Z.LUsolve(V)
    corriente = lambda a, b: i[a - 1] - (i[b - 1] if b else 0)
    corrientes = [corriente(a, b) for _, a, b in rs]
    avisos = []
    if not fs or all(v == 0 for v, _ in fs):
        avisos.append("no hay fuentes (o todas valen 0): todas las corrientes son cero")
    return {"Z": Z, "V": V, "i": i, "corrientes": corrientes, "corrientes_rama": _textos_rama(rs, corrientes, 4),
            "potencia_R": sum(R * c ** 2 for (R, _, _), c in zip(rs, corrientes)),
            "potencia_fuentes": sum(v * i[k - 1] for v, k in fs), "avisos": avisos, "ramas": rs, "fuentes": fs}


def nodos(ramas, fuentes):
    """Análisis de nodos, exacto. ramas: lista de (R, nodo_a, nodo_b), con 0 = tierra; fuentes: lista de (I, nodo),
    con I positiva si entra al nodo. Diccionario con G (conductancias), I, v (voltajes de los nodos respecto a
    tierra, exactos), corrientes (la de cada rama, de nodo_a a nodo_b), ramas y fuentes. Si G es singular,
    ValueError."""
    rs, fs = _ramas(ramas, "nodo"), _fuentes(fuentes, "nodo", "I en amperes")
    if not any(a == 0 or b == 0 for _, a, b in rs):
        raise Diagnostico("ninguna rama llega a tierra (nodo 0): los voltajes no quedan determinados")
    m = max([max(a, b) for _, a, b in rs] + [k for _, k in fs])
    G, I = sp.zeros(m, m), sp.zeros(m, 1)
    for R, a, b in rs:
        g = 1 / R
        for k in (a, b):
            if k:
                G[k - 1, k - 1] += g
        if a and b:
            G[a - 1, b - 1] -= g
            G[b - 1, a - 1] -= g
    for c, k in fs:
        I[k - 1] += c
    sueltos = [k + 1 for k in range(m) if G[k, k] == 0]
    if sueltos:
        raise ValueError(f"el nodo {sueltos[0]} no está conectado a ninguna resistencia: su renglón de G es cero y G es "
                         "singular (revisa la numeración de los nodos)")
    if G.det() == 0:
        raise Diagnostico("G es singular: algún grupo de nodos no tiene camino a tierra por resistencias, así que sus "
                         "voltajes no quedan determinados")
    v = G.LUsolve(I)
    volt = lambda k: v[k - 1] if k else 0
    return {"G": G, "I": I, "v": v, "corrientes": [(volt(a) - volt(b)) / R for R, a, b in rs], "ramas": rs,
            "fuentes": fs}


def balance_potencia(Z, V, i):
    """Balance de potencia del análisis de mallas: potencia_R = iᵀZ i (la que disipan las resistencias) y
    potencia_fuentes = iᵀV (la que entregan las fuentes). cierra es True si son iguales."""
    Z, V, i = como_matriz(Z, "Z"), vector(V, "V"), vector(i, "i")
    if Z.rows != Z.cols:
        raise ValueError(f"Z debe ser cuadrada (una ecuación y una corriente por malla); es de {dims(Z)}")
    if V.rows != Z.rows or i.rows != Z.rows:
        raise ValueError(f"Z es de {dims(Z)}, V tiene {_cuenta(V.rows, 'número', 'números')} e i tiene {i.rows}: V e i "
                         "deben tener un número por malla")
    pR, pF = (i.T * Z * i)[0], (i.T * V)[0]
    return {"potencia_R": pR, "potencia_fuentes": pF, "cierra": pR == pF}


def _informe_mallas(res, n_cifras):
    Z, V, i = res["Z"], res["V"], res["i"]
    m = Z.rows
    print(f"{_cuenta(m, 'malla', 'mallas')}, {_cuenta(len(res['ramas']), 'rama', 'ramas')} con resistencia y "
          f"{_cuenta(len(res['fuentes']), 'fuente', 'fuentes')}. Corrientes de malla en sentido horario.")
    muestra("Z (Ω)", Z, n_cifras, tex=r"Z\ (\Omega)")
    muestra("V (V)", V, n_cifras, tex=r"\mathbf V\ (\mathrm V)")
    print("Ecuaciones de malla (ley de voltajes), Z i = V:")
    for k in range(m):
        print(f"  malla {k + 1}: {_combinacion(Z.row(k), 'i')} = {_ntxt(V[k], cif=n_cifras)}")
    print("Corrientes de malla:")
    for k in range(m):
        print("  " + _magnitud(f"i{k + 1}", i[k], "A", n_cifras) +
              ("  (negativa: circula en sentido antihorario)" if i[k] < 0 else ""))
    print("Corriente en cada rama:")
    for texto in _textos_rama(res["ramas"], res["corrientes"], n_cifras):
        print("  " + texto)
    pR, pF = res["potencia_R"], res["potencia_fuentes"]
    print(f"Balance de potencia: {_magnitud('Σ R·i²', pR, 'W', n_cifras)} y {_magnitud('Σ V·i', pF, 'W', n_cifras)}: "
          + ("cierra." if pR == pF else "¡NO cierra!"))
    for aviso in res["avisos"]:
        print("Aviso:", aviso + ".")


def _informe_nodos(res, n_cifras):
    G, I, v = res["G"], res["I"], res["v"]
    print(f"{_cuenta(G.rows, 'nodo', 'nodos')} además de tierra (0). Incógnitas: los voltajes de los nodos respecto a tierra.")
    muestra("G (S)", G, n_cifras, tex=r"G\ (\mathrm S)")
    muestra("I (A)", I, n_cifras, tex=r"\mathbf i_s\ (\mathrm A)")
    print("Voltajes de los nodos (G v = i_s):")
    for k in range(G.rows):
        print("  " + _magnitud(f"v{k + 1}", v[k], "V", n_cifras))
    print("Corriente en cada rama (de su primer nodo al segundo):")
    for j, ((R, a, b), c) in enumerate(zip(res["ramas"], res["corrientes"]), 1):
        tramo = (f"del nodo {a} " if a else "de tierra ") + (f"al nodo {b}" if b else "a tierra")
        print(f"  rama {j} (R = {_ntxt(R, cif=n_cifras)} Ω, {tramo}): "
              + _magnitud("i", c, "A", n_cifras) + ("  (circula al revés)" if c < 0 else ""))

**Recorrido paso a paso.** El circuito de la figura 6.3 del libro tiene dos mallas: $R_1=2\ \Omega$ solo en la malla 1, $R_3=4\ \Omega$ en la rama que comparten y $R_2=4\ \Omega$ solo en la malla 2, con la fuente $V_1=10$ V en la malla 1 (empuja en sentido horario). Con $V_2=0$:

$$Z=\begin{bmatrix}6&-4\\-4&8\end{bmatrix},\qquad \mathbf i=\left(\tfrac52,\ \tfrac54\right)\ \text{A},$$

lo mismo que da la reducción serie-paralelo: $2+4\parallel4=4\ \Omega$ y $10/4=2.5$ A. Con $V_2=4$ V en la malla 2 y el $+$ arriba, la fuente se opone al sentido horario de esa malla, así que entra como $-4$: $\mathbf V=(10,-4)$, $\mathbf i=(2,\ \tfrac12)$ A, y por $R_3$ pasan $i_1-i_2=1.5$ A en el sentido de la malla 1 (en la figura, hacia abajo). En la calculadora, $R_1$, $R_3$ y $R_2$ son las ramas 1, 2 y 3.

In [ ]:
_fig63 = [(2, 1, 0), (4, 1, 2), (4, 2, 0)]          # R1, R3 (compartida) y R2 de la figura 6.3
print("(1) V1 = 10 V y V2 = 0")
_m1 = mallas(_fig63, [(10, 1)])
_informe_mallas(_m1, 4)
_paralelo = sp.Integer(4) * 4 / (4 + 4)
print(f"Serie-paralelo: R = 2 + 4∥4 = 2 + {_ntxt(_paralelo)} = {_ntxt(2 + _paralelo)} Ω  →  i1 = 10/{_ntxt(2 + _paralelo)} = "
      f"{_ntxt(10 / (2 + _paralelo))} A, y se reparte por igual entre R3 y R2: {_ntxt(5 / sp.Integer(4))} A cada una.")
print("\n" + "=" * 90 + "\n\n(2) V1 = 10 V y V2 = 4 V con el + arriba (se opone al sentido horario de la malla 2)")
_m2 = mallas(_fig63, [(10, 1), (-4, 2)])
_informe_mallas(_m2, 4)

**El balance de potencia no detecta un signo mal puesto.** Si en el caso (2) escribes la fuente de la malla 2 con el signo equivocado, $+4$ en lugar de $-4$, el lado derecho cambia y salen otras corrientes; aun así, el balance cierra, porque se calcula con las mismas ecuaciones. Para encontrar un signo mal puesto hay que revisar el planteamiento (el sentido en que cada fuente empuja), no el balance.

In [ ]:
for _nombre, _fs in (("bien planteado, V = (10, -4)", [(10, 1), (-4, 2)]), ("signo equivocado, V = (10, 4)", [(10, 1), (4, 2)])):
    _r = mallas(_fig63, _fs)
    _b = balance_potencia(_r["Z"], _r["V"], _r["i"])
    print(f"{_nombre}:  i = {_tupla_u(_r['i'], 'A', 4)};  iᵀZ i = {_ntxt(_b['potencia_R'])} W,  "
          f"iᵀV = {_ntxt(_b['potencia_fuentes'])} W  →  {'cierra' if _b['cierra'] else 'no cierra'}")
print("Las dos versiones cierran el balance y dan corrientes distintas: el balance no dice cuál está bien.")

**Calculadora de mallas.** Escribe **una rama por renglón** como `R, malla, otra malla` (R en Ω; 0 en la otra malla si la resistencia es solo de la primera) y **una fuente por renglón** como `V, malla` (V en volts, positivo si empuja en el sentido horario de esa malla). Hasta 12 mallas. Los valores iniciales son el caso (2) del recorrido. Prueba cambiar el signo de la segunda fuente, agregar una tercera malla y escribir una resistencia de 0 Ω.

In [ ]:
def calculadora_mallas(ramas_txt, fuentes_txt, n_cifras):
    with bloque():                 # el resultado aparece completo, de una sola vez
        try:
            res = mallas(ramas_txt, fuentes_txt)
        except ValueError as err:
            _reporta(err); return
        _aviso_coma(ramas_txt, fuentes_txt)
        _informe_mallas(res, n_cifras)


_CONTROLES_2A = dict(
    ramas_txt=widgets.Textarea(value="2, 1, 0\n4, 1, 2\n4, 2, 0", description="ramas (R en Ω, malla, malla o 0):",
                               continuous_update=False, layout=widgets.Layout(width="420px", height="110px"),
                               style={"description_width": "initial"}),
    fuentes_txt=widgets.Textarea(value="10, 1\n-4, 2", description="fuentes (V en volts, malla):",
                                 continuous_update=False, layout=widgets.Layout(width="420px", height="80px"),
                                 style={"description_width": "initial"}),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_mallas, **_CONTROLES_2A);

**Calculadora de nodos.** Escribe **una rama por renglón** como `R, nodo, otro nodo` (R en Ω; 0 es tierra) y **una fuente de corriente por renglón** como `I, nodo` (I en amperes, positiva si entra al nodo). Con los valores iniciales, una fuente de 3 A entra al nodo 1, que tiene 4 Ω a tierra y 2 Ω hacia el nodo 2, y el nodo 2 tiene 4 Ω a tierra.

In [ ]:
def calculadora_nodos(ramas_txt, fuentes_txt, n_cifras):
    with bloque():
        try:
            res = nodos(ramas_txt, fuentes_txt)
        except ValueError as err:
            _reporta(err); return
        _aviso_coma(ramas_txt, fuentes_txt)
        _informe_nodos(res, n_cifras)


_CONTROLES_2B = dict(
    ramas_txt=widgets.Textarea(value="4, 1, 0\n2, 1, 2\n4, 2, 0", description="ramas (R en Ω, nodo, nodo o 0):",
                               continuous_update=False, layout=widgets.Layout(width="420px", height="110px"),
                               style={"description_width": "initial"}),
    fuentes_txt=widgets.Textarea(value="3, 1", description="fuentes (I en amperes, nodo):", continuous_update=False,
                                 layout=widgets.Layout(width="420px", height="80px"), style={"description_width": "initial"}),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_nodos, **_CONTROLES_2B);

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
_tres = mallas([(2, 1, 0), (3, 1, 2), (1, 2, 0), (2, 2, 3), (4, 3, 0)], [(12, 1)])
PRUEBAS_2 = [
    ("figura 6.3 con V2 = 0 → Z = [[6, -4], [-4, 8]] e i = (5/2, 5/4)",
     lambda: (lambda r: r["Z"] == sp.Matrix([[6, -4], [-4, 8]]) and r["i"] == vector("5/2, 5/4"))(
         mallas(_fig63, [(10, 1)]))),
    ("con las fuentes (10, 1) y (-4, 2) → i = (2, 1/2) y la corriente en R3 (rama 2) es 3/2",
     lambda: (lambda r: r["i"] == vector("2, 1/2") and r["corrientes"][1] == sp.Rational(3, 2)
                        and r["V"] == vector("10, -4"))(mallas("2, 1, 0\n4, 1, 2\n4, 2, 0", "10, 1\n-4, 2"))),
    ("tres mallas: ramas (2, 1, 0), (3, 1, 2), (1, 2, 0), (2, 2, 3), (4, 3, 0) y fuente (12, 1) → "
     "Z = [[5, -3, 0], [-3, 6, -2], [0, -2, 6]] e i = (192/53, 108/53, 36/53)",
     lambda: _tres["Z"] == sp.Matrix([[5, -3, 0], [-3, 6, -2], [0, -2, 6]])
             and _tres["i"] == sp.Matrix([sp.Rational(192, 53), sp.Rational(108, 53), sp.Rational(36, 53)])),
    ("Z es simétrica en las tres pruebas anteriores",
     lambda: all(r["Z"] == r["Z"].T for r in (mallas(_fig63, [(10, 1)]), mallas(_fig63, [(10, 1), (-4, 2)]), _tres))),
    ("balance de potencia: cierra con (10, 1), (-4, 2) y también con la fuente 2 de signo cambiado, con corrientes "
     "distintas",
     lambda: (lambda a, b: balance_potencia(a["Z"], a["V"], a["i"])["cierra"] and balance_potencia(b["Z"], b["V"], b["i"])["cierra"]
                           and a["potencia_R"] == a["potencia_fuentes"] and a["i"] != b["i"])(
         mallas(_fig63, [(10, 1), (-4, 2)]), mallas(_fig63, [(10, 1), (4, 2)]))),
    ("nodos([(4, 1, 0), (2, 1, 2), (4, 2, 0)], [(3, 1)]) → v = (36/5, 24/5)",
     lambda: nodos([(4, 1, 0), (2, 1, 2), (4, 2, 0)], [(3, 1)])["v"] == vector("36/5, 24/5")),
    ("una rama con R = 0 o negativa → ValueError; una malla sin resistencias → ValueError (Z singular)",
     lambda: "positiva" in _mensaje(lambda: mallas("0, 1, 0", "5, 1"))
             and "positiva" in _mensaje(lambda: mallas("2, 1, 0\n-3, 1, 2\n4, 2, 0", "5, 1"))
             and "singular" in _mensaje(lambda: mallas("2, 1, 0", "5, 1\n3, 2"))),
    ("prueba de humo: las dos calculadoras corren con sus valores iniciales, sin error y sin «Revisa la entrada»",
     lambda: _humo(calculadora_mallas, _CONTROLES_2A) and _humo(calculadora_nodos, _CONTROLES_2B)
             and _humo(calculadora_mallas, _CONTROLES_2A, fuentes_txt="10, 1\n4, 2", n_cifras=6)),
]
corre_pruebas(PRUEBAS_2, 2)

**Contrasta (sección 2).** Para el circuito de la figura 6.3 con $R_1=1\ \Omega$, $R_3=2\ \Omega$, $R_2=1\ \Omega$, $V_1=6$ V y $V_2=0$, escribe las dos ecuaciones de malla, resuélvelas a mano y escribe `mis_corrientes = (i1, i2)` en amperes.

In [ ]:
mis_corrientes = None     # dos números (i1, i2) en A, por ejemplo (1.2, -0.7) o ("3/2", 1)

if mis_corrientes is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mias = vector(mis_corrientes, "mis_corrientes")
    except ValueError as err:
        print(f"Revisa tu respuesta: {str(err).rstrip('.')}.")
    else:
        if mias.rows != 2:          # respuesta imposible: la referencia todavía no se muestra
            print(f"El circuito tiene dos mallas, así que son dos corrientes, i1 e i2 (escribiste {mias.rows}): revisa tu "
                  "respuesta y vuelve a ejecutar la celda.")
        else:
            ref = mallas([(1, 1, 0), (2, 1, 2), (1, 2, 0)], [(6, 1)])          # los datos del enunciado
            print(f"La calculadora da: i = {_tupla_u(ref['i'], 'A', 4)}.")
            if all(abs(float(a - b)) <= 0.01 for a, b in zip(mias, ref["i"])):
                print("Coinciden (con tolerancia de 0.01 A).")
            elif all(abs(float(a - b)) <= 0.01 for a, b in zip(mias, reversed(list(ref["i"])))):
                print("NO coinciden: probablemente invertiste el orden. i1 es la corriente de la malla con la fuente.")
            else:
                print("NO coinciden: en la malla 1, Z11 = R1 + R3 y Z12 = -R3; en la malla 2, Z22 = R3 + R2. El lado "
                      "derecho es (V1, 0).")
            print("\nEl procedimiento completo:")
            _informe_mallas(ref, 4)

## 3. Mezclas y asignación de recursos

**Mezclas.** Cada ingrediente es una **columna**: sus porcentajes de los componentes que se controlan. Las incógnitas $x_1,\dots,x_n$ son las cantidades de cada ingrediente, y hay una ecuación por el total y otra por cada componente controlado. Con dos componentes, por ejemplo,

$$\begin{aligned} x_1+x_2+\dots+x_n&=\text{total}\\ p_{11}x_1+p_{12}x_2+\dots+p_{1n}x_n&=\text{total}\cdot c_1\\ p_{21}x_1+p_{22}x_2+\dots+p_{2n}x_n&=\text{total}\cdot c_2\end{aligned}\qquad\Longleftrightarrow\qquad A\mathbf x=\mathbf b ,$$

con los porcentajes $p_{ij}$ y los objetivos $c_i$ escritos como fracciones (46 % es 0.46).

- **La condición física** es $\mathbf x\ge\mathbf 0$: no se puede usar una cantidad negativa. Una solución con una cantidad negativa resuelve el sistema, pero la mezcla es imposible con esos ingredientes.
- Con tantos ingredientes como ecuaciones (y composiciones independientes), la solución es **única**.
- Con más ingredientes que ecuaciones hay **infinitas** soluciones: las cantidades dependen de variables libres, y la mezcla se escoge por otro criterio, como el costo. Escoger la más barata con $\mathbf x\ge\mathbf 0$ es un problema de *programación lineal*, que verás en un curso posterior.
- Con tres ingredientes y dos componentes, cada ingrediente es un punto del plano de composiciones $(p_1,p_2)$, y las mezclas posibles forman el **triángulo** que tiene a los tres como vértices: una mezcla con un objetivo fuera del triángulo necesitaría una cantidad negativa.

**Asignación de recursos.** Si el producto $j$ usa $a_{ij}$ horas del recurso $i$ (una máquina, una línea de ensamble) y del recurso $i$ hay $b_i$ horas, el plan que usa **todo** lo disponible cumple $A\mathbf x=\mathbf b$. Si alguna cantidad sale negativa, no se puede usar todo exactamente: hay que dejar sin usar parte de algún recurso.

**Cómo se escriben.** Cada ingrediente va en un renglón con sus porcentajes, en el mismo orden de componentes que el objetivo: `46, 0` es un ingrediente con 46 % del componente 1 y 0 % del componente 2.

In [ ]:
_MAX_INGREDIENTES, _MAX_COMPONENTES = 8, 5
_t = sp.Symbol("t")


def _intervalo(p, d):
    """Los valores de t con p + t·d ≥ 0 en todas sus componentes: (inferior, superior), con -oo u oo si no hay
    límite, o 'vacío'."""
    lo, hi = -sp.oo, sp.oo
    for pi, di in zip(p, d):
        if di > 0:
            lo = max(lo, -pi / di)
        elif di < 0:
            hi = min(hi, -pi / di)
        elif pi < 0:
            return "vacío"
    return "vacío" if lo > hi else (lo, hi)


def _analiza(A, b):
    """A x = b, exacto: estado ('única', 'infinitas' o 'sin solución'), x (si es única), solucion_general (con los
    parámetros t, o t1, t2, ...), parametros, libres (columnas libres, desde 0), intervalo (con un solo parámetro) y
    rango."""
    n = A.cols
    el = _elimina(A.row_join(b), n, con_b=True)
    base = {"x": None, "solucion_general": None, "parametros": [], "libres": [], "intervalo": None, "rango": el["r"]}
    if el["imposible"] is not None:
        return dict(base, estado="sin solución")
    R, piv = el["R"], el["pivotes"]
    xp = sp.zeros(n, 1)
    for f, c in enumerate(piv):
        xp[c] = R[f, n]
    dirs = _direcciones(R, piv, n)
    if not dirs:
        return dict(base, estado="única", x=xp)
    params = [_t] if len(dirs) == 1 else [sp.Symbol(f"t{k}") for k in range(1, len(dirs) + 1)]
    general = xp + sum((p * d for p, d in zip(params, dirs)), sp.zeros(n, 1))
    return dict(base, estado="infinitas", solucion_general=general, parametros=params,
                libres=[c for c in range(n) if c not in piv],
                intervalo=_intervalo(xp, dirs[0]) if len(dirs) == 1 else None)


def _porcentajes(valores, donde):
    for j, v in enumerate(valores, 1):
        if not 0 <= v <= 100:
            raise ValueError(f"{donde}, el porcentaje {j} vale {_ntxt(v)}: un porcentaje va de 0 a 100")
    if sum(valores) > 100:
        raise ValueError(f"{donde}, los porcentajes suman {_ntxt(sum(valores))}: no pueden pasar de 100")


def _sin_porciento(texto):
    """'20 %, 10 %' -> '20 , 10 ': el signo % se ignora (los porcentajes se escriben como números)."""
    return texto.replace("%", " ") if isinstance(texto, str) else texto


def mezcla(composiciones, objetivo, total, n_cifras=4):
    """Mezcla exacta. composiciones: un ingrediente por renglón con sus porcentajes de los componentes controlados;
    objetivo: los porcentajes deseados; total: la cantidad de mezcla. Arma A (un renglón de unos y un renglón de
    fracciones por componente) y b. Diccionario con A, b, estado, x (exacta, si es única), factible (x ≥ 0),
    avisos, solucion_general (paramétrica en t, si hay más ingredientes que ecuaciones), parametros, libres e
    intervalo (con UN parámetro libre: los t que dejan todo ≥ 0, o 'vacío'). Si dos ingredientes tienen la misma
    composición, Diagnostico (un ValueError) «... no determinan la mezcla ...». Se acepta el signo % ('20 %, 10 %').
    n_cifras: las cifras de los decimales en los avisos."""
    composiciones, objetivo = _sin_porciento(composiciones), _sin_porciento(objetivo)
    filas = tabla(composiciones, "los ingredientes", "el ingrediente", None, _MAX_INGREDIENTES,
                  "46, 0 (el porcentaje de cada componente controlado)")
    n, k = len(filas), len(filas[0])
    if k > _MAX_COMPONENTES:
        raise ValueError(f"cada ingrediente tiene {k} porcentajes; esta calculadora controla hasta {_MAX_COMPONENTES} "
                         "componentes")
    for j, f in enumerate(filas, 1):
        _porcentajes(f, f"en el ingrediente {j}")
    c = vector(objetivo, "el objetivo")
    if c.rows != k:
        raise ValueError(f"el objetivo tiene {_cuenta(c.rows, 'porcentaje', 'porcentajes')} y cada ingrediente tiene {k}: "
                         "escribe un porcentaje deseado por cada componente controlado, en el mismo orden")
    _porcentajes(list(c), "en el objetivo")
    tot = numero(total, "el total")
    if tot <= 0:
        raise ValueError(f"el total debe ser positivo (escribiste {_ntxt(tot)})")
    for i in range(n):
        for j in range(i + 1, n):
            if filas[i] == filas[j]:
                raise Diagnostico(f"los ingredientes {i + 1} y {j + 1} tienen la misma composición, así que no determinan "
                                 "la mezcla: cualquier reparto entre ellos da lo mismo (solo cuenta su suma); quita uno")
    A = sp.Matrix([[1] * n] + [[filas[j][i] / 100 for j in range(n)] for i in range(k)])
    b = sp.Matrix([tot] + [tot * c[i] / 100 for i in range(k)])
    res = _analiza(A, b)
    avisos = []
    if res["estado"] == "sin solución":
        factible = False
        avisos.append("ninguna mezcla de estos ingredientes tiene esa composición: el sistema A x = b no tiene solución")
    elif res["estado"] == "única":
        x = res["x"]
        factible = all(v >= 0 for v in x)
        for j, v in enumerate(x, 1):
            if v < 0:
                avisos.append(f"el ingrediente {j} saldría negativo: no se puede usar una cantidad negativa, así que esa "
                              "mezcla es imposible con estos ingredientes")
        cond = _condicion(A, por_renglones=True)        # renglones escalados: los porcentajes chicos no dan falsas alarmas
        if A.rows == A.cols and cond > _COND_AVISO:
            avisos.append(f"las composiciones son casi dependientes (número de condición de A, con sus renglones "
                          f"escalados, ≈ {cifras(cond, n_cifras)}): las cantidades pueden cambiar mucho con un cambio "
                          "pequeño en los porcentajes")
        if not factible and k == 1:
            lo, hi = min(f[0] for f in filas), max(f[0] for f in filas)
            avisos.append(f"el objetivo, {_ntxt(c[0])} %, está fuera del intervalo [{_ntxt(lo)}, {_ntxt(hi)}] % que "
                          "cubren los ingredientes")
    else:
        iv = res["intervalo"]
        factible = (iv != "vacío") if iv is not None else None
        if res["rango"] < min(n, k + 1):
            avisos.append(f"las composiciones son dependientes (rango de A = {res['rango']}): unos ingredientes son "
                          "mezcla de otros, así que no determinan la mezcla")
        if len(res["parametros"]) > 1:
            avisos.append("con dos o más parámetros libres, decidir si hay una mezcla con todas las cantidades no "
                          "negativas y escoger la más barata es un problema de programación lineal (curso posterior)")
        elif iv == "vacío":
            avisos.append("ningún valor del parámetro deja todas las cantidades no negativas: la mezcla es imposible")
    return dict(res, A=A, b=b, factible=factible, avisos=avisos, composiciones=filas, objetivo=c, total=tot, n=n, k=k)


def asignacion(A, b):
    """Asignación de recursos, exacta: A[i, j] = cuánto del recurso i usa una unidad del producto j; b = cuánto hay de
    cada recurso. Resuelve A x = b (usar todo lo disponible). Diccionario con A, b, estado, x, factible (x ≥ 0),
    avisos, solucion_general, parametros, libres e intervalo, como mezcla."""
    A, b = como_matriz(A, "A"), vector(b, "b")
    if A.rows != b.rows:
        raise ValueError(f"A tiene {_cuenta(A.rows, 'renglón', 'renglones')} (uno por recurso) y b tiene "
                         f"{_cuenta(b.rows, 'número', 'números')}: debe haber un dato de b por cada recurso")
    if A.rows > _MAX_EXACTO or A.cols > _MAX_EXACTO:
        raise ValueError(f"A es de {dims(A)}; esta calculadora acepta hasta {_MAX_EXACTO} recursos y {_MAX_EXACTO} productos")
    if any(e < 0 for e in A) or any(e < 0 for e in b):
        raise ValueError("los consumos de A y las disponibilidades de b no pueden ser negativos")
    res = _analiza(A, b)
    avisos, factible = [], False
    if res["estado"] == "sin solución":
        avisos.append("no hay ningún plan que use exactamente todos los recursos (el sistema no tiene solución)")
    elif res["estado"] == "única":
        factible = all(v >= 0 for v in res["x"])
        for j, v in enumerate(res["x"], 1):
            if v < 0:
                avisos.append(f"el producto {j} saldría negativo: no se puede usar todo exactamente; hay que dejar sin "
                              "usar parte de algún recurso")
    else:
        iv = res["intervalo"]
        factible = (iv != "vacío") if iv is not None else None
        if iv == "vacío":
            avisos.append("el sistema tiene infinitas soluciones, pero en todas alguna cantidad es negativa: ningún plan "
                          "con cantidades no negativas usa todo exactamente; hay que dejar sin usar parte de algún recurso")
        elif iv is None:
            avisos.append("hay infinitas soluciones; decidir cuáles tienen todas las cantidades no negativas y escoger "
                          "una por otro criterio, como la ganancia, es programación lineal (curso posterior)")
        else:
            avisos.append("hay infinitos planes que usan todo: se escoge uno por otro criterio, como la ganancia "
                          "(programación lineal, curso posterior)")
    return dict(res, A=A, b=b, factible=factible, avisos=avisos)


def _lineal_aprox(e, params, n):
    """'  ≈ 34.97 - 0.3043·t': la forma decimal (a n cifras) de una expresión lineal en los parámetros, o '' si ya es
    exacta en decimales o no tiene parte constante ni coeficientes fraccionarios."""
    const = e.subs({p: 0 for p in params})
    coefs = [e.coeff(p) for p in params]
    if all(_decimal_exacto(c, n) is not None for c in [const] + coefs):
        return ""
    partes = [] if const == 0 else [cifras(const, n)]
    for c, p in zip(coefs, params):
        if c == 0:
            continue
        signo = ("-" if c < 0 else "") if not partes else ("- " if c < 0 else "+ ")
        partes.append(f"{signo}{cifras(abs(c), n)}·{_txt(p)}")
    return "  ≈ " + " ".join(partes)


def _informe_sistema(res, letra, nombre_j, unidad, n_cifras):
    """La solución de A x = b para mezcla y asignacion: única, general o sin solución, con veredicto y comprobación."""
    A, b = res["A"], res["b"]
    if res["estado"] == "única":
        x = res["x"]
        for j, v in enumerate(x, 1):
            print("  " + _magnitud(f"{letra}{j}", v, unidad, n_cifras) + f"  ({nombre_j} {j})" +
                  ("  ← negativa" if v < 0 else ""))
        print("Comprobación: A x = b" if A * x == b else "Comprobación FALLÓ: A x ≠ b")
        print("Veredicto: es posible, todas las cantidades son no negativas (≥ 0)." if res["factible"] else
              "Veredicto: IMPOSIBLE con estos datos: el sistema tiene solución, pero alguna cantidad es negativa.")
    elif res["estado"] == "infinitas":
        params, libres = res["parametros"], res["libres"]
        print("Hay infinitas soluciones. " + ", ".join(f"{_txt(p)} = {letra}{c + 1}" for p, c in zip(params, libres)) +
              (" es libre" if len(params) == 1 else " son libres") + ":")
        for j, e in enumerate(res["solucion_general"], 1):
            print(f"  {letra}{j} = {_txt(e)}" + _lineal_aprox(e, params, n_cifras))
        iv = res["intervalo"]
        if iv == "vacío":
            print("Ningún valor del parámetro deja todas las cantidades no negativas: es imposible.")
        elif iv is not None:
            lo, hi = iv
            if lo == hi:                              # intervalo de un solo punto
                print(f"Solo un valor del parámetro deja todas las cantidades no negativas (≥ 0): t = "
                      f"{_cantidad(lo, unidad, n_cifras)}.")
            else:
                print(f"Para que todas las cantidades sean no negativas (≥ 0), t debe ir de "
                      f"{_cantidad(lo, unidad, n_cifras) if lo.is_finite else '-∞'} a "
                      f"{_cantidad(hi, unidad, n_cifras) if hi.is_finite else '∞'}.")
            for extremo in ((lo,) if lo == hi else (lo, hi)):
                if extremo.is_finite:
                    v = res["solucion_general"].subs(_t, extremo)
                    ceros = [f"{letra}{j}" for j, e in enumerate(v, 1) if e == 0]
                    print(f"  En t = {_valor(extremo, n_cifras)}: {letra} = {_tupla_u(v, unidad, n_cifras)}" +
                          (f"; {_lista(ceros)} {'vale' if len(ceros) == 1 else 'valen'} 0." if ceros else "."))
    else:
        print("El sistema A x = b no tiene solución.")
    for aviso in res["avisos"]:
        print("Aviso:", aviso + ".")


def _informe_mezcla(res, unidad, n_cifras):
    n, k, A, b = res["n"], res["k"], res["A"], res["b"]
    print(f"{_cuenta(n, 'ingrediente', 'ingredientes')}, {_cuenta(k, 'componente controlado', 'componentes controlados')}; "
          f"total {_ntxt(res['total'], cif=n_cifras)} {unidad}, con objetivo " +
          ", ".join(f"{_ntxt(v)} %" for v in res["objetivo"]) + ".")
    print("Ecuaciones (una por el total y una por cada componente):")
    print(f"  total: {_combinacion(A.row(0), 'x')} = {_ntxt(b[0])}")
    for i in range(k):
        print(f"  componente {i + 1}: {_combinacion(A.row(i + 1), 'x')} = {_ntxt(res['total'])}·{_ntxt(res['objetivo'][i] / 100)}"
              f" = {_ntxt(b[i + 1])}")
    muestra("A", A, n_cifras)
    muestra("b", b, n_cifras, tex=r"\mathbf b")
    print(f"Cantidades (en {unidad}):")
    _informe_sistema(res, "x", "ingrediente", unidad, n_cifras)


def dibuja_triangulo(composiciones, objetivos, n_cifras=4):
    """Plano de composiciones para TRES ingredientes y DOS componentes: el triángulo de las mezclas posibles (sus
    vértices son los ingredientes) y cada objetivo, dentro (círculo) o fuera (equis). Con otros tamaños no dibuja y lo
    dice."""
    filas = tabla(composiciones, "los ingredientes", "el ingrediente", None, _MAX_INGREDIENTES, "46, 0")
    if len(filas) != 3 or len(filas[0]) != 2:
        print("No se dibuja: el triángulo de mezclas es solo para tres ingredientes con dos componentes controlados.")
        return
    objs = tabla(objetivos, "los objetivos", "el objetivo", 2, 10, "20, 10")
    M = sp.Matrix([[1, 1, 1], [f[0] for f in filas], [f[1] for f in filas]])
    if M.det() == 0:
        print("No se dibuja: los tres ingredientes están alineados en el plano de composiciones (no forman un triángulo).")
        return
    V = np.array([[float(f[0]), float(f[1])] for f in filas])
    fig, ax = plt.subplots(figsize=(6.6, 7.2))
    ax.fill(V[:, 0], V[:, 1], facecolor="0.88", edgecolor="black", hatch="//", lw=1.5,
            label="mezclas posibles (todas las cantidades ≥ 0)")
    for j, (p, f) in enumerate(zip(V, filas)):
        ax.plot(*p, _MARCAS[j], color=_COLORES[j], ms=10, mec="black",
                label=f"ingrediente {j + 1}: ({_ntxt(f[0])}, {_ntxt(f[1])}) %")
    for o in objs:
        lam = M.LUsolve(sp.Matrix([1, o[0], o[1]]))
        dentro = all(v >= 0 for v in lam)
        ax.plot(float(o[0]), float(o[1]), "o" if dentro else "X", color="black" if dentro else _COLORES[1],
                ms=11 if dentro else 12, mec="black",
                label=f"objetivo ({_ntxt(o[0])}, {_ntxt(o[1])}) %: " + ("dentro" if dentro else "fuera, alguna cantidad sería negativa"))
    todos = np.vstack([V, np.array([[float(o[0]), float(o[1])] for o in objs])])
    lo, hi = todos.min(axis=0), todos.max(axis=0)
    margen = 0.12 * max(float((hi - lo).max()), 1.0)
    ax.set_xlim(lo[0] - margen, hi[0] + margen); ax.set_ylim(lo[1] - margen, hi[1] + margen)
    ax.set_aspect("equal"); ax.grid(alpha=0.4)
    ax.set_xlabel("componente 1 (%)"); ax.set_ylabel("componente 2 (%)")
    ax.set_title("Plano de composiciones: el triángulo de los tres ingredientes", fontsize=10)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=1)
    plt.tight_layout()
    plt.show()

**Recorrido paso a paso.** (1) Dos soluciones de un mismo ácido, al 30 % y al 70 %, para preparar 100 L al 40 %: $x_1+x_2=100$ y $0.30\,x_1+0.70\,x_2=40$ dan $(75,\ 25)$ L. Al 80 %, el sistema da $(-25,\ 125)$: imposible, porque 80 % está fuera del intervalo $[30,\,70]$ % que cubren las dos soluciones. (2) Un fertilizante de 100 kg con 20 % de nitrógeno (N) y 10 % de fósforo (P) a partir de urea (46 % N, 0 % P), fosfato diamónico (18 % N, 46 % P) y un relleno inerte (0 %, 0 %): el sistema de $3\times3$ da $\left(\tfrac{18500}{529},\ \tfrac{500}{23},\ \tfrac{22900}{529}\right)\approx(34.97,\ 21.74,\ 43.29)$ kg. En el triángulo, el objetivo $(20,10)$ queda dentro y $(30,30)$ queda fuera. (3) Un plan de dos productos: el primero usa 2 h de máquina y 1 h de ensamble, el segundo 1 h y 3 h, y hay 100 h de máquina y 150 h de ensamble: usar todo da $(30,\ 40)$ unidades.

In [ ]:
print("(1) Dos soluciones, al 30 % y al 70 %, para 100 L al 40 %")
_informe_mezcla(mezcla("30\n70", "40", 100), "L", 4)
print("\nAl 80 %:")
_informe_mezcla(mezcla("30\n70", "80", 100), "L", 4)
print("\n" + "=" * 90 + "\n\n(2) Fertilizante: urea, fosfato diamónico y relleno")
_fert = mezcla("46, 0\n18, 46\n0, 0", "20, 10", 100)
_informe_mezcla(_fert, "kg", 4)
dibuja_triangulo("46, 0\n18, 46\n0, 0", "20, 10\n30, 30")
print("=" * 90 + "\n\n(3) Plan de producción: 2 h de máquina y 1 h de ensamble por unidad del producto 1; 1 h y 3 h del producto 2")
_plan = asignacion([[2, 1], [1, 3]], [100, 150])
muestra("A (h por unidad)", _plan["A"], tex=r"A\ (\text{h por unidad})")
muestra("b (h disponibles)", _plan["b"], tex=r"\mathbf b\ (\text{h disponibles})")
print("  máquina: " + _combinacion(_plan["A"].row(0), "x") + " = 100;   ensamble: " + _combinacion(_plan["A"].row(1), "x") + " = 150")
_informe_sistema(_plan, "x", "producto", "unidades", 4)

**Más ingredientes que ecuaciones.** Si al fertilizante le agregas un cuarto ingrediente con 23 % de N y 23 % de P, hay cuatro incógnitas y tres ecuaciones: la cantidad del cuarto, $t=x_4$, queda libre. Cada cantidad es una función de $t$, y pedir que las cuatro sean $\ge 0$ deja un intervalo de valores de $t$; en cada extremo, alguna cantidad vale 0. Escoger un valor de ese intervalo (por ejemplo, el más barato) ya es programación lineal.

In [ ]:
_cuatro = mezcla("46, 0\n18, 46\n0, 0\n23, 23", "20, 10", 100)
_informe_mezcla(_cuatro, "kg", 4)

**Calculadora.** Escribe **un ingrediente por renglón** con sus porcentajes de los componentes controlados (en el mismo orden que el objetivo), el objetivo en porcentaje, el total y su unidad. Con tres ingredientes y dos componentes, la calculadora dibuja el triángulo de mezclas posibles. Prueba un objetivo fuera del triángulo (`30, 30`), un cuarto ingrediente (`23, 23`) y dos ingredientes iguales.

In [ ]:
def calculadora_mezcla(ingredientes_txt, objetivo_txt, total_txt, unidad, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        try:
            u = nombre_unidad(unidad, "cantidad", "kg o L")
            res = mezcla(ingredientes_txt, objetivo_txt, total_txt, n_cifras=n_cifras)
        except ValueError as err:
            _reporta(err); return
        _aviso_coma(ingredientes_txt, objetivo_txt)
        _informe_mezcla(res, u, n_cifras)
    if res["n"] == 3 and res["k"] == 2:
        dibuja_triangulo(res["composiciones"], [list(res["objetivo"])], n_cifras)


_CONTROLES_3 = dict(
    ingredientes_txt=widgets.Textarea(value="46, 0\n18, 46\n0, 0", description="ingredientes (%):",
                                      continuous_update=False, layout=widgets.Layout(width="320px", height="110px"),
                                      style={"description_width": "initial"}),
    objetivo_txt=widgets.Text(value="20, 10", description="objetivo (%):", continuous_update=False,
                              style={"description_width": "initial"}),
    total_txt=widgets.Text(value="100", description="total:", continuous_update=False),
    unidad=widgets.Text(value="kg", description="unidad:", continuous_update=False),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_mezcla, **_CONTROLES_3);

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
_iv = _cuatro["intervalo"]
PRUEBAS_3 = [
    ("dos soluciones, 30 % y 70 %, para 100 al 40 % → (75, 25) y factible; para 50 al 40 % → (37.5, 12.5)",
     lambda: (lambda r: r["x"] == vector("75, 25") and r["factible"] and not r["avisos"])(mezcla([(30,), (70,)], (40,), 100))
             and mezcla("30\n70", "40", 50)["x"] == vector("37.5, 12.5")),
    ("objetivo 80 % → (-25, 125), no factible y con aviso",
     lambda: (lambda r: r["x"] == vector("-25, 125") and r["factible"] is False
                        and any("ingrediente 1 saldría negativo" in a for a in r["avisos"])
                        and any("fuera del intervalo" in a for a in r["avisos"]))(mezcla("30\n70", "80", 100))),
    ("fertilizante → (18500/529, 500/23, 22900/529) exactas y factible (también si se escriben los signos %)",
     lambda: _fert["x"] == sp.Matrix([sp.Rational(18500, 529), sp.Rational(500, 23), sp.Rational(22900, 529)])
             and _fert["factible"]
             and mezcla("46 %, 0 %\n18 %, 46 %\n0, 0", "20 %, 10 %", 100)["x"] == _fert["x"]),
    ("cuatro ingredientes (con 23 %, 23 %) → solución general con un parámetro t = x4 e intervalo [0, 1000/23]; en "
     "los extremos alguna cantidad es 0 y fuera de él alguna es negativa",
     lambda: _cuatro["estado"] == "infinitas" and _cuatro["parametros"] == [_t] and _cuatro["libres"] == [3]
             and _iv == (0, sp.Rational(1000, 23)) and _cuatro["factible"]
             and all(any(e == 0 for e in _cuatro["solucion_general"].subs(_t, v)) for v in _iv)
             and all(any(e < 0 for e in _cuatro["solucion_general"].subs(_t, v)) for v in (-1, _iv[1] + 1))
             and _cuatro["A"] * _cuatro["solucion_general"].subs(_t, 7) == _cuatro["b"]
             and mezcla("0\n100\n50", "0", 10)["intervalo"] == (0, 0)
             and "Solo un valor" in _captura(calculadora_mezcla, "0\n100\n50", "0", "10", "kg", 4)[0]),
    ("asignacion([[2, 1], [1, 3]], [100, 150]) → (30, 40); con A = [[1, 2], [3, 1]] y b = (50, 60) → (14, 18) (no la "
     "de Aᵀ); [[1, 1, 1], [1, 2, 3]] con (10, 40): infinitas soluciones, ninguna no negativa",
     lambda: (lambda r: r["x"] == vector("30, 40") and r["factible"])(asignacion([[2, 1], [1, 3]], [100, 150]))
             and asignacion([[1, 2], [3, 1]], [50, 60])["x"] == vector("14, 18")
             and (lambda r: r["estado"] == "infinitas" and r["factible"] is False and r["intervalo"] == "vacío"
                            and any("ningún plan con cantidades no negativas" in a for a in r["avisos"]))(
                 asignacion([[1, 1, 1], [1, 2, 3]], [10, 40]))),
    ("dos ingredientes con la misma composición y nada más → ValueError «no determinan la mezcla» (sin traza)",
     lambda: "no determinan la mezcla" in _mensaje(lambda: mezcla("30\n30", "40", 100))
             and "no determinan la mezcla" in _captura(calculadora_mezcla, "30\n30", "30", "100", "L", 4)[0]),
    ("prueba de humo: la calculadora corre con sus valores iniciales, con dos ingredientes y con cuatro, sin error y "
     "sin «Revisa la entrada»",
     lambda: _humo(calculadora_mezcla, _CONTROLES_3)
             and _humo(calculadora_mezcla, _CONTROLES_3, ingredientes_txt="30\n70", objetivo_txt="80", unidad="L")
             and _humo(calculadora_mezcla, _CONTROLES_3, ingredientes_txt="46, 0\n18, 46\n0, 0\n23, 23", n_cifras=6)),
]
corre_pruebas(PRUEBAS_3, 3)

**Contrasta (sección 3).** Dos aleaciones tienen 90 % y 60 % de cobre. ¿Cuántos kg de cada una necesitas para 30 kg al 70 %? Plantea las dos ecuaciones (total y cobre), resuélvelas a mano y escribe `mis_kg = (a, b)`: primero la de 90 %.

In [ ]:
mis_kg = None     # dos números (a, b) en kg, por ejemplo (12.5, 17.5)

if mis_kg is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mias = vector(mis_kg, "mis_kg")
    except ValueError as err:
        print(f"Revisa tu respuesta: {str(err).rstrip('.')}.")
    else:
        if mias.rows != 2:          # respuestas imposibles: la referencia todavía no se muestra
            print(f"Son dos aleaciones, así que son dos cantidades (escribiste {mias.rows}): revisa tu respuesta y vuelve "
                  "a ejecutar la celda.")
        elif any(v < 0 for v in mias):
            print("Una cantidad de aleación no puede ser negativa: revisa tu cálculo y vuelve a ejecutar la celda.")
        elif abs(float(sum(mias)) - 30) > 0.01:
            print(f"Tus cantidades suman {_ntxt(sum(mias))} kg y deben sumar 30 kg (es la primera ecuación): revisa y "
                  "vuelve a ejecutar la celda.")
        else:
            ref = mezcla("90\n60", "70", 30)          # los datos del enunciado
            print(f"La calculadora da: (a, b) = {_tupla_u(ref['x'], 'kg', 4)}.")
            if all(abs(float(a - b)) <= 0.01 for a, b in zip(mias, ref["x"])):
                print("Coinciden (con tolerancia de 0.01 kg).")
            elif all(abs(float(a - b)) <= 0.01 for a, b in zip(mias, reversed(list(ref["x"])))):
                print("NO coinciden: probablemente invertiste el orden; a es la aleación de 90 %.")
            else:
                print("NO coinciden: la ecuación del cobre es 0.90 a + 0.60 b = 0.70·30 = 21.")
            print("\nEl procedimiento completo:")
            _informe_mezcla(ref, "kg", 4)

## 4. Armadura por el método de nodos

**Armadura ideal.** Barras rectas unidas por nodos articulados (pasadores que dejan girar), con las cargas aplicadas solo en los nodos. Así cada barra trabaja solo a lo largo de su eje: o jala o empuja.

**Convención.** La fuerza $F$ de una barra es **positiva si es tensión**: la barra jala a cada uno de sus nodos hacia ella, es decir, hacia el otro extremo. En el nodo $i$, la barra que lo une con el nodo $j$ hace la fuerza $F\,\mathbf u_{ij}$, con $\mathbf u_{ij}=(P_j-P_i)/\|P_j-P_i\|$. Una fuerza negativa es **compresión** (la barra empuja).

**Método de nodos.** Cada nodo está en equilibrio: $\sum F_x=0$ y $\sum F_y=0$. Con $n$ nodos, $b$ barras y $r$ reacciones de los apoyos (un apoyo articulado da dos, $R_x$ y $R_y$; un rodillo da una) hay $2n$ ecuaciones y $b+r$ incógnitas, $A\mathbf x=\mathbf b$, con una columna de $A$ por barra (los vectores unitarios en sus dos nodos) y una por reacción.

- Si $b+r=2n$ y $A$ es invertible, la armadura es **estáticamente determinada**: el equilibrio da todas las fuerzas.
- Si $b+r<2n$, es un **mecanismo**: le faltan barras o apoyos y se mueve.
- Si $b+r>2n$, es **hiperestática**: sobran incógnitas, y las ecuaciones de equilibrio no bastan (hace falta cómo se deforman las barras, que verás en mecánica de materiales).
- **Contar no basta:** con $b+r=2n$, una geometría inestable (por ejemplo, sin nada que impida el movimiento horizontal) da una matriz singular, de rango menor que $2n$.

**Barras de fuerza cero.** En un nodo sin carga ni apoyo al que llegan solo dos barras no colineales, las dos tienen fuerza cero: ninguna puede equilibrar a la otra.

**Cómo se escriben.** Un nodo por renglón como `nombre, x, y` (en metros); una barra como `nombre, nombre`; un apoyo como `nombre, x` o `nombre, y` (un articulado son dos renglones, uno por dirección); una carga como `nombre, Fx, Fy`. Hasta 20 nodos.

In [ ]:
_MAX_NODOS, _MAX_BARRAS = 20, 40
_NOMBRE_NODO = re.compile(r"[A-Za-z][A-Za-z0-9_]{0,7}")


def _nombre_nodo(t, donde):
    t = str(t).strip()
    if not _NOMBRE_NODO.fullmatch(t):
        raise ValueError(f"{donde}: «{t}» no es un nombre de nodo; usa una letra, seguida si quieres de letras o números "
                         "(A, B, N1)")
    return t


def _filas_nombradas(texto, nombre, cosa, ejemplo, maximo):
    if isinstance(texto, dict):
        filas = [[k] + (list(v) if isinstance(v, (list, tuple, np.ndarray, sp.MatrixBase)) else [v])
                 for k, v in texto.items()]
    else:
        filas = _filas_crudas(texto, nombre, cosa, ejemplo)
    if len(filas) > maximo:
        raise ValueError(f"{nombre}: escribiste {len(filas)} renglones y esta calculadora acepta hasta {maximo}")
    return filas


def _lee_nodos(nodos):
    filas = _filas_nombradas(nodos, "los nodos", "el nodo", "A, 0, 0 (nombre, x, y)", _MAX_NODOS)
    N = {}
    for i, f in enumerate(filas, 1):
        if len(f) != 3:
            raise ValueError(f"en los nodos, el renglón {i} debe tener nombre, x, y, por ejemplo: A, 0, 0"
                             f"{_pista_decimal(nodos)}")
        nm = _nombre_nodo(f[0], f"en los nodos, renglón {i}")
        if nm in N:
            raise ValueError(f"el nodo {nm} está escrito dos veces")
        try:
            N[nm] = sp.Matrix([numero(f[1], "x"), numero(f[2], "y")])
        except ValueError as err:
            raise ValueError(f"en el nodo {nm}: {err}") from None
    for a, b in ((a, b) for k, a in enumerate(N) for b in list(N)[k + 1:]):
        if N[a] == N[b]:
            raise ValueError(f"los nodos {a} y {b} están en el mismo punto")
    return N


def _existe(nm, N, donde):
    nm = _nombre_nodo(nm, donde)
    if nm not in N:
        raise ValueError(f"{donde}: el nodo {nm} no está en la lista de nodos ({_coma(N)})")
    return nm


def _lee_barras(barras, N):
    filas = _filas_nombradas(barras, "las barras", "la barra", "A, B (los nombres de sus dos nodos)", _MAX_BARRAS)
    B = []
    for k, f in enumerate(filas, 1):
        if len(f) != 2:
            raise ValueError(f"en las barras, el renglón {k} debe tener los nombres de dos nodos, por ejemplo: A, B")
        i, j = (_existe(e, N, f"en la barra {k}") for e in f)
        if i == j:
            raise ValueError(f"la barra {k} une al nodo {i} consigo mismo")
        if (i, j) in B or (j, i) in B:
            raise ValueError(f"la barra {i}{j} está escrita dos veces")
        B.append((i, j))
    return B


def _lee_apoyos(apoyos, N):
    filas = _filas_nombradas(apoyos, "los apoyos", "el apoyo", "A, x (el nodo y la dirección de la reacción)", 2 * _MAX_NODOS)
    S = []
    for k, f in enumerate(filas, 1):
        if len(f) != 2:
            raise ValueError(f"en los apoyos, el renglón {k} debe tener un nodo y una dirección, por ejemplo: A, x")
        nm = _existe(f[0], N, f"en el apoyo {k}")
        e = normaliza_texto(f[1])
        if e not in ("x", "y"):
            raise ValueError(f"en el apoyo {k}, la dirección debe ser x o y (escribiste «{f[1]}»)")
        if (nm, e) in S:
            raise ValueError(f"el apoyo {nm} en {e} está escrito dos veces")
        S.append((nm, e))
    return S


def _lee_cargas(cargas, N):
    C = {nm: sp.zeros(2, 1) for nm in N}
    if cargas is None or (isinstance(cargas, str) and not cargas.strip()) or (
            isinstance(cargas, (list, tuple, dict)) and len(cargas) == 0):
        return C
    filas = _filas_nombradas(cargas, "las cargas", "la carga", "C, 0, -10 (nodo, Fx, Fy)", 2 * _MAX_NODOS)
    for k, f in enumerate(filas, 1):
        if len(f) != 3:
            raise ValueError(f"en las cargas, el renglón {k} debe tener nodo, Fx, Fy, por ejemplo: C, 0, -10"
                             f"{_pista_decimal(cargas)}")
        nm = _existe(f[0], N, f"en la carga {k}")
        try:
            C[nm] += sp.Matrix([numero(f[1], "Fx"), numero(f[2], "Fy")])     # dos renglones del mismo nodo se suman
        except ValueError as err:
            raise ValueError(f"en la carga {k} (nodo {nm}): {err}") from None
    return C


def _nombre_barra(i, j):
    return f"{i}{j}" if len(i) == len(j) == 1 else f"{i}-{j}"


_EXPLICA = {
    "mecanismo por conteo": "faltan barras o apoyos: la armadura se puede mover (es un mecanismo) y no hay equilibrio para "
                            "una carga cualquiera",
    "hiperestática por conteo": "sobran incógnitas: las ecuaciones de equilibrio no bastan para encontrar las fuerzas "
                                "(hace falta cómo se deforman las barras, que verás en mecánica de materiales)",
    "inestable: la matriz es singular aunque el conteo cuadra": "la geometría permite un movimiento (por ejemplo, nada "
                                                                 "impide que se deslice o gire), así que contar no basta",
    "hiperestática por conteo e inestable (rango < 2n)": "sobran incógnitas y, aun así, la geometría permite un "
                                                         "movimiento: las barras de más no están donde hacen falta",
}


def armadura(nodos, barras, apoyos, cargas, n_cifras=4):
    """Método de nodos, exacto. nodos: dict nombre -> (x, y) o texto 'A, 0, 0' por renglón; barras: lista de pares de
    nombres; apoyos: lista de (nodo, 'x' o 'y'); cargas: dict nodo -> (Fx, Fy) o texto 'C, 0, -10'. Diccionario con
    A (2n × (b + r): vectores unitarios de las barras en sus nodos y una columna por reacción), b (menos las cargas),
    fuerzas (dict barra -> fuerza exacta, positiva = tensión; las longitudes pueden dar raíces), tipo (dict barra ->
    'T', 'C' o '0'), reacciones (dict 'Ax', 'Ay', ... -> valor), longitudes, conteo (n, b, r), rango, diagnostico
    ('determinada', 'mecanismo por conteo', 'hiperestática por conteo', 'hiperestática por conteo e inestable (rango <
    2n)' o 'inestable: la matriz es singular aunque el conteo cuadra') y comprobacion_global (suma de fuerzas externas y momento respecto al primer nodo, que deben ser
    cero) y avisos (por ejemplo, si la armadura está cerca de ser inestable). Si no es determinada, fuerzas, tipo,
    reacciones y comprobacion_global son None."""
    N = _lee_nodos(nodos)
    B, S, C = _lee_barras(barras, N), _lee_apoyos(apoyos, N), _lee_cargas(cargas, N)
    nombres = list(N)
    idx = {nm: k for k, nm in enumerate(nombres)}
    n, b, r = len(N), len(B), len(S)
    M = sp.zeros(2 * n, b + r)                 # columnas de barra multiplicadas por su longitud: entradas racionales
    L = []
    for k, (i, j) in enumerate(B):
        d = N[j] - N[i]
        L.append(sp.sqrt(_punto(d, d)))
        M[2 * idx[i], k], M[2 * idx[i] + 1, k] = d[0], d[1]
        M[2 * idx[j], k], M[2 * idx[j] + 1, k] = -d[0], -d[1]
    for k, (nm, e) in enumerate(S):
        M[2 * idx[nm] + (0 if e == "x" else 1), b + k] = 1
    A = M.copy()
    for k in range(b):
        A[:, k] = M[:, k] / L[k]
    rhs = -sp.Matrix([C[nm][c] for nm in nombres for c in (0, 1)])
    rango = M.rank(iszerofunc=lambda e: e == 0)
    if b + r < 2 * n:
        diag = "mecanismo por conteo"
    elif b + r > 2 * n:
        diag = "hiperestática por conteo" if rango == 2 * n else "hiperestática por conteo e inestable (rango < 2n)"
    elif rango < 2 * n:
        diag = "inestable: la matriz es singular aunque el conteo cuadra"
    else:
        diag = "determinada"
    res = {"A": A, "b": rhs, "fuerzas": None, "tipo": None, "reacciones": None, "comprobacion_global": None,
           "longitudes": {_nombre_barra(i, j): l for (i, j), l in zip(B, L)}, "conteo": (n, b, r), "rango": rango,
           "diagnostico": diag, "nodos": N, "barras": B, "apoyos": S, "cargas": C, "avisos": []}
    if diag != "determinada":
        return res
    cond = _condicion(A)
    if cond > _COND_AVISO:
        res["avisos"] = [f"la armadura está cerca de ser inestable (número de condición de A ≈ {cifras(cond, n_cifras)}): "
                         "las fuerzas pueden salir muy grandes y cambian mucho si la geometría cambia un poco"]
    q = M.LUsolve(rhs)                         # q_k = F_k/L_k en las barras; las reacciones, directas
    fuerzas = {_nombre_barra(i, j): q[k] * L[k] for k, (i, j) in enumerate(B)}
    reacciones = {f"{nm}{e}": q[b + k] for k, (nm, e) in enumerate(S)}
    externas = [(N[nm], C[nm]) for nm in nombres] + [
        (N[nm], sp.Matrix([v, 0]) if e == "x" else sp.Matrix([0, v])) for (nm, e), v in zip(S, reacciones.values())]
    P0 = N[nombres[0]]
    suma = sum((F for _, F in externas), sp.zeros(2, 1))
    mom = sum(((P - P0)[0] * F[1] - (P - P0)[1] * F[0] for P, F in externas), sp.Integer(0))
    res.update(fuerzas=fuerzas, tipo={k: ("T" if v > 0 else "C" if v < 0 else "0") for k, v in fuerzas.items()},
               reacciones=reacciones,
               comprobacion_global={"suma_fuerzas": suma, "momento": mom, "respecto_a": nombres[0],
                                    "cero": _es_nulo(suma) and mom == 0})
    return res


def _informe_armadura(res, unidad, n_cifras):
    N, (n, b, r) = res["nodos"], res["conteo"]
    print("Nodos: " + ", ".join(f"{nm} {_tupla_u(P, '')}" for nm, P in N.items()) + " m")
    print("Barras: " + _coma(_nombre_barra(i, j) for i, j in res["barras"]) +
          ";  apoyos (reacciones): " + (_coma(f"{nm}{e}" for nm, e in res["apoyos"]) or "ninguno"))
    cargadas = [f"{nm} {_tupla_u(F, '')}" for nm, F in res["cargas"].items() if not _es_nulo(F)]
    print(f"Cargas ({unidad}): " + (", ".join(cargadas) if cargadas else "ninguna"))
    print(f"Conteo: n = {n} nodos, b = {b} barras, r = {r} reacciones → 2n = {2 * n} ecuaciones y b + r = {b + r} "
          f"incógnitas;  rango de A = {res['rango']}.")
    diag = res["diagnostico"]
    if diag != "determinada":
        print(f"Diagnóstico: {diag.upper().replace('2N', '2n')} — {_EXPLICA[diag]}.")
        print("No se calculan fuerzas: con esta armadura el equilibrio no da una solución única.")
        return
    print("Diagnóstico: DETERMINADA (b + r = 2n y A es invertible): el equilibrio da todas las fuerzas.")
    if 2 * n <= 8:
        print("Columnas de A: " + _coma([_nombre_barra(i, j) for i, j in res["barras"]] + list(res["reacciones"])) +
              ";  renglones: ΣFx y ΣFy en " + _lista(list(N)) + ".")
        muestra("A", res["A"], n_cifras)
    else:
        print(f"A es de {dims(res['A'])}: no se muestra aquí (está en el resultado de armadura, clave 'A').")
    print(f"Fuerzas en las barras (positiva = tensión, en {unidad}):")
    nombres_tipo = {"T": "tensión", "C": "compresión", "0": "fuerza cero"}
    for k, F in res["fuerzas"].items():
        t = res["tipo"][k]
        print(f"  {k}: {_magnitud('L', res['longitudes'][k], 'm', n_cifras)},  "
              + _magnitud("F", F, unidad, n_cifras) + f"  ({t}, {nombres_tipo[t]})")
    print("Reacciones: " + ",  ".join(_magnitud(k, v, unidad, n_cifras) for k, v in res["reacciones"].items()))
    cg = res["comprobacion_global"]
    print(f"Comprobación global (cargas y reacciones): suma de fuerzas = {_tupla_u(cg['suma_fuerzas'], unidad)}, momento "
          f"respecto a {cg['respecto_a']} = {_ntxt(cg['momento'])} {unidad}·m" + ("  (cero: la armadura completa está en "
                                                                                 "equilibrio)" if cg["cero"] else "  (¡no es cero!)"))
    for aviso in res["avisos"]:
        print("Aviso:", aviso + ".")
    ceros = [k for k, t in res["tipo"].items() if t == "0"]
    if ceros:
        print(f"Barras de fuerza cero: {_coma(ceros)}. No cargan nada con esta carga, pero no se pueden quitar: "
              "rigidizan la armadura y otra carga sí las haría trabajar.")


def dibuja_armadura(resultado, unidad="kN", n_cifras=4):
    """Dibuja la armadura: tensión con línea continua, compresión discontinua y fuerza cero punteada, con su valor y
    T/C; las cargas (flechas negras) y las reacciones (flechas de rayas y puntos). Con más de 12 barras, cada barra
    lleva solo su nombre, y con más de 24, ningún rótulo (los nombres y las fuerzas están en el texto). Si no es
    determinada, dibuja solo la geometría y los apoyos."""
    N, B = resultado["nodos"], resultado["barras"]
    P = {nm: _flotante(v).ravel() for nm, v in N.items()}
    xy = np.array(list(P.values()))
    s = max(float((xy.max(axis=0) - xy.min(axis=0)).max()), 1.0)
    muchas = len(B) > 12                       # con muchas barras, los rótulos con valor se enciman
    sin_rotulos = len(B) > 24                  # y con más de 24, también los nombres
    a = (0.1 if muchas else 0.18) * s
    fig, ax = plt.subplots(figsize=(11.0, 8.6) if muchas else (8.4, 7.0))
    fuerzas, tipo = resultado["fuerzas"], resultado["tipo"]
    estilos = {"T": ("-", _COLORES[0], 2.8), "C": ("--", _COLORES[1], 2.8), "0": (":", "0.45", 2.2)}
    letra = 7.5 if muchas else 8
    for i, j in B:
        k = _nombre_barra(i, j)
        ls, color, lw = estilos[tipo[k]] if tipo else ("-", "0.55", 2.0)
        ax.plot(*np.array([P[i], P[j]]).T, linestyle=ls, color=color, lw=lw, zorder=2)
        if sin_rotulos:
            continue
        texto = k if (not fuerzas or muchas) else f"{k} = {_dec_o_cifras(fuerzas[k], n_cifras)} {unidad} ({tipo[k]})"
        d = P[j] - P[i]
        giro = math.degrees(math.atan2(d[1], d[0]))
        giro = giro - 180 if giro > 90 else (giro + 180 if giro < -90 else giro)      # el texto nunca de cabeza
        ax.annotate(texto, xy=(P[i] + P[j]) / 2, ha="center", va="center", fontsize=letra, zorder=4, rotation=giro,
                    rotation_mode="anchor",
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))
    for nm, p in P.items():
        ax.plot(*p, "o", color="black", ms=7, zorder=5)
        ax.annotate(nm, xy=p, xytext=(-12, 8), textcoords="offset points", fontsize=8 if len(P) > 10 else 11,
                    fontweight="bold")
    for nm, F in resultado["cargas"].items():
        f = _flotante(F).ravel()
        if np.linalg.norm(f) > 0:
            u = f / np.linalg.norm(f)
            _flecha(ax, P[nm] - a * u, P[nm], "black", "-", 2.6)
            dx, dy = -14 * u                         # el rótulo, más allá de la cola de la flecha (lejos de las barras)
            ax.annotate(f"{_dec_o_cifras(sp.sqrt(_punto(F, F)), n_cifras)} {unidad}", xy=P[nm] - a * u, xytext=(dx, dy),
                        textcoords="offset points", fontsize=9, zorder=6,
                        ha="center" if abs(dx) < 5 else ("right" if dx < 0 else "left"),
                        va="bottom" if dy > 5 else ("top" if dy < -5 else "center"),
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))
    reac = resultado["reacciones"]
    for (nm, e) in resultado["apoyos"]:
        eje = np.array([1.0, 0.0]) if e == "x" else np.array([0.0, 1.0])
        ax.plot(*(P[nm] - 0.06 * s * eje), ">" if e == "x" else "^", color="0.3", ms=11, zorder=3)
        if reac is not None:
            v = reac[f"{nm}{e}"]
            base = P[nm] - 0.06 * s * eje            # junto al símbolo del apoyo
            cola = base - a * eje
            if v > 0:                                # la flecha apunta en el sentido de la reacción
                _flecha(ax, cola, base, _COLORES[2], "-.", 2.2)
            elif v < 0:
                _flecha(ax, base, cola, _COLORES[2], "-.", 2.2)
            # una reacción cero no se dibuja como flecha: solo su rótulo
            ax.annotate(f"{nm}{e} = {_dec_o_cifras(v, n_cifras)} {unidad}", xy=cola if e == "y" else (cola + P[nm]) / 2,
                        xytext=(0, -10) if e == "y" else (0, 9), textcoords="offset points", fontsize=9,
                        ha="center", va="top" if e == "y" else "bottom",
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))
    for ls, color, lw, etiqueta in (("-", _COLORES[0], 2.8, "tensión (T): continua"),
                                    ("--", _COLORES[1], 2.8, "compresión (C): discontinua"),
                                    (":", "0.45", 2.2, "fuerza cero: punteada"),
                                    ("-", "black", 2.6, "carga (flecha negra)"),
                                    ("-.", _COLORES[2], 2.2, "reacción (rayas y puntos)")):
        ax.plot([], [], linestyle=ls, color=color, lw=lw, label=etiqueta)
    ax.set_xlim(xy[:, 0].min() - 0.35 * s, xy[:, 0].max() + 0.35 * s)
    ax.set_ylim(xy[:, 1].min() - 0.35 * s, xy[:, 1].max() + 0.35 * s)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    nota = ("" if not muchas else " (los nombres y las fuerzas de las barras están en el texto)" if sin_rotulos else
            " (las fuerzas de cada barra están en el texto)" if fuerzas else "")
    ax.set_title(f"Armadura: {resultado['diagnostico']}" + nota, fontsize=10)
    ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
    plt.tight_layout()
    plt.show()

**Recorrido paso a paso.** (1) Un triángulo: $A(0,0)$ articulado, $B(4,0)$ sobre un rodillo (solo reacción vertical) y $C(2,\,1.5)$ con una carga de 10 kN hacia abajo. Las barras $AC$ y $BC$ miden 2.5 m. En $C$: $\sum F_x=0$ da $F_{AC}=F_{BC}$ y $\sum F_y=0$ da $-0.6\,F_{AC}-0.6\,F_{BC}-10=0$, así que $F_{AC}=F_{BC}=-\tfrac{25}{3}$ kN (compresión); en $B$, $F_{AB}=\tfrac{20}{3}$ kN (tensión), y las reacciones son $A_x=0$, $A_y=B_y=5$ kN. (2) Una armadura Warren: $A(0,0)$, $B(2,0)$, $C(4,0)$, $D(1,2)$, $E(3,2)$, barras $AB$, $BC$, $AD$, $BD$, $BE$, $CE$, $DE$, $A$ articulado, $C$ sobre un rodillo y 6 kN hacia abajo en $B$: $F_{AB}=F_{BC}=1.5$, $F_{AD}=F_{CE}=-1.5\sqrt5$, $F_{BD}=F_{BE}=1.5\sqrt5$, $F_{DE}=-3$ kN, y $A_y=C_y=3$ kN. Las dos cumplen $b+r=2n$ y su matriz es invertible.

In [ ]:
print("(1) Triángulo")
_tri = armadura({"A": (0, 0), "B": (4, 0), "C": (2, 1.5)}, [("A", "B"), ("A", "C"), ("B", "C")],
                [("A", "x"), ("A", "y"), ("B", "y")], {"C": (0, -10)})
_informe_armadura(_tri, "kN", 4)
dibuja_armadura(_tri, "kN")
print("=" * 90 + "\n\n(2) Armadura Warren")
_NODOS_W = "A, 0, 0\nB, 2, 0\nC, 4, 0\nD, 1, 2\nE, 3, 2"
_BARRAS_W = "A, B\nB, C\nA, D\nB, D\nB, E\nC, E\nD, E"
_APOYOS_W = "A, x\nA, y\nC, y"
_warren = armadura(_NODOS_W, _BARRAS_W, _APOYOS_W, "B, 0, -6")
_informe_armadura(_warren, "kN", 4)
dibuja_armadura(_warren, "kN")

**Calculadora.** Escribe los nodos (`nombre, x, y` en metros), las barras (`nombre, nombre`), los apoyos (`nombre, x` o `nombre, y`; un articulado son dos renglones) y las cargas (`nombre, Fx, Fy`), uno por renglón, y la unidad de fuerza. Hasta 20 nodos. Prueba quitar el apoyo `A, x` (mecanismo), cambiarlo por `C, y` (el conteo cuadra, pero la matriz es singular) y agregar un nodo `D, 4, 1.5` con las barras `C, D` y `B, D`: a $D$ no llega ninguna carga y solo llegan dos barras no colineales, así que las dos quedan con fuerza cero.

In [ ]:
def calculadora_armadura(nodos_txt, barras_txt, apoyos_txt, cargas_txt, unidad, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        try:
            u = nombre_unidad(unidad, "fuerza", "kN o N")
            res = armadura(nodos_txt, barras_txt, apoyos_txt, cargas_txt, n_cifras=n_cifras)
        except ValueError as err:
            _reporta(err); return
        _aviso_coma(nodos_txt, cargas_txt)
        _informe_armadura(res, u, n_cifras)
    dibuja_armadura(res, u, n_cifras)


_caja = lambda alto: widgets.Layout(width="250px", height=alto)
_CONTROLES_4 = dict(
    nodos_txt=widgets.Textarea(value="A, 0, 0\nB, 4, 0\nC, 2, 1.5", description="nodos (m):", continuous_update=False,
                               layout=_caja("110px")),
    barras_txt=widgets.Textarea(value="A, B\nA, C\nB, C", description="barras:", continuous_update=False,
                                layout=_caja("110px")),
    apoyos_txt=widgets.Textarea(value="A, x\nA, y\nB, y", description="apoyos:", continuous_update=False,
                                layout=_caja("90px")),
    cargas_txt=widgets.Textarea(value="C, 0, -10", description="cargas:", continuous_update=False, layout=_caja("70px")),
    unidad=widgets.Text(value="kN", description="unidad de fuerza:", continuous_update=False,
                        style={"description_width": "initial"}),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_armadura, **_CONTROLES_4);

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
_s5 = sp.sqrt(5)
_esperado_W = {"AB": sp.Rational(3, 2), "BC": sp.Rational(3, 2), "AD": -sp.Rational(3, 2) * _s5, "BD": sp.Rational(3, 2) * _s5,
               "BE": sp.Rational(3, 2) * _s5, "CE": -sp.Rational(3, 2) * _s5, "DE": -3}
_iguales = lambda a, b: set(a) == set(b) and all(sp.simplify(a[k] - b[k]) == 0 for k in a)
_cuadro = "A, 0, 0\nB, 2, 0\nC, 2, 2\nD, 0, 2"
PRUEBAS_4 = [
    ("triángulo del recorrido → AB = 20/3, AC = BC = -25/3, reacciones (Ax, Ay, By) = (0, 5, 5), determinada",
     lambda: _tri["fuerzas"] == {"AB": sp.Rational(20, 3), "AC": sp.Rational(-25, 3), "BC": sp.Rational(-25, 3)}
             and list(_tri["reacciones"].values()) == [0, 5, 5] and _tri["diagnostico"] == "determinada"
             and _tri["tipo"] == {"AB": "T", "AC": "C", "BC": "C"}
             and "punto decimal" in _mensaje(lambda: armadura("A, 0, 0\nB, 4, 0\nC, 2, 1,5", "A, B\nA, C\nB, C",
                                                              "A, x\nA, y\nB, y", "C, 0, -10"))),
    ("Warren → AB = BC = 3/2, AD = CE = -(3/2)√5, BD = BE = (3/2)√5, DE = -3, Ay = Cy = 3",
     lambda: _iguales(_warren["fuerzas"], _esperado_W) and _warren["reacciones"] == {"Ax": 0, "Ay": 3, "Cy": 3}),
    ("comprobacion_global: suma de fuerzas externas y momento respecto al primer nodo son cero en las pruebas 1 y 2",
     lambda: all(r["comprobacion_global"]["cero"] and r["comprobacion_global"]["suma_fuerzas"] == sp.zeros(2, 1)
                 and r["comprobacion_global"]["momento"] == 0 for r in (_tri, _warren))),
    ("triángulo A(0, 0), B(4, 0), C(2, 2) con tres apoyos verticales y carga en C → «inestable» (rango 5 < 6), sin traza",
     lambda: (lambda r: r["diagnostico"].startswith("inestable") and r["rango"] == 5 and r["conteo"] == (3, 3, 3)
                        and r["fuerzas"] is None)(
         armadura("A, 0, 0\nB, 4, 0\nC, 2, 2", "A, B\nA, C\nB, C", "A, y\nB, y\nC, y", "C, 0, -6"))
             and "INESTABLE" in _captura(calculadora_armadura, "A, 0, 0\nB, 4, 0\nC, 2, 2", "A, B\nA, C\nB, C",
                                         "A, y\nB, y\nC, y", "C, 0, -6", "kN", 4)[0]),
    ("cuadrado A(0, 0), B(2, 0), C(2, 2), D(0, 2) con sus 4 lados y apoyos A x, A y, B y → «mecanismo por conteo»",
     lambda: (lambda r: r["diagnostico"] == "mecanismo por conteo" and r["conteo"] == (4, 4, 3))(
         armadura(_cuadro, "A, B\nB, C\nC, D\nD, A", "A, x\nA, y\nB, y", "D, 1, 0"))),
    ("el mismo cuadrado con sus dos diagonales → «hiperestática por conteo»; con solo sus 4 lados y apoyos A x, A y, "
     "B x, B y, C y → «hiperestática por conteo e inestable (rango < 2n)»",
     lambda: (lambda r: r["diagnostico"] == "hiperestática por conteo" and r["conteo"] == (4, 6, 3))(
         armadura(_cuadro, "A, B\nB, C\nC, D\nD, A\nA, C\nB, D", "A, x\nA, y\nB, y", "D, 1, 0"))
             and (lambda r: r["diagnostico"] == "hiperestática por conteo e inestable (rango < 2n)"
                            and r["conteo"] == (4, 4, 5) and r["rango"] < 8 and r["fuerzas"] is None)(
                 armadura(_cuadro, "A, B\nB, C\nC, D\nD, A", "A, x\nA, y\nB, x\nB, y\nC, y", "D, 1, 0"))),
    ("linealidad: duplicar la carga de la Warren duplica todas las fuerzas y reacciones",
     lambda: (lambda r: _iguales(r["fuerzas"], {k: 2 * v for k, v in _esperado_W.items()})
                        and r["reacciones"] == {"Ax": 0, "Ay": 6, "Cy": 6})(
         armadura(_NODOS_W, _BARRAS_W, _APOYOS_W, "B, 0, -12"))),
    ("prueba de humo: la calculadora corre con sus valores iniciales y con la Warren, sin error y sin «Revisa la entrada»",
     lambda: _humo(calculadora_armadura, _CONTROLES_4)
             and _humo(calculadora_armadura, _CONTROLES_4, nodos_txt=_NODOS_W, barras_txt=_BARRAS_W,
                       apoyos_txt=_APOYOS_W, cargas_txt="B, 0, -6", unidad="N", n_cifras=6)),
]
corre_pruebas(PRUEBAS_4, 4)

**Contrasta (sección 4).** Para la armadura con $A(0,0)$ articulado, $B(2,0)$ sobre un rodillo y $C(1,1)$ con una carga de 4 kN hacia abajo, calcula a mano las fuerzas (empieza por el nodo $C$) y escribe `mis_fuerzas = (AB, AC, BC)` en kN, con tensión positiva.

In [ ]:
mis_fuerzas = None     # tres números (AB, AC, BC) en kN, por ejemplo (1.5, -2.5, 0)

if mis_fuerzas is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mias = vector(mis_fuerzas, "mis_fuerzas")
    except ValueError as err:
        print(f"Revisa tu respuesta: {str(err).rstrip('.')}.")
    else:
        if mias.rows != 3:          # respuesta imposible: la referencia todavía no se muestra
            print(f"La armadura tiene tres barras, así que son tres fuerzas, AB, AC y BC (escribiste {mias.rows}): "
                  "revisa tu respuesta y vuelve a ejecutar la celda.")
        else:
            ref = armadura("A, 0, 0\nB, 2, 0\nC, 1, 1", "A, B\nA, C\nB, C", "A, x\nA, y\nB, y", "C, 0, -4")
            F = [ref["fuerzas"][k] for k in ("AB", "AC", "BC")]
            print("La calculadora da: " + ",  ".join(_magnitud(k, v, "kN", 4) for k, v in zip(("AB", "AC", "BC"), F)) + ".")
            if all(abs(float(a - b)) <= 0.01 for a, b in zip(mias, F)):
                print("Coinciden (con tolerancia de 0.01 kN).")
            elif all(abs(float(a + b)) <= 0.01 for a, b in zip(mias, F)):
                print("NO coinciden: tienes los valores correctos con los signos al revés. Revisa la convención: la "
                      "fuerza es positiva si la barra está en tensión (jala a sus nodos hacia ella).")
            else:
                print("NO coinciden: en C, las barras AC y BC forman 45° con la horizontal; por simetría son iguales, y "
                      "ΣFy = 0 da la suma de sus componentes verticales. Después, ΣFx = 0 en B da AB.")
            print("\nEl procedimiento completo:")
            _informe_armadura(ref, "kN", 4)

## 5. Ajuste de datos por mínimos cuadrados

**Repaso de la Unidad 5.** Con más ecuaciones que incógnitas, $A\boldsymbol\theta=\mathbf y$ casi nunca tiene solución; mínimos cuadrados busca el $\hat{\boldsymbol\theta}$ que hace más pequeño $\|\mathbf y-A\boldsymbol\theta\|$ y lo da la solución de las **ecuaciones normales** $A^TA\,\hat{\boldsymbol\theta}=A^T\mathbf y$. En la práctica se usa `np.linalg.lstsq`, que no forma $A^TA$. Lo único que se pide es que el modelo sea **lineal en los parámetros**: cada dato da un renglón de $A$.

**Recta por el origen.** La ley de Ohm $V=R\,I$, la de Hooke $F=k\,x$ o la dilatación $\Delta L=\alpha L_0\,\Delta T$ tienen la forma $y=k\,x$: $A$ es una sola columna (los $x_i$) y las ecuaciones normales se reducen a

$$k=\frac{\sum_i x_iy_i}{\sum_i x_i^2}.$$

**Modelo discreto de primer orden.** Un horno, un motor o un tanque que se mide cada $\Delta t$ segundos se describe muchas veces con

$$x[k+1]=a\,x[k]+b\,u[k],$$

donde $u[k]$ es la entrada (el voltaje, la apertura de una válvula) y $x[k]$ es la **desviación de la salida respecto al reposo** (por ejemplo, la temperatura del horno sobre la del ambiente, no la temperatura absoluta). Como el modelo no tiene término constante, con la temperatura absoluta el ajuste queda sesgado: el modelo diría que sin entrada la salida tiende a 0 °C. Es lineal en $a$ y $b$: **cada paso medido es un renglón** $\big(x[k],\ u[k]\big)$ de $A$, con $x[k+1]$ del lado derecho. Con $N$ entradas hacen falta $N+1$ salidas, $x[0],\dots,x[N]$. De $a$ y $b$ salen dos constantes físicas, solo si $0<a<1$ (el sistema se estabiliza sin oscilar):

$$\tau=-\frac{\Delta t}{\ln a}\ \ (\text{constante de tiempo}),\qquad K=\frac{b}{1-a}\ \ (\text{cuánto sube la salida en estado estacionario por unidad de entrada}).$$

Si $a\le 0$ o $a\ge 1$ no hay constante de tiempo: con $a=0$ la salida copia la entrada un paso después ($K=b$); con $-1<a<0$ converge a $K\,u$ oscilando, pero $\tau$ no existe porque $a=e^{-\Delta t/\tau}$ tendría que ser positivo; con $a\le-1$ o $a\ge 1$ no se estabiliza.

**Validar.** La predicción a un paso usa el dato medido $x[k]$ para predecir $x[k+1]$, así que casi siempre se ve bien. La **simulación libre** es una prueba más exigente: arrancar en $x[0]$ y dejar que el modelo use sus propias predicciones; si se aleja de los datos, el modelo no sirve aunque el error a un paso sea chico. Mejor todavía es comparar el modelo con datos que no se usaron en el ajuste (por ejemplo, otra corrida del mismo equipo).

**Columnas casi dependientes.** Si una columna de $A$ es casi un múltiplo (o una combinación) de otras, el número de condición de $A$ es grande: el ajuste puede verse bien, pero los coeficientes **por separado** no son confiables (al quitar un dato cambian mucho), y las predicciones solo valen cerca de los datos. La calculadora avisa cuando el número de condición pasa de 1000.

In [ ]:
def _pares(x, y, nx, ny):
    xs, ys = datos(x, nx), datos(y, ny)
    if len(xs) != len(ys):
        raise ValueError(f"hay {_cuenta(len(xs), 'dato', 'datos')} de x y {len(ys)} de y: debe haber un y por cada x"
                         f"{pista_coma(x) or pista_coma(y) or _pista_decimal(x) or _pista_decimal(y)}")
    return xs, ys


def ajuste_por_origen(x, y):
    """Recta por el origen y = k x por mínimos cuadrados: k = Σxy/Σx². Exacto con hasta 12 datos; con más, en
    decimales. Diccionario con k, residuos (y - kx), SSE, RMSE, Sxy, Sxx, exacto, x, y y n."""
    xs, ys = _pares(x, y, "los datos de x", "los datos de y")
    if all(v == 0 for v in xs):
        raise ValueError("todos los x son 0: la recta por el origen no queda determinada")
    n = len(xs)
    exacto = n <= _MAX_EXACTO
    if exacto:
        sxy, sxx = sum(a * b for a, b in zip(xs, ys)), sum(a * a for a in xs)
        k = sxy / sxx
        res = [b - k * a for a, b in zip(xs, ys)]
        sse = sum(r ** 2 for r in res)
        rmse = sp.sqrt(sse / n)
    else:
        X, Y = np.array([float(v) for v in xs]), np.array([float(v) for v in ys])
        sxy, sxx = float(X @ Y), float(X @ X)
        k = sxy / sxx
        res = list(Y - k * X)
        sse = float(np.dot(res, res))
        rmse = math.sqrt(sse / n)
    return {"k": k, "residuos": res, "SSE": sse, "RMSE": rmse, "Sxy": sxy, "Sxx": sxx, "exacto": exacto,
            "x": xs, "y": ys, "n": n}


def ajuste_lineal(columnas, y, nombres=None):
    """Ajuste por mínimos cuadrados de y ≈ θ1·c1 + θ2·c2 + ... con las columnas dadas (listas de números del mismo
    largo que y), en decimales con np.linalg.lstsq. Diccionario con coeficientes, nombres, residuos, RMSE,
    condicion (número de condición de A) y avisos: si condicion pasa de 1000, «columnas casi dependientes: los
    coeficientes no son confiables por separado». Si las columnas son dependientes, ValueError."""
    if isinstance(columnas, (str, np.ndarray)) or not isinstance(columnas, (list, tuple)) or not columnas:
        raise ValueError("escribe las columnas como una lista de listas de números, una lista por columna")
    if len(columnas) > 6:
        raise ValueError(f"escribiste {len(columnas)} columnas; esta función acepta hasta 6")
    cols = [datos(c, f"la columna {j}") for j, c in enumerate(columnas, 1)]
    ys = datos(y, "los datos de y")
    nombres = list(nombres) if nombres is not None else [f"θ{j}" for j in range(1, len(cols) + 1)]
    if len(nombres) != len(cols):
        raise ValueError(f"hay {_cuenta(len(cols), 'columna', 'columnas')} y {_cuenta(len(nombres), 'nombre', 'nombres')}: "
                         "debe haber un nombre por columna")
    for j, c in enumerate(cols, 1):
        if len(c) != len(ys):
            raise ValueError(f"la columna {j} tiene {_cuenta(len(c), 'dato', 'datos')} y y tiene {len(ys)}: deben ser "
                             "los mismos")
    if len(ys) < len(cols):
        raise ValueError(f"hay {_cuenta(len(ys), 'dato', 'datos')} y {len(cols)} coeficientes: hacen falta al menos "
                         "tantos datos como coeficientes")
    A = np.column_stack([[float(v) for v in c] for c in cols])
    Y = np.array([float(v) for v in ys])
    with np.errstate(all="ignore"):
        rango = int(np.linalg.matrix_rank(A))
        if rango < A.shape[1]:
            raise Diagnostico(f"las columnas son dependientes (rango {rango} < {A.shape[1]}): los coeficientes no quedan "
                             "determinados")
        cond = float(np.linalg.cond(A))
        coef = np.linalg.lstsq(A, Y, rcond=None)[0]
    pred = A @ coef
    res = Y - pred
    avisos = []
    if cond > _COND_AVISO:
        avisos.append("columnas casi dependientes: los coeficientes no son confiables por separado")
    return {"coeficientes": [float(c) for c in coef], "nombres": nombres, "residuos": [float(r) for r in res],
            "predichos": [float(p) for p in pred], "RMSE": math.sqrt(float(res @ res) / len(Y)), "condicion": cond,
            "avisos": avisos, "A": A}


def _decimal(v, nombre):
    """Un número como decimal de Python: el texto se lee con numero(); un número (de Python, numpy o sympy) se
    convierte y debe ser finito. Mensajes en español."""
    if isinstance(v, str):
        v = numero(v, nombre)
    try:
        f = float(v)
    except (TypeError, ValueError):
        raise ValueError(f"{nombre} debe ser un número ({_AYUDA_NUMERO})") from None
    if not math.isfinite(f):
        raise ValueError(f"{nombre} debe ser un número finito (escribiste {v})")
    return f


def simula_arx(a, b, u, x0):
    """Simulación libre de x[k+1] = a·x[k] + b·u[k] desde x[0] = x0: devuelve [x[0], x[1], ..., x[N]] (N = len(u)).
    Exacta (fracciones) con hasta 12 pasos y datos racionales; con más, en decimales. Si la simulación diverge, los
    valores que ya no caben en los decimales quedan como inf (identifica_arx lo avisa)."""
    us = datos(u, "la entrada u")
    if len(us) <= _MAX_EXACTO:
        a, b, x = numero(a, "a"), numero(b, "b"), numero(x0, "x0")
        salida = [x]
        for uk in us:
            x = a * x + b * uk
            salida.append(x)
        return salida
    a, b, x = _decimal(a, "a"), _decimal(b, "b"), _decimal(x0, "x0")
    salida = [x]
    for uk in us:
        x = a * x + b * float(uk) if math.isfinite(x) else math.inf
        salida.append(x if math.isfinite(x) else math.inf)
    return salida


def identifica_arx(u, x, dt, n_cifras=4):
    """Identifica a y b del modelo x[k+1] = a·x[k] + b·u[k] por mínimos cuadrados: cada paso es un renglón (x[k], u[k])
    de A y x[k+1] va del lado derecho. Necesita len(x) = len(u) + 1. Exacto con hasta 12 pasos; con más, en decimales
    con np.linalg.lstsq. Diccionario con a, b, tau = -Δt/ln a y K = b/(1 - a) (solo si 0 < a < 1; si no, None y un
    aviso; con a = 0, K = b y con -1 < a < 0, K = b/(1 - a), sin τ), A, y, prediccion (a un paso), residuos, RMSE_1 (a un
    paso), simulacion (libre, desde x[0]), RMSE_libre, cond (número de condición de A), avisos, exacto, u, x, dt y n.
    n_cifras: las cifras de los decimales en los avisos."""
    us, xs = datos(u, "la entrada u"), datos(x, "la salida x")
    if len(xs) != len(us) + 1:
        raise ValueError(f"x debe tener un dato más que u (x[0], ..., x[N] y u[0], ..., u[N-1]): escribiste "
                         f"{_cuenta(len(us), 'dato', 'datos')} de u y {len(xs)} de x"
                         f"{pista_coma(u) or pista_coma(x) or _pista_decimal(u) or _pista_decimal(x)}")
    n = len(us)
    if n < 2:
        raise ValueError("hacen falta al menos dos pasos (dos ecuaciones) para encontrar a y b: escribe al menos 2 datos "
                         "de u y 3 de x")
    h = numero(dt, "Δt")
    if h <= 0:
        raise ValueError(f"Δt debe ser positivo (escribiste {_ntxt(h)})")
    exacto = n <= _MAX_EXACTO
    dependientes = ("las columnas x[k] y u[k] de A son dependientes (por ejemplo, x es proporcional a u en todos los "
                    "pasos): los datos no distinguen a de b; usa una entrada que cambie (escalones) o más pasos")
    if exacto:
        A, Y = sp.Matrix([[xs[k], us[k]] for k in range(n)]), sp.Matrix(xs[1:])
        G = A.T * A
        if G.det() == 0:
            raise Diagnostico(dependientes)
        a, b = G.LUsolve(A.T * Y)
        pred = list(A * sp.Matrix([a, b]))
        res = [v - p for v, p in zip(xs[1:], pred)]
        rmse1 = sp.sqrt(sum(r ** 2 for r in res) / n)
        cond = float(np.linalg.cond(_flotante(A)))
    else:
        A = np.column_stack([[float(v) for v in xs[:-1]], [float(v) for v in us]])
        Y = np.array([float(v) for v in xs[1:]])
        with np.errstate(all="ignore"):
            if int(np.linalg.matrix_rank(A)) < 2:
                raise Diagnostico(dependientes)
            cond = float(np.linalg.cond(A))
            a, b = (float(c) for c in np.linalg.lstsq(A, Y, rcond=None)[0])
        pred = list(A @ np.array([a, b]))
        res = [float(v) - p for v, p in zip(Y, pred)]
        rmse1 = math.sqrt(sum(r * r for r in res) / n)
    sim = simula_arx(a, b, us, xs[0])
    diverge = not all(math.isfinite(float(s)) and abs(float(s)) < 1e100 for s in sim)
    dif = [v - s for v, s in zip(xs[1:], sim[1:])]
    if diverge:
        rmse_libre = math.inf
    elif exacto:
        rmse_libre = sp.sqrt(sum(d ** 2 for d in dif) / n)
    else:
        rmse_libre = math.sqrt(sum(float(d) ** 2 for d in dif) / n)
    avisos, tau, K = [], None, None
    if 0 < a < 1:
        K = b / (1 - a)
        tau = -h / sp.log(a) if exacto else -float(h) / math.log(a)
    elif a == 0:
        K = b
        avisos.append("a = 0: la salida copia la entrada un paso después (x[k+1] = b·u[k]), sin dinámica; K = b y no "
                      "hay constante de tiempo τ")
    elif -1 < a < 0:
        K = b / (1 - a)
        avisos.append("-1 < a < 0: la salida converge a K·u oscilando (cambia de lado en cada paso); K = b/(1 - a) "
                      "existe, pero no hay constante de tiempo τ, porque a = e^(-Δt/τ) tendría que ser positivo")
    elif a <= -1:
        avisos.append("a ≤ -1: la salida oscila sin amortiguarse o crece en cada paso; el modelo no se estabiliza, así "
                      "que no hay constante de tiempo τ ni ganancia K")
    else:
        avisos.append("a ≥ 1: el modelo no se estabiliza (la salida crece sin límite o no regresa), así que no hay "
                      "constante de tiempo τ ni ganancia K")
    if cond > _COND_AVISO:
        avisos.append(f"el número de condición de A es ≈ {cifras(cond, n_cifras)}: columnas casi dependientes, a y b "
                      "no son confiables por separado")
    escala = max([abs(float(v)) for v in xs] + [1e-300])
    if diverge:
        avisos.append("la simulación libre diverge (sus valores crecen sin límite): el modelo no reproduce el sistema")
    elif float(rmse_libre) > 5 * float(rmse1) and float(rmse_libre) > 1e-6 * escala:
        avisos.append("la simulación libre se aleja de los datos mucho más que la predicción a un paso: el modelo no "
                      "reproduce bien el sistema")
    return {"a": a, "b": b, "tau": tau, "K": K, "A": A, "y": Y, "prediccion": pred, "residuos": res, "RMSE_1": rmse1,
            "simulacion": sim, "RMSE_libre": rmse_libre, "cond": cond, "avisos": avisos, "exacto": exacto, "u": us,
            "x": xs, "dt": h, "n": n}


def _informe_origen(res, k_nombre, unidad_k, n_cifras):
    print(f"{_cuenta(res['n'], 'dato', 'datos')}; modelo y = {k_nombre}·x (A es la columna de los x).")
    if res["exacto"]:
        print(f"Σ x·y = {_ntxt(res['Sxy'])},  Σ x² = {_ntxt(res['Sxx'])}  →  {k_nombre} = Σxy/Σx² = "
              f"{_cantidad(res['k'], unidad_k, n_cifras)}")
        print(f"Residuos y - {k_nombre}·x = {_tupla_u(sp.Matrix(res['residuos']), '', n_cifras)}")
        print(f"{_magnitud('SSE', res['SSE'], '', n_cifras)},  {_magnitud('RMSE', res['RMSE'], '', n_cifras)}")
    else:
        print(f"{k_nombre} ≈ {cifras(res['k'], n_cifras)} {unidad_k};  RMSE ≈ {cifras(res['RMSE'], n_cifras)}")


def _informe_arx(res, ut, n_cifras):
    n, a, b = res["n"], res["a"], res["b"]
    nt = lambda v: _ntxt(v, cif=n_cifras)
    print(f"Modelo x[k+1] = a·x[k] + b·u[k];  N = {n} pasos (una ecuación por paso), Δt = {nt(res['dt'])} {ut}.")
    if n <= 6:
        for k in range(n):
            print(f"  k = {k}:  x[{k + 1}] = a·x[{k}] + b·u[{k}]  →  {nt(res['x'][k + 1])} = "
                  f"{nt(res['x'][k])}·a + {nt(res['u'][k])}·b")
    if res["exacto"]:
        print("Cálculo exacto con fracciones: ecuaciones normales AᵀA θ = Aᵀy, con θ = (a, b).")
        if n <= 6:
            muestra("A (columnas x[k] y u[k])", res["A"], n_cifras, tex=r"A\ (\text{columnas } x[k] \text{ y } u[k])")
    else:
        print(f"Cálculo en decimales con np.linalg.lstsq, porque hay más de {_MAX_EXACTO} pasos.")
    print(f"{_magnitud('a', a, '', n_cifras)},  {_magnitud('b', b, '', n_cifras)}")
    print(f"Número de condición de A ≈ {cifras(res['cond'], n_cifras)}")
    if res["K"] is not None:
        print(_magnitud("K = b/(1 - a)" if a != 0 else "K = b", res["K"], "", n_cifras) +
              "  (subida de la salida en estado estacionario por unidad de entrada)")
    if res["tau"] is not None:
        print(f"τ = -Δt/ln a = -{nt(res['dt'])}/ln({_dec_o_cifras(a, n_cifras)}) ≈ {cifras(res['tau'], n_cifras)} {ut}  "
              "(constante de tiempo)")
    medida = lambda v: "= 0" if (isinstance(v, sp.Basic) and v == 0) else f"≈ {cifras(v, n_cifras)}"
    print(f"RMSE a un paso {medida(res['RMSE_1'])};  RMSE de la simulación libre {medida(res['RMSE_libre'])}" +
          ("  (el modelo pasa exactamente por todos los datos)" if res["RMSE_libre"] == 0 else ""))
    for aviso in res["avisos"]:
        print("Aviso:", aviso + ".")


def dibuja_identificacion(resultado, unidad_t="s", n_cifras=4):
    """Tres paneles: la salida medida con la simulación libre y la predicción a un paso; la entrada en escalones; y
    los residuos a un paso."""
    n, h = resultado["n"], float(resultado["dt"])
    t = np.arange(n + 1) * h
    X = np.array([float(v) for v in resultado["x"]])
    S = np.array([float(v) for v in resultado["simulacion"]])
    S[~np.isfinite(S) | (np.abs(S) > 1e100)] = np.nan             # una simulación que diverge no se dibuja completa
    P = np.array([float(v) for v in resultado["prediccion"]])
    U = np.array([float(v) for v in resultado["u"]])
    R = np.array([float(v) for v in resultado["residuos"]])
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8.6, 8.4), sharex=True,
                                        gridspec_kw={"height_ratios": [2.3, 1.0, 1.2]})
    marca = n <= 60
    ax1.plot(t, X, "o", color="black", ms=6 if marca else 3, label="x medida", zorder=3)
    rmse = lambda v: "= 0" if (isinstance(v, sp.Basic) and v == 0) else f"≈ {cifras(v, n_cifras)}"
    ax1.plot(t, S, "--", color=_COLORES[0], lw=2.2, marker="s" if marca else None, ms=4,
             label=f"simulación libre desde x[0] (RMSE {rmse(resultado['RMSE_libre'])})")
    ax1.plot(t[1:], P, ":", color=_COLORES[1], lw=2.0, marker="^" if marca else None, ms=5,
             label=f"predicción a un paso (RMSE {rmse(resultado['RMSE_1'])})")
    ax1.set_ylabel("salida x"); ax1.grid(alpha=0.4); ax1.legend(fontsize=8, loc="best")
    igual = lambda v: ("= " if isinstance(v, sp.Basic) and _decimal_exacto(v, n_cifras) else "≈ ") + _dec_o_cifras(v, n_cifras)
    ax1.set_title(f"x[k+1] = a·x[k] + b·u[k] con a {igual(resultado['a'])} y b {igual(resultado['b'])}"
                  + (f";  τ ≈ {cifras(resultado['tau'], n_cifras)} {unidad_t}" if resultado["tau"] is not None else ""),
                  fontsize=10)
    ax2.step(t, np.append(U, U[-1]), where="post", color=_COLORES[2], lw=2.0, label="entrada u (escalones)")
    ax2.set_ylabel("entrada u"); ax2.grid(alpha=0.4); ax2.legend(fontsize=8, loc="best")
    escala = max(float(np.max(np.abs(X))), 1e-300)
    if np.all(np.abs(R) <= 1e-12 * escala):
        R = np.zeros_like(R)
    ax3.vlines(t[1:], 0, R, colors="0.35", lw=1.6)
    ax3.plot(t[1:], R, "s", color=_COLORES[3], ms=5, label="residuo a un paso x[k+1] - predicción")
    ax3.axhline(0, color="black", lw=1.0, ls="--")
    ax3.set_ylabel("residuo"); ax3.set_xlabel(f"tiempo ({unidad_t})"); ax3.grid(alpha=0.4)
    ax3.legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()

**Recorrido paso a paso.** (1) Ley de Ohm por el origen: con $I=0.1,\ 0.2,\ 0.3,\ 0.4$ A se miden $V=1.9,\ 4.1,\ 6.0,\ 8.1$ V; $\sum IV=6.05$ y $\sum I^2=0.30$, así que $R=6.05/0.30=\tfrac{121}{6}\approx 20.17\ \Omega$. (2) Modelo de primer orden con $u=1,1,1$ y $x=0,\ 2,\ 3.6,\ 4.88$: la primera ecuación, $2=a\cdot0+b\cdot1$, da $b=2$, y la segunda, $3.6=2a+2$, da $a=0.8$; la tercera también se cumple, así que $a$ y $b$ son exactos. Entonces $K=2/(1-0.8)=10$ y, con $\Delta t=1$ s, $\tau=-1/\ln 0.8\approx 4.48$ s. (3) Una serie de 12 pasos generada con $a=0.95$, $b=0.5$, la entrada $u=1,1,1,0,0,0,0,0,0,1,1,1$ y $x[0]=0$: la identificación recupera $a$ y $b$, y la simulación libre pasa por todos los datos. (4) Columnas casi dependientes: con $x_2\approx 10\,x_1$, el ajuste se ve bien, pero al quitar un solo dato los coeficientes cambian mucho.

In [ ]:
print("(1) Ley de Ohm, V = R·I")
_ohm = ajuste_por_origen("0.1 0.2 0.3 0.4", "1.9 4.1 6.0 8.1")
_informe_origen(_ohm, "R", "Ω", 4)
print("\n" + "=" * 90 + "\n\n(2) Modelo de primer orden con datos exactos")
_arx = identifica_arx("1 1 1", "0 2 3.6 4.88", 1)
_informe_arx(_arx, "s", 4)
dibuja_identificacion(_arx, "s")
print("=" * 90 + "\n\n(3) Serie sintética de 12 pasos con a = 0.95 y b = 0.5")
_u_sint = [1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1]
_x_sint = simula_arx(0.95, 0.5, _u_sint, 0)
print("x generada:", ", ".join(cifras(v, 4) for v in _x_sint))
_sint = identifica_arx(_u_sint, _x_sint, 1)
_informe_arx(_sint, "s", 4)
dibuja_identificacion(_sint, "s")

In [ ]:
print("(4) Columnas casi dependientes: x2 = 10·x1 + (±0.01)")
_x1 = [1, 2, 3, 4, 5, 6]
_x2 = [10 * a + d for a, d in zip(_x1, [0.01, -0.01, 0.01, -0.01, 0.01, -0.01])]
_yc = [3.0, 6.2, 8.9, 12.1, 15.1, 17.6]
_todos = ajuste_lineal([_x1, _x2], _yc, ["θ1", "θ2"])
_sin_ultimo = ajuste_lineal([_x1[:-1], _x2[:-1]], _yc[:-1], ["θ1", "θ2"])
for _nombre, _r in (("con los 6 datos", _todos), ("sin el último dato", _sin_ultimo)):
    print(f"  {_nombre}: θ1 ≈ {cifras(_r['coeficientes'][0], 4)}, θ2 ≈ {cifras(_r['coeficientes'][1], 4)};  "
          f"RMSE ≈ {cifras(_r['RMSE'], 4)};  número de condición ≈ {cifras(_r['condicion'], 4)}")
print("  Aviso:", _todos["avisos"][0] + ".")
print(f"  La combinación θ1 + 10·θ2, que es lo que los datos sí determinan, casi no cambia: "
      f"{cifras(_todos['coeficientes'][0] + 10 * _todos['coeficientes'][1], 4)} y "
      f"{cifras(_sin_ultimo['coeficientes'][0] + 10 * _sin_ultimo['coeficientes'][1], 4)}.")

**Calculadora.** Escribe la entrada $u[0],\dots,u[N-1]$ y la salida $x[0],\dots,x[N]$ (un dato más que $u$), separadas por espacios o comas, el paso de tiempo $\Delta t$ y su unidad. La salida es la **desviación respecto al reposo** (por ejemplo, la temperatura sobre la del ambiente): como el modelo no tiene término constante, con valores absolutos el ajuste queda sesgado. Con hasta 12 pasos el cálculo es exacto; con más (hasta 500), en decimales. Prueba agregarle ruido a la serie sintética (cambia algunos datos en la segunda cifra decimal), una entrada constante con la salida también constante (columnas dependientes) y una salida que crece sin parar (por ejemplo `0 1 2.2 3.64` con `1 1 1`).

In [ ]:
def calculadora_arx(u_txt, x_txt, dt_txt, unidad_t, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; las gráficas van después
        try:
            ut = nombre_unidad(unidad_t, "tiempo", "s o min")
            res = identifica_arx(u_txt, x_txt, dt_txt, n_cifras=n_cifras)
        except ValueError as err:
            _reporta(err); return
        _aviso_coma(u_txt, x_txt)
        _informe_arx(res, ut, n_cifras)
    dibuja_identificacion(res, ut, n_cifras)


_CONTROLES_5 = dict(
    u_txt=widgets.Textarea(value="1 1 1", description="u (entrada):", continuous_update=False,
                           layout=widgets.Layout(width="420px", height="60px"), style={"description_width": "initial"}),
    x_txt=widgets.Textarea(value="0 2 3.6 4.88", description="x (salida):", continuous_update=False,
                           layout=widgets.Layout(width="420px", height="60px"), style={"description_width": "initial"}),
    dt_txt=widgets.Text(value="1", description="Δt =", continuous_update=False),
    unidad_t=widgets.Text(value="s", description="unidad de tiempo:", continuous_update=False,
                          style={"description_width": "initial"}),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_arx, **_CONTROLES_5);

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
PRUEBAS_5 = [
    ("ley de Ohm por el origen con I = 0.1, ..., 0.4 A y V = 1.9, 4.1, 6.0, 8.1 V → R = 121/6 (tolerancia 1e-12)",
     lambda: _cerca(ajuste_por_origen("0.1 0.2 0.3 0.4", "1.9 4.1 6.0 8.1")["k"], 121 / 6, 1e-12)
             and ajuste_por_origen("0.1 0.2 0.3 0.4", "1.9 4.1 6.0 8.1")["k"] == sp.Rational(121, 6)),
    ("ARX exacto con u = 1, 1, 1 y x = 0, 2, 3.6, 4.88 → a = 0.8, b = 2, K = 10, τ = -1/ln 0.8",
     lambda: (lambda r: r["a"] == sp.Rational(4, 5) and r["b"] == 2 and r["K"] == 10
                        and _cerca(r["tau"], -1 / math.log(0.8), 1e-12) and r["RMSE_1"] == 0)(
         identifica_arx("1 1 1", "0 2 3.6 4.88", 1))),
    ("serie sintética (a = 0.95, b = 0.5, 12 pasos) → a y b recuperados (1e-9) y simulación libre con RMSE < 1e-9; "
     "también con 24 pasos (u repetida dos veces), que usa la rama en decimales de simula_arx, identifica_arx y "
     "ajuste_por_origen",
     lambda: (lambda r: _cerca(r["a"], 0.95) and _cerca(r["b"], 0.5) and float(r["RMSE_libre"]) < 1e-9)(
         identifica_arx(_u_sint, simula_arx(0.95, 0.5, _u_sint, 0), 1))
             and (lambda x24: all(isinstance(v, float) for v in x24)
                              and all(_cerca(p, q, 1e-12) for p, q in zip(x24, _x_sint))
                              and (lambda r: not r["exacto"] and _cerca(r["a"], 0.95) and _cerca(r["b"], 0.5)
                                             and r["RMSE_libre"] < 1e-9 and _cerca(r["K"], 10, 1e-8))(
                                  identifica_arx(_u_sint * 2, x24, 1)))(simula_arx(0.95, 0.5, _u_sint * 2, 0))
             and (lambda r: not r["exacto"] and _cerca(r["k"], 2.5) and r["RMSE"] < 1e-9)(
                 ajuste_por_origen(list(range(1, 25)), [2.5 * v for v in range(1, 25)]))),
    ("simula_arx(0.8, 2, [1, 1, 1], 0) = [0, 2, 3.6, 4.88]",
     lambda: simula_arx(0.8, 2, [1, 1, 1], 0) == datos("0 2 3.6 4.88")),
    ("ajuste_lineal con x2 = 10·x1 ± 0.01 → aviso de condición; con columnas independientes, sin aviso",
     lambda: _todos["condicion"] > 1000 and _todos["avisos"] == ["columnas casi dependientes: los coeficientes no son "
                                                                 "confiables por separado"]
             and (lambda r: r["condicion"] < 1000 and not r["avisos"])(
                 ajuste_lineal([_x1, [3, 1, 4, 1, 5, 9]], _yc))),
    ("identifica_arx con len(x) ≠ len(u) + 1 → ValueError «x debe tener un dato más que u» (y con «3,6» la pista del "
     "punto decimal)",
     lambda: _mensaje(lambda: identifica_arx("1 1 1", "0 2 3.6", 1)).startswith("x debe tener un dato más que u")
             and _mensaje(lambda: identifica_arx("1 1", "0 2 3.6 4.88", 1)).startswith("x debe tener un dato más que u")
             and "punto decimal" in _mensaje(lambda: identifica_arx("1 1 1", "0 2 3,6 4,88", 1))),
    ("a ≥ 1 (u = 1, 1, 1 y x = 0, 1, 2.2, 3.64, con a = 1.2) → aviso de que no se estabiliza, sin τ ni K; a = 0 → K = b "
     "sin τ; a = -1/2 → K = 2/3 sin τ (converge oscilando); a = -1 → no se estabiliza",
     lambda: (lambda r: r["a"] == sp.Rational(6, 5) and r["tau"] is None and r["K"] is None
                        and any("no se estabiliza" in a for a in r["avisos"]))(identifica_arx("1 1 1", "0 1 2.2 3.64", 1))
             and (lambda r: r["a"] == 0 and r["K"] == r["b"] == 2 and r["tau"] is None)(
                 identifica_arx("1 1 1", "0 2 2 2", 1))
             and (lambda r: r["a"] == sp.Rational(-1, 2) and r["K"] == sp.Rational(2, 3) and r["tau"] is None
                            and any("oscilando" in a for a in r["avisos"]))(identifica_arx("1 1 1", "0 1 0.5 0.75", 1))
             and (lambda r: r["a"] == -1 and r["K"] is None and r["tau"] is None
                            and any("no se estabiliza" in a for a in r["avisos"]))(identifica_arx("1 1 1", "0 1 0 1", 1))),
    ("prueba de humo: la calculadora corre con sus valores iniciales y con la serie sintética, sin error y sin "
     "«Revisa la entrada»",
     lambda: _humo(calculadora_arx, _CONTROLES_5)
             and _humo(calculadora_arx, _CONTROLES_5, u_txt=" ".join(str(v) for v in _u_sint),
                       x_txt=" ".join(str(v) for v in _x_sint), dt_txt="0.5", unidad_t="min", n_cifras=6)),
]
corre_pruebas(PRUEBAS_5, 5)

**Contrasta (sección 5).** Con $u=1,\ 1$ y $x=0,\ 3,\ 4.5$, plantea las dos ecuaciones de $x[k+1]=a\,x[k]+b\,u[k]$ (una por paso), resuélvelas a mano y escribe `mis_parametros = (a, b)`.

In [ ]:
mis_parametros = None     # dos números (a, b), por ejemplo (0.4, 2.5)

if mis_parametros is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mios = vector(mis_parametros, "mis_parametros")
    except ValueError as err:
        print(f"Revisa tu respuesta: {str(err).rstrip('.')}.")
    else:
        if mios.rows != 2:          # respuesta imposible: la referencia todavía no se muestra
            print(f"El modelo tiene dos parámetros, a y b (escribiste {mios.rows}): revisa tu respuesta y vuelve a "
                  "ejecutar la celda.")
        else:
            ref = identifica_arx("1 1", "0 3 4.5", 1)          # los datos del enunciado
            print(f"La calculadora da: {_magnitud('a', ref['a'], '', 4)}, {_magnitud('b', ref['b'], '', 4)}.")
            if abs(float(mios[0] - ref["a"])) <= 0.01 and abs(float(mios[1] - ref["b"])) <= 0.01:
                print("Coinciden (con tolerancia de 0.01).")
            elif abs(float(mios[1] - ref["a"])) <= 0.01 and abs(float(mios[0] - ref["b"])) <= 0.01:
                print("NO coinciden: probablemente invertiste el orden; a multiplica a x[k] y b a u[k].")
            else:
                print("NO coinciden: el paso k = 0 da x[1] = a·x[0] + b·u[0], con x[0] = 0, así que sale b directo; "
                      "después, el paso k = 1 da a.")
            print("\nEl procedimiento completo:")
            _informe_arx(ref, "s", 4)

## 6. Transformaciones de coordenadas

**Un marco de referencia** (el de una cámara, el de la pinza de un robot) se describe respecto a otro, fijo, con dos cosas:

- una **rotación** $R$ de $3\times3$, cuyas columnas son los ejes $x$, $y$, $z$ del marco móvil escritos en el fijo. Por eso son ortonormales: $R^TR=I$, y además $\det R=1$ (si fuera $-1$, sería una reflexión);
- una **traslación** $\mathbf t$: dónde está el origen del marco móvil, escrito en el fijo.

Un punto que en el marco móvil $C$ tiene coordenadas $\mathbf p_C$ tiene en el fijo $B$ las coordenadas

$$\mathbf p_B=R\,\mathbf p_C+\mathbf t .$$

Las rotaciones básicas son antihorarias vistas desde la punta de su eje (regla de la mano derecha); por ejemplo, $R_z(\theta)=\begin{bmatrix}\cos\theta&-\operatorname{sen}\theta&0\\ \operatorname{sen}\theta&\cos\theta&0\\0&0&1\end{bmatrix}$.

**Coordenadas homogéneas.** Para que rotar y trasladar sea **un solo producto de matrices**, se usa la matriz de $4\times4$

$$T=\begin{bmatrix}R&\mathbf t\\ \mathbf 0^T&1\end{bmatrix},\qquad T\begin{bmatrix}\mathbf p\\1\end{bmatrix}=\begin{bmatrix}R\,\mathbf p+\mathbf t\\1\end{bmatrix},\qquad T\begin{bmatrix}\mathbf d\\0\end{bmatrix}=\begin{bmatrix}R\,\mathbf d\\0\end{bmatrix}.$$

Un **punto** lleva un 1 al final y se traslada; una **dirección** (una velocidad, el eje de una herramienta) lleva un 0 y solo se rota.

**Composición.** Si $T_{AB}$ describe al marco $B$ en el $A$ y $T_{BC}$ al $C$ en el $B$, entonces $T_{AC}=T_{AB}\,T_{BC}$. Aplicada a un punto, actúa primero la de la derecha. **El orden importa:** en general $T_1T_2\neq T_2T_1$.

**Inversa.** Para regresar del marco fijo al móvil:

$$T^{-1}=\begin{bmatrix}R^T&-R^T\mathbf t\\ \mathbf 0^T&1\end{bmatrix}.$$

Ojo: **no es la transpuesta** de $T$. Solo la parte de rotación se invierte transponiendo; la traslación cambia a $-R^T\mathbf t$. La comprobación es $T\,T^{-1}=I$, que conviene hacer con una rotación y una traslación que no sean triviales.

In [ ]:
_UNIDADES = {"grados": "grados", "grado": "grados", "radianes": "radianes", "radian": "radianes", "rad": "radianes"}
_AYUDA_ANGULO = "escribe un número (30, 37.5, 1/3) o, en radianes, un múltiplo de pi (pi/6, 2*pi/3, π/4, -pi)"


def _unidad(unidad):
    """'grados' o 'radianes' (acepta mayúsculas, acentos, el singular y 'rad'). No hay unidad por omisión: sin unidad,
    o con otra cosa, ValueError en español."""
    if unidad is None or (isinstance(unidad, str) and not unidad.strip()):
        raise ValueError("indica la unidad: 'grados' o 'radianes' (no hay unidad por omisión)")
    t = normaliza_texto(unidad) if isinstance(unidad, str) else ""
    if t not in _UNIDADES:
        raise ValueError(f"indica la unidad: 'grados' o 'radianes' (escribiste «{unidad}»)")
    return _UNIDADES[t]


def _limpia_angulo(texto):
    """Quita el signo de grados (90°), los paréntesis que envuelven todo ((pi/2)) y lee '·' como producto (2·pi/3)."""
    t = str(texto).replace("°", "").replace("º", "").replace("·", "*").replace("×", "*").strip()
    while t.startswith("(") and t.endswith(")"):
        dentro, nivel = t[1:-1], 0
        for ch in dentro:                       # solo si los paréntesis de afuera son pareja: '(1)/(2)' no se toca
            nivel += 1 if ch == "(" else (-1 if ch == ")" else 0)
            if nivel < 0:
                break
        if nivel != 0:
            break
        t = dentro.strip()
    return t


def angulo_exacto(angulo):
    """El ángulo como número exacto: 30 -> 30, '37.5' -> 75/2, '90°' -> 90, 'pi/6' -> π/6, '2*pi/3', '2·pi/3', '2π/3',
    '(pi/2)', '-π/4', sp.pi/2. El lector es restringido: solo números, pi, productos y cocientes (no evalúa expresiones
    libres)."""
    if isinstance(angulo, sp.Basic) and not angulo.is_Rational and not angulo.is_Float:
        if angulo.is_number and (angulo / sp.pi).is_Rational:
            return angulo
        raise ValueError(f"«{angulo}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}")
    if not isinstance(angulo, str):
        return numero(angulo, "el ángulo")
    limpio = _limpia_angulo(angulo)
    if "(" in limpio or ")" in limpio:
        raise ValueError(f"«{angulo}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}")
    t = _normaliza_entrada(limpio).replace(" ", "").lower().replace("π", "pi")
    if "pi" not in t:
        try:
            return numero(limpio, "el ángulo")
        except ValueError:
            raise ValueError(f"«{angulo.strip()}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}") from None
    signo = 1
    if t[:1] in "+-":
        signo, t = (-1 if t[0] == "-" else 1), t[1:]
    t = re.sub(r"(?<=[\d.])pi", "*pi", t)                     # 2pi -> 2*pi
    trozos = re.split(r"([*/])", t)
    if not t or "" in trozos[::2]:
        raise ValueError(f"«{angulo}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}")
    coef, potencia, op = sp.Integer(1), 0, "*"
    for i, trozo in enumerate(trozos):
        if i % 2:
            op = trozo
            continue
        if trozo == "pi":
            potencia += 1 if op == "*" else -1
            continue
        try:
            q = numero(trozo, "el ángulo")
        except ValueError:
            raise ValueError(f"«{angulo}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}") from None
        if op == "/" and q == 0:
            raise ValueError(f"«{angulo}» divide entre cero")
        coef = coef * q if op == "*" else coef / q
    if potencia not in (0, 1):
        raise ValueError(f"«{angulo}» no es un ángulo que entienda esta calculadora: {_AYUDA_ANGULO}")
    return signo * coef * sp.pi ** potencia


def _angulo_rad(angulo, unidad):
    """El ángulo exacto en radianes: 30 grados -> pi/6; 30 radianes -> 30; 'pi/6' radianes -> pi/6."""
    u = _unidad(unidad)
    if u == "radianes" and isinstance(angulo, str) and ("°" in angulo or "º" in angulo):
        raise ValueError("escribiste el ángulo con °, pero la unidad es 'radianes': cambia la unidad a 'grados' o quita "
                         "el °")
    a = angulo_exacto(angulo)
    if u == "grados":
        if a.has(sp.pi):
            raise ValueError("con la unidad 'grados' escribe el ángulo sin pi (por ejemplo 30); pi/6 ya está en radianes")
        return sp.rad(a)
    return a


def _texto_angulo(angulo, unidad):
    """'90 grados', 'π/2 radianes'."""
    a = angulo_exacto(angulo)
    return f"{_ntxt(a) if a.is_Rational else _txt(a).replace('·π', 'π')} {_unidad(unidad)}"


def rot(eje, angulo, unidad=None):
    """Rotación exacta de R³ alrededor del eje 'x', 'y' o 'z', antihoraria vista desde la punta del eje (regla de la
    mano derecha). La unidad es OBLIGATORIA: rot('z', 90, 'grados') o rot('z', 'pi/2', 'radianes'); sin unidad,
    ValueError «indica la unidad». Con ángulos notables (30, 45, 60, 90 grados...) las entradas son exactas."""
    u = _unidad(unidad)
    e = normaliza_texto(eje) if isinstance(eje, str) else ""
    if e not in ("x", "y", "z"):
        raise ValueError(f"el eje de rotación debe ser 'x', 'y' o 'z' (escribiste «{eje}»)")
    t = _angulo_rad(angulo, u)
    c, s = sp.cos(t), sp.sin(t)
    if sp.count_ops(c) + sp.count_ops(s) > 20:     # sympy los desarrolló en radicales anidados (1.5°, 12°...): se dejan
        c, s = sp.cos(t, evaluate=False), sp.sin(t, evaluate=False)          # como cos(π/120), que se calcula rápido
    if e == "x":
        return sp.Matrix([[1, 0, 0], [0, c, -s], [0, s, c]])
    if e == "y":
        return sp.Matrix([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    return sp.Matrix([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def _matriz_exacta(M, nombre):
    """Una Matrix de sympy se toma tal cual (puede traer senos, cosenos y raíces exactos); texto o listas se leen como
    racionales exactos."""
    if isinstance(M, sp.MatrixBase):
        if not all(sp.sympify(e).is_number for e in M):
            raise ValueError(f"{nombre} debe tener solo números")
        return sp.Matrix(M)
    return como_matriz(M, nombre)


def _motivo_no_rotacion(R):
    """None si R es una rotación (RᵀR = I y det R = 1); si no, el motivo."""
    if R.shape != (3, 3):
        return f"debe ser de 3×3 (es de {dims(R)})"
    if not all(_es_cero(e) for e in R.T * R - sp.eye(3)):
        return ("RᵀR ≠ I: sus columnas no son ortonormales, así que no conserva longitudes ni ángulos (si escribiste "
                "decimales redondeados, construye R con rot())")
    d = R.det()
    if _es_cero(d + 1):
        return "det R = -1: es una reflexión (cambia la orientación), no una rotación"
    return None if _es_cero(d - 1) else f"det R = {_valor(d)}, no 1"


def homogenea(R, t):
    """La matriz homogénea T = [[R, t], [0, 0, 0, 1]] de 4×4. R debe ser una rotación (RᵀR = I y det R = 1); si no,
    ValueError «R no es una rotación: ...». t es la traslación, un vector de R³."""
    R = _matriz_exacta(R, "R")
    motivo = _motivo_no_rotacion(R)
    if motivo:
        raise ValueError(f"R no es una rotación: {motivo}")
    t = _vector3(t, "la traslación t")
    T = sp.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T


def _valida_T(T, nombre="T"):
    """T como Matrix de 4×4 homogénea: último renglón (0, 0, 0, 1) y bloque R de rotación."""
    T = _matriz_exacta(T, nombre)
    if T.shape != (4, 4):
        raise ValueError(f"{nombre} debe ser de 4×4 (es de {dims(T)})")
    if list(T.row(3)) != [0, 0, 0, 1]:
        raise ValueError(f"{nombre} no es homogénea: su último renglón debe ser (0, 0, 0, 1)")
    motivo = _motivo_no_rotacion(T[:3, :3])
    if motivo:
        raise ValueError(f"el bloque R de {nombre} no es una rotación: {motivo}")
    return T


def _tipo(tipo):
    t = normaliza_texto(tipo) if isinstance(tipo, str) else ""
    if t not in ("punto", "direccion"):
        raise ValueError("indica el tipo: 'punto' (lleva 1 al final y se traslada) o 'dirección' (lleva 0 y solo se rota)")
    return t


def transforma(T, p, tipo=None):
    """Aplica T a p. El tipo es OBLIGATORIO: 'punto' (p se escribe [p; 1] y se traslada) o 'dirección' ([p; 0], solo
    se rota). Devuelve el vector de R³ resultante, exacto."""
    T, tp = _valida_T(T), _tipo(tipo)
    p = _vector3(p, "el punto" if tp == "punto" else "la dirección")
    h = p.col_join(sp.Matrix([1 if tp == "punto" else 0]))
    return _simplifica(T * h)[:3, 0]


def inversa_homogenea(T):
    """Inversa de T con la fórmula T⁻¹ = [[Rᵀ, -Rᵀt], [0, 0, 0, 1]], sin invertir una matriz de 4×4. Diccionario con
    Tinv, R, t, traslacion (-Rᵀt), comprobacion (T·T⁻¹, simplificada) y es_identidad (True si T·T⁻¹ = I)."""
    T = _valida_T(T)
    R, t = T[:3, :3], T[:3, 3]
    Ti = sp.eye(4)
    Ti[:3, :3] = R.T
    Ti[:3, 3] = _simplifica(-R.T * t)
    C = _simplifica(T * Ti)
    return {"Tinv": Ti, "R": R, "t": t, "traslacion": Ti[:3, 3], "comprobacion": C,
            "es_identidad": all(_es_cero(x) for x in C - sp.eye(4))}


def compone(*Ts, nombres=None):
    """Producto T1·T2·...·Tk en el orden dado. Diccionario con T (el producto, exacto) y texto (en qué orden actúan
    sobre un punto: primero la de la derecha)."""
    if not Ts:
        raise ValueError("escribe al menos una transformación")
    if len(Ts) > 6:
        raise ValueError(f"escribiste {len(Ts)} transformaciones; esta función compone hasta 6")
    Ms = [_valida_T(T, f"T{k}") for k, T in enumerate(Ts, 1)]
    nombres = list(nombres) if nombres is not None else [f"T{k}" for k in range(1, len(Ms) + 1)]
    if len(nombres) != len(Ms):
        raise ValueError(f"hay {_cuenta(len(Ms), 'transformación', 'transformaciones')} y "
                         f"{_cuenta(len(nombres), 'nombre', 'nombres')}: debe haber un nombre por cada una")
    P = Ms[0]
    for M in Ms[1:]:
        P = _simplifica(P * M)
    if len(Ms) == 1:
        texto = f"{nombres[0]}: una sola transformación"
    else:
        orden = list(reversed(nombres))
        if len(orden) == 2:
            texto = f"{'·'.join(nombres)}: aplicada a un punto, actúa primero {orden[0]} y después {orden[1]} (el orden importa)"
        else:
            texto = (f"{'·'.join(nombres)}: aplicada a un punto, actúa primero {orden[0]}"
                     + "".join(f", después {nm}" for nm in orden[1:-1]) + f" y al final {orden[-1]} (el orden importa)")
    return {"T": P, "texto": texto}


def _informe_transformacion(R, T, p, tipo, angulo_texto, eje, n_cifras):
    tp = _tipo(tipo)
    t = T[:3, 3]
    muestra(f"R = R_{eje}({angulo_texto})", R, n_cifras, tex=rf"R=R_{{{eje}}}(\text{{{angulo_texto}}})")
    print(f"t = {_tupla_u(t, 'm', n_cifras)}  (el origen del marco móvil, en el marco fijo)")
    muestra("T", T, n_cifras, tex=r"T=\begin{bmatrix}R&\mathbf t\\ \mathbf 0^T&1\end{bmatrix}")
    q = transforma(T, p, tp)
    if tp == "punto":
        print(f"Punto p = {_tupla_u(p, 'm')} (en el marco móvil) → [p; 1] → T·[p; 1] da R·p + t = "
              f"{_tupla_u(q, 'm', n_cifras)} (en el marco fijo)")
    else:
        print(f"Dirección d = {_tupla(p)} → [d; 0] → T·[d; 0] da R·d = {_tupla(q, n_cifras)}: la traslación no la "
              "mueve, solo se rota")
    inv = inversa_homogenea(T)
    muestra("T⁻¹", inv["Tinv"], n_cifras, tex=r"T^{-1}=\begin{bmatrix}R^T&-R^T\mathbf t\\ \mathbf 0^T&1\end{bmatrix}")
    print(f"-Rᵀt = {_tupla_u(inv['traslacion'], 'm', n_cifras)};  comprobación: T·T⁻¹ " +
          ("= I" if inv["es_identidad"] else "≠ I (¡revisa!)"))
    if not _es_nulo(t):
        print("Tᵀ no es la inversa: T·Tᵀ ≠ I, porque t ≠ 0.")
    else:
        print("Con t = 0 solo hay rotación, y entonces Tᵀ sí es la inversa: T·Tᵀ = I.")
    return q


def _redondo(o):
    """Un origen en decimales como Matrix con racionales cortos (2.0 -> 2), solo para rotularlo."""
    return sp.Matrix([sp.Rational(repr(float(v))) if abs(v) < 1e15 else v for v in o])


def dibuja_marcos(marcos, puntos=None, n_cifras=4, unidad="m"):
    """Dibujo 3D de hasta 6 marcos (nombre, T): sus ejes x, y, z (continuo, discontinuo y punteado), con un color y
    un marcador por marco. puntos: lista opcional de (rótulo, vector, 'punto' o 'dirección') en el marco fijo; un
    punto se marca con una estrella y una dirección con una flecha desde el origen."""
    if len(marcos) > 6:
        print("No se dibuja: son más de 6 marcos.")
        return
    datos_m = []
    for nombre, T in marcos:
        T = _valida_T(T)
        datos_m.append((nombre, _flotante(T[:3, :3]), _flotante(T[:3, 3]).ravel()))
    origenes = np.array([o for _, _, o in datos_m])
    extension = float(np.ptp(origenes, axis=0).max()) if len(datos_m) > 1 else 0.0
    largo = max(1.0, 0.45 * extension)
    fig = plt.figure(figsize=(7.4, 8.2))
    ax = fig.add_subplot(projection="3d")
    fig.subplots_adjust(bottom=0.2)
    todos = [origenes]
    estilos_eje = ("-", "--", ":")
    for k, (nombre, R, o) in enumerate(datos_m):
        color = _COLORES[k % len(_COLORES)]
        for i, letra in enumerate("xyz"):
            fin = o + (0.8 if k % 2 == 0 else 0.6) * largo * R[:, i]   # largos distintos: las puntas no se enciman
            ax.plot(*np.array([o, fin]).T, linestyle=estilos_eje[i], color=color, lw=2.4)
            ax.text(*fin, f" {letra}{_sub(k)}", fontsize=9, color="black")
            todos.append(fin[None, :])
        ax.scatter(*o, marker=_MARCAS[k % len(_MARCAS)], s=60, color=color, depthshade=False,
                   label=f"marco {nombre}: origen {_tupla_dec(_redondo(o), n_cifras)} {unidad}")
    for etiqueta, v, tp in (puntos or []):
        vf = _flotante(sp.Matrix(v)).ravel()
        if normaliza_texto(tp) == "punto":
            ax.scatter(*vf, marker="*", s=120, color="black", depthshade=False, zorder=5,
                       label=f"{etiqueta}: {_tupla_dec(sp.Matrix(v), n_cifras)} {unidad}")
            todos.append(vf[None, :])
        else:
            punta = largo * vf / max(np.linalg.norm(vf), 1e-300)
            ax.quiver(0, 0, 0, *punta, color="black", linewidth=2.6, arrow_length_ratio=0.22)
            ax.plot([], [], "-", color="black", lw=2.6, label=f"{etiqueta}: {_tupla_dec(sp.Matrix(v), n_cifras)} (flecha)")
            todos.append(punta[None, :])
    for ls, letra in zip(estilos_eje, "xyz"):
        ax.plot([], [], linestyle=ls, color="0.3", lw=2.4, label=f"eje {letra} de cada marco")
    P = np.vstack(todos)
    centro = (P.max(axis=0) + P.min(axis=0)) / 2
    radio = max(float((P.max(axis=0) - P.min(axis=0)).max()) / 2, 1.0) * 1.15
    ax.set_xlim(centro[0] - radio, centro[0] + radio)
    ax.set_ylim(centro[1] - radio, centro[1] + radio)
    ax.set_zlim(centro[2] - radio, centro[2] + radio)
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel(f"x ({unidad})"); ax.set_ylabel(f"y ({unidad})"); ax.set_zlabel(f"z ({unidad})")
    ax.view_init(elev=25, azim=-58)
    ax.set_title("Marcos de referencia (ejes escritos en el marco fijo)", fontsize=10)
    fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncol=2, fontsize=8)
    plt.show()

**Recorrido paso a paso.** (1) Un marco móvil girado $90^\circ$ alrededor de $z$ y con el origen en $\mathbf t=(2,0,0)$: el punto $(1,0,0)$ del marco móvil está en $(2,1,0)$ en el fijo (se rota a $(0,1,0)$ y se traslada), y la dirección $(1,0,0)$ pasa a $(0,1,0)$: se rota, pero no se traslada. (2) La inversa de $T$ con $R=R_z(90^\circ)$ y $\mathbf t=(1,2,3)$: su traslación es $-R^T\mathbf t=(-2,\,1,\,-3)$, y $T\,T^{-1}=I$; en cambio, $T\,T^T\neq I$. (3) Dos composiciones con $T_1$ = rotación $R_x(90^\circ)$ y $T_2$ = traslación $(0,0,1)$, aplicadas al punto $(0,1,0)$: $T_1T_2$ primero traslada y después rota, y $T_2T_1$ primero rota y después traslada. Dan puntos distintos.

In [ ]:
print("(1) R = R_z(90 grados), t = (2, 0, 0)")
_R90 = rot("z", 90, "grados")
_T1 = homogenea(_R90, (2, 0, 0))
_q1 = _informe_transformacion(_R90, _T1, vector("1, 0, 0"), "punto", "90 grados", "z", 4)
print(f"La dirección (1, 0, 0), en cambio: {_tupla(transforma(_T1, (1, 0, 0), 'dirección'))}")
dibuja_marcos([("0 (fijo)", sp.eye(4)), ("1 (móvil)", _T1)], puntos=[("p en el marco fijo", _q1, "punto")])
print("=" * 90 + "\n\n(2) Inversa con R = R_z(90 grados) y t = (1, 2, 3)")
_T2 = homogenea(_R90, (1, 2, 3))
_inv2 = inversa_homogenea(_T2)
muestra("T", _T2)
muestra("T⁻¹", _inv2["Tinv"], tex=r"T^{-1}")
print(f"-Rᵀt = {_tupla(_inv2['traslacion'])};  T·T⁻¹ = I: {'sí' if _inv2['es_identidad'] else 'no'};  "
      f"T·Tᵀ = I: {'sí' if _T2 * _T2.T == sp.eye(4) else 'no'}")
muestra("T·Tᵀ", _T2 * _T2.T, tex=r"T\,T^T")
print("=" * 90 + "\n\n(3) El orden importa: T1 = rotación R_x(90 grados), T2 = traslación (0, 0, 1), punto (0, 1, 0)")
_Rx = homogenea(rot("x", 90, "grados"), (0, 0, 0))
_Tz = homogenea(sp.eye(3), (0, 0, 1))
for _orden in ((_Rx, _Tz), (_Tz, _Rx)):
    _c = compone(*_orden, nombres=["T1", "T2"] if _orden[0] is _Rx else ["T2", "T1"])
    print(f"{_c['texto']}  →  {_tupla(transforma(_c['T'], (0, 1, 0), 'punto'))}")

**Calculadora.** Elige el eje y el ángulo de la rotación **con su unidad** (en radianes puedes escribir `pi/2` o `π/4`), la traslación $\mathbf t$ en metros, el vector y si es un **punto** o una **dirección**. La calculadora arma $T$, transforma el vector, calcula $T^{-1}$ con la fórmula y comprueba $T\,T^{-1}=I$, y dibuja los dos marcos. Prueba 90 en radianes (no es lo mismo que 90 grados), cambiar el punto por una dirección y una traslación $(0,0,0)$, con la que $T^T$ sí es la inversa.

In [ ]:
def calculadora_transformacion(eje, angulo_txt, unidad, traslacion_txt, punto_txt, tipo, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        try:
            R = rot(eje, angulo_txt, unidad)
            T = homogenea(R, traslacion_txt)
            p = _vector3(punto_txt, "el punto" if _tipo(tipo) == "punto" else "la dirección")
            texto = _texto_angulo(angulo_txt, unidad)
        except ValueError as err:
            _reporta(err); return
        _aviso_coma(traslacion_txt, punto_txt)
        q = _informe_transformacion(R, T, p, tipo, texto, normaliza_texto(eje), n_cifras)
    dibuja_marcos([("0 (fijo)", sp.eye(4)), ("1 (móvil)", T)],
                  puntos=[("p en el marco fijo" if _tipo(tipo) == "punto" else "R·d", q, tipo)], n_cifras=n_cifras)


_CONTROLES_6 = dict(
    eje=widgets.Dropdown(options=["x", "y", "z"], value="z", description="eje:"),
    angulo_txt=widgets.Text(value="90", description="ángulo:", continuous_update=False),
    unidad=widgets.ToggleButtons(options=["grados", "radianes"], value="grados", description="unidad:"),
    traslacion_txt=widgets.Text(value="2, 0, 0", description="t (m) =", continuous_update=False),
    punto_txt=widgets.Text(value="1, 0, 0", description="punto/dirección:", continuous_update=False,
                           style={"description_width": "initial"}),
    tipo=widgets.ToggleButtons(options=["punto", "dirección"], value="punto", description="es un:"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_transformacion, **_CONTROLES_6);

In [ ]:
# Casos de prueba de la sección 6 (resultado conocido)
def _rapida(angulo, tope=6.0):
    """La calculadora con un ángulo cuyo coseno sympy desarrolla en radicales anidados: termina en menos de tope
    segundos, sin «Revisa la entrada», y comprueba T·T⁻¹ = I."""
    inicio = time.time()
    texto, error = _captura(calculadora_transformacion, "z", angulo, "grados", "1, 2, 3", "1, 1, 1", "punto", 4)
    return (error is None and time.time() - inicio < tope and "Revisa la entrada" not in texto
            and "comprobación: T·T⁻¹ = I" in texto)


PRUEBAS_6 = [
    ("rot('z', 90, 'grados') = [[0, -1, 0], [1, 0, 0], [0, 0, 1]], y en radianes con pi/2 (y con 'pi/2', '(pi/2)' o "
     "'90°' en grados) es igual; R_y(90 grados)·(0, 0, 1) = (1, 0, 0); '2·pi/3' = 2π/3",
     lambda: rot("z", 90, "grados") == sp.Matrix([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
             and rot("z", sp.pi / 2, "radianes") == rot("z", 90, "grados") == rot("z", "pi/2", "radianes")
             and rot("z", "(pi/2)", "radianes") == rot("z", "90°", "grados") == rot("z", 90, "grados")
             and rot("y", 90, "grados") * vector("0, 0, 1") == vector("1, 0, 0")
             and angulo_exacto("2·pi/3") == 2 * sp.pi / 3),
    ("R_z(90 grados) con t = (2, 0, 0): el punto (1, 0, 0) → (2, 1, 0) y la dirección (1, 0, 0) → (0, 1, 0)",
     lambda: transforma(_T1, (1, 0, 0), "punto") == vector("2, 1, 0")
             and transforma(_T1, (1, 0, 0), "dirección") == vector("0, 1, 0")),
    ("inversa_homogenea con R_z(90 grados) y t = (1, 2, 3) → traslación (-2, 1, -3) y T·T⁻¹ = I",
     lambda: _inv2["traslacion"] == vector("-2, 1, -3") and _inv2["es_identidad"]
             and _inv2["Tinv"] == _T2.inv()),
    ("la transpuesta de T no es su inversa cuando t ≠ 0: T·Tᵀ ≠ I (y con t = 0 sí lo es)",
     lambda: _T2 * _T2.T != sp.eye(4) and (lambda T: T * T.T == sp.eye(4))(homogenea(_R90, (0, 0, 0)))),
    ("composición: T1·T2 ≠ T2·T1 con una rotación y una traslación no nulas, y (T1·T2)⁻¹ = T2⁻¹·T1⁻¹",
     lambda: compone(_Rx, _Tz)["T"] != compone(_Tz, _Rx)["T"]
             and inversa_homogenea(compone(_Rx, _Tz)["T"])["Tinv"] ==
                 _simplifica(inversa_homogenea(_Tz)["Tinv"] * inversa_homogenea(_Rx)["Tinv"])
             and transforma(compone(_Rx, _Tz)["T"], (0, 1, 0), "punto") == vector("0, -1, 1")
             and transforma(compone(_Tz, _Rx)["T"], (0, 1, 0), "punto") == vector("0, 0, 2")),
    ("homogenea con R = [[1, 1, 0], [0, 1, 0], [0, 0, 1]] → ValueError; con la reflexión diag(1, 1, -1) → ValueError "
     "(det = -1)",
     lambda: _mensaje(lambda: homogenea([[1, 1, 0], [0, 1, 0], [0, 0, 1]], (0, 0, 0))).startswith("R no es una rotación")
             and (lambda m: m.startswith("R no es una rotación") and "det R = -1" in m and "reflexión" in m)(
                 _mensaje(lambda: homogenea(sp.diag(1, 1, -1), (0, 0, 0))))),
    ("rot('z', 90) sin unidad → ValueError «indica la unidad»; con 'sexagesimales', también",
     lambda: _mensaje(lambda: rot("z", 90)).startswith("indica la unidad")
             and _mensaje(lambda: rot("z", 90, "sexagesimales")).startswith("indica la unidad")),
    ("prueba de humo: la calculadora corre con sus valores iniciales, con una dirección, en radianes y con 1.5 y 12 "
     "grados (cada uno en menos de 6 s y con T·T⁻¹ = I), sin error y sin «Revisa la entrada»",
     lambda: _humo(calculadora_transformacion, _CONTROLES_6)
             and _humo(calculadora_transformacion, _CONTROLES_6, tipo="dirección", eje="x", n_cifras=6)
             and _humo(calculadora_transformacion, _CONTROLES_6, unidad="radianes", angulo_txt="pi/3")
             and all(_rapida(angulo) for angulo in ("1.5", "12"))),
]
corre_pruebas(PRUEBAS_6, 6)

**Contrasta (sección 6).** Con $R=R_z(90^\circ)$ y $\mathbf t=(0,0,2)$, transforma a mano el punto $(1,1,0)$: primero $R\,\mathbf p$, después suma $\mathbf t$. Escribe `mi_punto = (x, y, z)`.

In [ ]:
mi_punto = None     # tres números (x, y, z), por ejemplo (0.5, -1, 3)

if mi_punto is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mio = vector(mi_punto, "mi_punto")
    except ValueError as err:
        print(f"Revisa tu respuesta: {str(err).rstrip('.')}.")
    else:
        if mio.rows != 3:          # respuesta imposible: la referencia todavía no se muestra
            print(f"El punto transformado es de R³, con tres coordenadas (escribiste {mio.rows}): revisa tu respuesta y "
                  "vuelve a ejecutar la celda.")
        else:
            T_c = homogenea(rot("z", 90, "grados"), (0, 0, 2))          # los datos del enunciado
            ref = transforma(T_c, (1, 1, 0), "punto")
            print(f"La calculadora da: R·p + t = {_tupla(ref)}.")
            al_reves = transforma(homogenea(rot("z", -90, "grados"), (0, 0, 2)), (1, 1, 0), "punto")
            if all(abs(float(a - b)) <= 0.01 for a, b in zip(mio, ref)):
                print("Coinciden (con tolerancia de 0.01).")
            elif all(abs(float(a - b)) <= 0.01 for a, b in zip(mio, al_reves)):
                print("NO coinciden: giraste en sentido horario. Revisa el signo de la rotación: R_z(90°) gira en sentido "
                      "antihorario visto desde +z, así que manda (1, 0, 0) a (0, 1, 0) y (0, 1, 0) a (-1, 0, 0).")
            elif all(abs(float(a - b)) <= 0.01 for a, b in zip(mio, ref - vector("0, 0, 2"))):
                print("NO coinciden: te faltó sumar la traslación t = (0, 0, 2).")
            else:
                print("NO coinciden: calcula R·p con R = [[0, -1, 0], [1, 0, 0], [0, 0, 1]] y después suma t.")
            print("\nEl procedimiento completo:")
            _informe_transformacion(rot("z", 90, "grados"), T_c, vector("1, 1, 0"), "punto", "90 grados", "z", 4)

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
_ESPERADAS = {1: 9, 2: 8, 3: 7, 4: 8, 5: 8, 6: 8}          # pruebas por sección (el libro cita estos números)
fallas, total = [], 0
for k, pruebas in enumerate([PRUEBAS_1, PRUEBAS_2, PRUEBAS_3, PRUEBAS_4, PRUEBAS_5, PRUEBAS_6], 1):
    assert len(pruebas) == _ESPERADAS[k], (f"la sección {k} tiene {len(pruebas)} pruebas y deben ser "
                                           f"{_ESPERADAS[k]}: falta o sobra alguna")
    resultados = evalua(pruebas)                      # cada caso se ejecuta una sola vez
    print(f"Sección {k}: {sum(ok for _, ok in resultados)}/{len(pruebas)}")
    fallas += [f"sección {k}: {nombre}" for nombre, ok in resultados if not ok]
    total += len(pruebas)
for falla in fallas:
    print("FALLA", falla)
assert not fallas, f"Fallan {len(fallas)} de {total} pruebas (están arriba)"
print(f"Todas las verificaciones pasaron: {total}/{total}")